In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ================================
# RoadVision - Colab GPU Setup
# ================================

import sys
import subprocess
import torch

print("=" * 55)
print("RoadVision - GPU Environment Check")
print("=" * 55)

# 1. Check GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)

    print("✅ GPU AVAILABLE")
    print(f"GPU       : {gpu_name}")
    print(f"GPU Memory: {gpu_memory:.1f} GB")
    print(f"CUDA      : {torch.version.cuda}")
else:
    print("❌ GPU NOT AVAILABLE")
    print("Go to: Runtime → Change runtime type → GPU")

# 2. Install Ultralytics
print("\nInstalling Ultralytics...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "ultralytics"
])

# 3. Verify Ultralytics
from ultralytics import YOLO
import ultralytics

print(f"\nUltralytics: {ultralytics.__version__}")
print(f"PyTorch    : {torch.__version__}")
print(f"CUDA ready : {torch.cuda.is_available()}")

# 4. Final status
print("\n" + "=" * 55)

if torch.cuda.is_available():
    print("🚀 READY FOR ROADVISION GPU TRAINING!")
    print(f"Using: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU unavailable — DO NOT START TRAINING YET.")

print("=" * 55)

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

print("\n✅ Google Drive mounted!")

# Drive ke RoadVision folder ke andar files dekho
ROADVISION_DRIVE = Path("/content/drive/MyDrive/RoadVision")

if ROADVISION_DRIVE.exists():
    print("\n📁 RoadVision folder found:")
    for item in ROADVISION_DRIVE.iterdir():
        print("  ", item.name)
else:
    print("\n❌ RoadVision folder nahi mila.")
    print("Google Drive mein 'RoadVision' folder banao aur dataset ZIP uske andar rakho.")

In [ ]:
from pathlib import Path
import zipfile
import shutil
import yaml

# Find the ZIP already uploaded to Colab
zips = list(Path("/content").glob("*.zip"))

print("ZIP files found:")
for z in zips:
    print("  ", z.name)

if not zips:
    raise FileNotFoundError("❌ Uploaded ZIP /content mein nahi mili.")

ZIP_PATH = zips[0]

# RoadVision working directory
WORK_DIR = Path("/content/RoadVision")
DATA_DIR = WORK_DIR / "data" / "raw"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n✅ Using existing upload: {ZIP_PATH.name}")
print(f"📦 Size: {ZIP_PATH.stat().st_size / 1024**2:.1f} MB")

# Extract
print("\n📂 Extracting...")
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(DATA_DIR)

print("✅ Extraction complete!")

# Find data.yaml
yaml_files = list(DATA_DIR.rglob("data.yaml"))

if not yaml_files:
    raise FileNotFoundError("❌ data.yaml nahi mila.")

DATA_YAML = yaml_files[0]
DATASET_ROOT = DATA_YAML.parent

print(f"\n📁 Dataset root:")
print(DATASET_ROOT)

# Load YAML
with open(DATA_YAML, "r") as f:
    config = yaml.safe_load(f)

print("\nClasses:", config["names"])
print("Number of classes:", config["nc"])

# Set correct absolute paths
config["train"] = str(DATASET_ROOT / "train" / "images")
config["val"]   = str(DATASET_ROOT / "valid" / "images")
config["test"]  = str(DATASET_ROOT / "test" / "images")

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

# Verify dataset
print("\n" + "=" * 60)
print("DATASET VERIFICATION")
print("=" * 60)

for name, path in [
    ("Train", Path(config["train"])),
    ("Validation", Path(config["val"])),
    ("Test", Path(config["test"]))
]:
    count = len(list(path.glob("*")))
    print(f"{name:12}: {count} images | Exists: {path.exists()}")

print("\nClasses:", config["names"])
print("YAML:", DATA_YAML)

print("\n" + "=" * 60)
print("🚀 ROADVISION DATASET READY")
print("=" * 60)

In [ ]:
# ============================================================
# RoadVision — BASELINE YOLO11n TRAINING
# Experiment 01: Clean Baseline
# ============================================================

from ultralytics import YOLO
from pathlib import Path
import shutil
import torch

# ------------------------------------------------------------
# 1. Verify GPU
# ------------------------------------------------------------

assert torch.cuda.is_available(), "❌ GPU not available!"

print("🚀 GPU:", torch.cuda.get_device_name(0))
print("💾 VRAM:",
      round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
      "GB")

# ------------------------------------------------------------
# 2. Paths
# ------------------------------------------------------------

DATA_YAML = "/content/RoadVision/data/raw/data.yaml"

RUN_DIR = "/content/RoadVision/runs/detect/baseline_yolo11n"

# ------------------------------------------------------------
# 3. Load pretrained YOLO11n
# ------------------------------------------------------------

print("\n📦 Loading pretrained YOLO11n...")

model = YOLO("yolo11n.pt")

# ------------------------------------------------------------
# 4. Baseline training
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("🚀 STARTING ROADVISION BASELINE TRAINING")
print("=" * 65)

results = model.train(
    data=DATA_YAML,

    # Baseline configuration
    epochs=30,
    imgsz=640,
    batch=16,

    # GPU
    device=0,

    # Data loading
    workers=2,

    # Early stopping
    patience=10,

    # Save checkpoints/results
    project="/content/RoadVision/runs/detect",
    name="baseline_yolo11n",
    exist_ok=True,
    save=True,

    # Keep baseline clean
    cache=False,

    # Reproducibility
    seed=42,

    # Generate plots
    plots=True,

    # Verbose logs
    verbose=True
)

# ------------------------------------------------------------
# 5. Best model
# ------------------------------------------------------------

BEST_MODEL = Path(RUN_DIR) / "weights" / "best.pt"

print("\n" + "=" * 65)
print("✅ BASELINE TRAINING FINISHED")
print("=" * 65)

print("\nBest model:")
print(BEST_MODEL)

print("\nExists:", BEST_MODEL.exists())

# ------------------------------------------------------------
# 6. Copy best model + results to Google Drive
# ------------------------------------------------------------

DRIVE_DIR = Path(
    "/content/drive/MyDrive/RoadVision/baseline_yolo11n"
)

DRIVE_DIR.mkdir(parents=True, exist_ok=True)

if BEST_MODEL.exists():

    shutil.copy2(
        BEST_MODEL,
        DRIVE_DIR / "best.pt"
    )

    # Copy results CSV if available
    results_csv = Path(RUN_DIR) / "results.csv"

    if results_csv.exists():
        shutil.copy2(
            results_csv,
            DRIVE_DIR / "results.csv"
        )

    print("\n☁️ Saved to Google Drive:")
    print(DRIVE_DIR)

print("\n🎯 NEXT STEP: BASELINE EVALUATION")

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import shutil
import pandas as pd

# Best trained model
BEST_MODEL = "/content/RoadVision/runs/detect/baseline_yolo11n/weights/best.pt"
DATA_YAML = "/content/RoadVision/data/raw/data.yaml"

print("📦 Loading best model...")
model = YOLO(BEST_MODEL)

print("\n" + "=" * 65)
print("🔎 BASELINE VALIDATION")
print("=" * 65)

metrics = model.val(
    data=DATA_YAML,
    split="val",
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    verbose=True
)

# Overall metrics
print("\n" + "=" * 65)
print("📊 OVERALL BASELINE RESULTS")
print("=" * 65)

print(f"Precision    : {metrics.box.mp:.4f}")
print(f"Recall       : {metrics.box.mr:.4f}")
print(f"mAP50        : {metrics.box.map50:.4f}")
print(f"mAP50-95     : {metrics.box.map:.4f}")

# Per-class metrics
classes = ["D00", "D10", "D20", "D40"]

print("\n" + "=" * 65)
print("📊 PER-CLASS RESULTS")
print("=" * 65)

rows = []

for i, cls in enumerate(classes):
    p, r, ap50, ap5095 = metrics.box.class_result(i)

    rows.append({
        "Class": cls,
        "Precision": p,
        "Recall": r,
        "mAP50": ap50,
        "mAP50-95": ap5095
    })

df = pd.DataFrame(rows)

print(df.to_string(index=False))

# Save results
RESULT_DIR = Path(
    "/content/RoadVision/runs/detect/baseline_yolo11n"
)

df.to_csv(
    RESULT_DIR / "per_class_metrics.csv",
    index=False
)

# Copy important files to Drive
DRIVE_DIR = Path(
    "/content/drive/MyDrive/RoadVision/baseline_yolo11n"
)

DRIVE_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(
    BEST_MODEL,
    DRIVE_DIR / "best.pt"
)

shutil.copy2(
    RESULT_DIR / "per_class_metrics.csv",
    DRIVE_DIR / "per_class_metrics.csv"
)

print("\n☁️ Saved to Google Drive:")
print(DRIVE_DIR)

print("\n" + "=" * 65)
print("✅ BASELINE EVALUATION COMPLETE")
print("=" * 65)

In [ ]:
# ============================================================
# RoadVision — D10 Error Analysis
# ============================================================

from ultralytics import YOLO
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

MODEL_PATH = "/content/RoadVision/runs/detect/baseline_yolo11n/weights/best.pt"

VAL_IMAGES = Path("/content/RoadVision/data/raw/valid/images")
VAL_LABELS = Path("/content/RoadVision/data/raw/valid/labels")

CONFUSION_MATRIX = Path("/content/runs/detect/val/confusion_matrix.png")

model = YOLO(MODEL_PATH)

# ------------------------------------------------------------
# 1. Find validation images containing D10
# Class ID:
# D00 = 0
# D10 = 1
# D20 = 2
# D40 = 3
# ------------------------------------------------------------

d10_images = []

for label_file in VAL_LABELS.glob("*.txt"):

    text = label_file.read_text().strip()

    if not text:
        continue

    for line in text.splitlines():

        parts = line.split()

        if len(parts) >= 5 and int(parts[0]) == 1:
            image_file = VAL_IMAGES / f"{label_file.stem}.jpg"

            if image_file.exists():
                d10_images.append(image_file)

            break

print("=" * 65)
print("D10 VALIDATION ERROR ANALYSIS")
print("=" * 65)

print(f"\nD10 validation images found: {len(d10_images)}")

for img in d10_images:
    print(" ", img.name)

# ------------------------------------------------------------
# 2. Show confusion matrix
# ------------------------------------------------------------

if CONFUSION_MATRIX.exists():

    print("\n📊 Confusion Matrix")

    cm = cv2.imread(str(CONFUSION_MATRIX))
    cm = cv2.cvtColor(cm, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(9, 7))
    plt.imshow(cm)
    plt.axis("off")
    plt.title("Baseline YOLO11n — Validation Confusion Matrix")
    plt.show()

else:
    print("\n⚠️ Confusion matrix not found at:")
    print(CONFUSION_MATRIX)

# ------------------------------------------------------------
# 3. Run predictions specifically on D10 images
# ------------------------------------------------------------

if d10_images:

    print("\n🔎 Running predictions on D10 images...")

    predictions = model.predict(
        source=[str(x) for x in d10_images],
        imgsz=640,
        conf=0.10,
        device=0,
        save=False,
        verbose=False
    )

    # --------------------------------------------------------
    # 4. Display every D10 image with model predictions
    # --------------------------------------------------------

    for image_path, result in zip(d10_images, predictions):

        image = cv2.imread(str(image_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        annotated = result.plot()

        plt.figure(figsize=(8, 8))
        plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        plt.title(f"D10 Ground Truth: {image_path.name}")
        plt.axis("off")
        plt.show()

        print("Predictions:")

        if len(result.boxes) == 0:
            print("  ❌ No detections")
        else:
            for box in result.boxes:
                cls_id = int(box.cls[0])
                conf = float(box.conf[0])

                print(
                    f"  {model.names[cls_id]} "
                    f"| confidence={conf:.3f}"
                )

print("\n" + "=" * 65)
print("✅ D10 ERROR ANALYSIS COMPLETE")
print("=" * 65)

In [ ]:
from pathlib import Path
from collections import Counter

TRAIN_LABELS = Path("/content/RoadVision/data/raw/train/labels")

d10_images = []
d10_objects = 0

for label_file in TRAIN_LABELS.glob("*.txt"):
    lines = label_file.read_text().strip().splitlines()

    count = sum(
        1 for line in lines
        if line.strip() and int(line.split()[0]) == 1
    )

    if count > 0:
        d10_images.append(label_file.stem)
        d10_objects += count

print("=" * 60)
print("D10 TRAINING DISTRIBUTION")
print("=" * 60)

print(f"D10 objects       : {d10_objects}")
print(f"Images containing D10: {len(d10_images)}")

print("\nSample image names:")
for x in d10_images[:15]:
    print(" ", x)

print("\n" + "=" * 60)

In [ ]:
# ============================================================
# RoadVision — Experiment 2
# D10 Targeted Augmentation Dataset
# ============================================================

import cv2
import numpy as np
from pathlib import Path
import random
import shutil
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

ROOT = Path("/content/RoadVision/data/raw")

TRAIN_IMAGES = ROOT / "train/images"
TRAIN_LABELS = ROOT / "train/labels"

AUG_ROOT = Path("/content/RoadVision/data/augmented_d10")
AUG_IMAGES = AUG_ROOT / "images"
AUG_LABELS = AUG_ROOT / "labels"

AUG_IMAGES.mkdir(parents=True, exist_ok=True)
AUG_LABELS.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

TARGET_AUGMENTED_PER_IMAGE = 3

random.seed(42)
np.random.seed(42)

# ------------------------------------------------------------
# Find D10 images
# ------------------------------------------------------------

d10_files = []

for label_file in TRAIN_LABELS.glob("*.txt"):

    lines = label_file.read_text().strip().splitlines()

    if any(
        line.strip() and int(line.split()[0]) == 1
        for line in lines
    ):
        image_file = None

        for ext in [".jpg", ".jpeg", ".png"]:
            candidate = TRAIN_IMAGES / f"{label_file.stem}{ext}"
            if candidate.exists():
                image_file = candidate
                break

        if image_file:
            d10_files.append((image_file, label_file))

print("=" * 65)
print("D10 TARGETED AUGMENTATION")
print("=" * 65)

print(f"D10 source images: {len(d10_files)}")
print(f"Augmentations per image: {TARGET_AUGMENTED_PER_IMAGE}")
print(
    f"New augmented images: "
    f"{len(d10_files) * TARGET_AUGMENTED_PER_IMAGE}"
)

# ------------------------------------------------------------
# Augmentation function
# ------------------------------------------------------------

def augment_image(image, labels):

    h, w = image.shape[:2]

    # 1. Mild brightness/contrast
    alpha = random.uniform(0.85, 1.15)
    beta = random.randint(-20, 20)

    aug = cv2.convertScaleAbs(
        image,
        alpha=alpha,
        beta=beta
    )

    # 2. Mild horizontal flip
    if random.random() < 0.5:

        aug = cv2.flip(aug, 1)

        new_labels = []

        for cls, xc, yc, bw, bh in labels:

            xc = 1.0 - xc

            new_labels.append(
                [cls, xc, yc, bw, bh]
            )

        labels = new_labels

    # 3. Mild Gaussian blur
    if random.random() < 0.20:

        aug = cv2.GaussianBlur(
            aug,
            (3, 3),
            0
        )

    # 4. Mild scaling
    if random.random() < 0.30:

        scale = random.uniform(0.90, 1.10)

        M = np.array([
            [scale, 0, w * (1 - scale) / 2],
            [0, scale, h * (1 - scale) / 2]
        ], dtype=np.float32)

        aug = cv2.warpAffine(
            aug,
            M,
            (w, h),
            borderMode=cv2.BORDER_REFLECT_101
        )

        new_labels = []

        for cls, xc, yc, bw, bh in labels:

            xc_new = (
                xc * w * scale
                + w * (1 - scale) / 2
            ) / w

            yc_new = (
                yc * h * scale
                + h * (1 - scale) / 2
            ) / h

            bw_new = bw * scale
            bh_new = bh * scale

            # Keep boxes valid
            xc_new = np.clip(xc_new, 0.0, 1.0)
            yc_new = np.clip(yc_new, 0.0, 1.0)
            bw_new = np.clip(bw_new, 0.001, 1.0)
            bh_new = np.clip(bh_new, 0.001, 1.0)

            new_labels.append([
                cls,
                xc_new,
                yc_new,
                bw_new,
                bh_new
            ])

        labels = new_labels

    return aug, labels


# ------------------------------------------------------------
# Generate augmented images
# ------------------------------------------------------------

generated = []

for image_path, label_path in d10_files:

    image = cv2.imread(str(image_path))

    if image is None:
        continue

    # Read YOLO labels
    labels = []

    for line in label_path.read_text().strip().splitlines():

        parts = line.split()

        if len(parts) != 5:
            continue

        labels.append([
            int(parts[0]),
            float(parts[1]),
            float(parts[2]),
            float(parts[3]),
            float(parts[4])
        ])

    for i in range(TARGET_AUGMENTED_PER_IMAGE):

        aug_image, aug_labels = augment_image(
            image.copy(),
            labels.copy()
        )

        stem = f"{image_path.stem}_d10aug_{i+1}"

        out_image = AUG_IMAGES / f"{stem}.jpg"
        out_label = AUG_LABELS / f"{stem}.txt"

        cv2.imwrite(
            str(out_image),
            aug_image,
            [cv2.IMWRITE_JPEG_QUALITY, 95]
        )

        with open(out_label, "w") as f:

            for cls, xc, yc, bw, bh in aug_labels:

                f.write(
                    f"{cls} "
                    f"{xc:.6f} "
                    f"{yc:.6f} "
                    f"{bw:.6f} "
                    f"{bh:.6f}\n"
                )

        generated.append((out_image, out_label))


print("\n✅ Generation complete!")
print(f"Generated: {len(generated)} images")

# ------------------------------------------------------------
# Validate generated labels
# ------------------------------------------------------------

invalid = 0
d10_objects = 0

for image_path, label_path in generated:

    for line in label_path.read_text().strip().splitlines():

        p = line.split()

        if len(p) != 5:
            invalid += 1
            continue

        cls, xc, yc, bw, bh = map(float, p)

        if not (
            0 <= cls <= 3
            and 0 <= xc <= 1
            and 0 <= yc <= 1
            and 0 < bw <= 1
            and 0 < bh <= 1
        ):
            invalid += 1

        if int(cls) == 1:
            d10_objects += 1

print(f"Invalid labels: {invalid}")
print(f"D10 objects in augmented set: {d10_objects}")

# ------------------------------------------------------------
# Visual verification
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("VISUAL CHECK — AUGMENTED D10 SAMPLES")
print("=" * 65)

samples = generated[:8]

fig, axes = plt.subplots(
    2, 4,
    figsize=(16, 8)
)

for ax, (image_path, label_path) in zip(
    axes.ravel(),
    samples
):

    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    h, w = image.shape[:2]

    for line in label_path.read_text().strip().splitlines():

        cls, xc, yc, bw, bh = map(
            float,
            line.split()
        )

        x1 = int((xc - bw / 2) * w)
        y1 = int((yc - bh / 2) * h)
        x2 = int((xc + bw / 2) * w)
        y2 = int((yc + bh / 2) * h)

        ax.add_patch(
            plt.Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                fill=False,
                linewidth=2
            )
        )

        ax.text(
            x1,
            max(0, y1 - 5),
            "D10",
            fontsize=9
        )

    ax.imshow(image)
    ax.set_title(image_path.name[:18])
    ax.axis("off")

plt.tight_layout()
plt.show()

print("\n" + "=" * 65)
print("🚦 AUGMENTATION DATASET READY FOR REVIEW")
print("=" * 65)

In [ ]:
# ============================================================
# RoadVision — EXPERIMENT 2
# D10 Targeted Augmentation
# ============================================================

from pathlib import Path
import shutil
import yaml
import torch
from ultralytics import YOLO

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

ROOT = Path("/content/RoadVision/data/raw")

ORIG_IMAGES = ROOT / "train/images"
ORIG_LABELS = ROOT / "train/labels"

AUG_IMAGES = Path("/content/RoadVision/data/augmented_d10/images")
AUG_LABELS = Path("/content/RoadVision/data/augmented_d10/labels")

EXP2_ROOT = Path("/content/RoadVision/data/experiment2")

EXP2_IMAGES = EXP2_ROOT / "train/images"
EXP2_LABELS = EXP2_ROOT / "train/labels"

EXP2_IMAGES.mkdir(parents=True, exist_ok=True)
EXP2_LABELS.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Create Experiment 2 training set
# ------------------------------------------------------------

print("=" * 65)
print("BUILDING EXPERIMENT 2 DATASET")
print("=" * 65)

# Copy original training data
print("\n📦 Copying original training set...")

for image in ORIG_IMAGES.iterdir():
    if image.is_file():
        shutil.copy2(
            image,
            EXP2_IMAGES / image.name
        )

for label in ORIG_LABELS.iterdir():
    if label.is_file():
        shutil.copy2(
            label,
            EXP2_LABELS / label.name
        )

# Add D10 augmented images
print("➕ Adding D10 augmented images...")

aug_count = 0

for image in AUG_IMAGES.iterdir():

    if image.is_file():

        shutil.copy2(
            image,
            EXP2_IMAGES / image.name
        )

        label = AUG_LABELS / f"{image.stem}.txt"

        if label.exists():

            shutil.copy2(
                label,
                EXP2_LABELS / label.name
            )

            aug_count += 1

print(f"\nOriginal training images : {len(list(ORIG_IMAGES.iterdir()))}")
print(f"D10 augmented images     : {aug_count}")
print(
    f"Experiment 2 total       : "
    f"{len(list(EXP2_IMAGES.iterdir()))}"
)

# ------------------------------------------------------------
# 2. Create separate YAML
# ------------------------------------------------------------

EXP2_YAML = EXP2_ROOT / "data.yaml"

config = {
    "train": str(EXP2_IMAGES),
    "val": str(ROOT / "valid/images"),
    "test": str(ROOT / "test/images"),
    "nc": 4,
    "names": ["D00", "D10", "D20", "D40"]
}

with open(EXP2_YAML, "w") as f:
    yaml.safe_dump(
        config,
        f,
        sort_keys=False
    )

print("\n✅ Experiment 2 dataset ready")
print("YAML:", EXP2_YAML)

# ------------------------------------------------------------
# 3. GPU check
# ------------------------------------------------------------

assert torch.cuda.is_available(), "❌ GPU unavailable!"

print("\n🚀 GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# 4. Train from SAME pretrained YOLO11n
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("🚀 STARTING EXPERIMENT 2")
print("D10 TARGETED AUGMENTATION")
print("=" * 65)

model = YOLO("yolo11n.pt")

results = model.train(

    data=str(EXP2_YAML),

    # Same core settings as baseline
    epochs=30,
    imgsz=640,
    batch=16,

    device=0,
    workers=2,

    patience=10,

    project="/content/RoadVision/runs/detect",
    name="experiment2_d10_aug",
    exist_ok=True,

    save=True,
    cache=False,

    seed=42,
    plots=True,
    verbose=True
)

print("\n" + "=" * 65)
print("✅ EXPERIMENT 2 TRAINING COMPLETE")
print("=" * 65)

BEST = Path(
    "/content/RoadVision/runs/detect/"
    "experiment2_d10_aug/weights/best.pt"
)

print("\nBest model:")
print(BEST)

print("Exists:", BEST.exists())

# ------------------------------------------------------------
# 5. Save best model to Drive
# ------------------------------------------------------------

DRIVE_DIR = Path(
    "/content/drive/MyDrive/RoadVision/experiment2_d10_aug"
)

DRIVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

if BEST.exists():

    shutil.copy2(
        BEST,
        DRIVE_DIR / "best.pt"
    )

    print("\n☁️ Best model saved to:")
    print(DRIVE_DIR)

print("\n🎯 NEXT: Evaluate Experiment 2 on the SAME validation set")

In [ ]:
# ============================================================
# RoadVision — D10 TRAINING DATA ERROR ANALYSIS
# Original D10 training images + model predictions
# ============================================================

from ultralytics import YOLO
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

MODEL_PATH = (
    "/content/RoadVision/runs/detect/"
    "experiment2_d10_aug/weights/best.pt"
)

TRAIN_IMAGES = Path(
    "/content/RoadVision/data/raw/train/images"
)

TRAIN_LABELS = Path(
    "/content/RoadVision/data/raw/train/labels"
)

model = YOLO(MODEL_PATH)

# ------------------------------------------------------------
# Find ALL original training images containing D10
# ------------------------------------------------------------

d10_samples = []

for label_file in sorted(TRAIN_LABELS.glob("*.txt")):

    lines = label_file.read_text().strip().splitlines()

    has_d10 = any(
        line.strip()
        and int(line.split()[0]) == 1
        for line in lines
    )

    if not has_d10:
        continue

    image_path = None

    for ext in [".jpg", ".jpeg", ".png"]:

        candidate = TRAIN_IMAGES / (
            label_file.stem + ext
        )

        if candidate.exists():
            image_path = candidate
            break

    if image_path:
        d10_samples.append(
            (image_path, label_file)
        )

print("=" * 70)
print("D10 TRAINING IMAGE INSPECTION")
print("=" * 70)

print(f"\nD10 training images: {len(d10_samples)}")

# ------------------------------------------------------------
# Run predictions
# ------------------------------------------------------------

results = model.predict(
    source=[str(x[0]) for x in d10_samples],
    imgsz=640,
    conf=0.10,
    device=0,
    verbose=False,
    save=False
)

# ------------------------------------------------------------
# Class names
# ------------------------------------------------------------

names = {
    0: "D00",
    1: "D10",
    2: "D20",
    3: "D40"
}

# ------------------------------------------------------------
# Display 24 images in batches of 8
# ------------------------------------------------------------

for start in range(0, len(d10_samples), 8):

    batch_samples = d10_samples[start:start + 8]
    batch_results = results[start:start + 8]

    fig, axes = plt.subplots(
        2,
        4,
        figsize=(18, 9)
    )

    axes = axes.ravel()

    for ax, (image_path, label_path), result in zip(
        axes,
        batch_samples,
        batch_results
    ):

        # Read image
        image = cv2.imread(str(image_path))
        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        h, w = image.shape[:2]

        ax.imshow(image)

        # ----------------------------------------------------
        # Draw GROUND TRUTH boxes
        # ----------------------------------------------------

        for line in label_path.read_text().strip().splitlines():

            parts = line.split()

            if len(parts) != 5:
                continue

            cls, xc, yc, bw, bh = map(
                float,
                parts
            )

            # Only highlight D10
            if int(cls) != 1:
                continue

            x1 = int((xc - bw / 2) * w)
            y1 = int((yc - bh / 2) * h)

            x2 = int((xc + bw / 2) * w)
            y2 = int((yc + bh / 2) * h)

            ax.add_patch(
                plt.Rectangle(
                    (x1, y1),
                    x2 - x1,
                    y2 - y1,
                    fill=False,
                    linewidth=3
                )
            )

            ax.text(
                x1,
                max(10, y1 - 5),
                "GT: D10",
                fontsize=9,
                backgroundcolor="white"
            )

        # ----------------------------------------------------
        # Draw MODEL predictions
        # ----------------------------------------------------

        prediction_count = 0

        for box in result.boxes:

            cls_id = int(box.cls[0])
            confidence = float(box.conf[0])

            x1, y1, x2, y2 = (
                box.xyxy[0]
                .cpu()
                .numpy()
                .astype(int)
            )

            ax.add_patch(
                plt.Rectangle(
                    (x1, y1),
                    x2 - x1,
                    y2 - y1,
                    fill=False,
                    linewidth=2,
                    linestyle="--"
                )
            )

            ax.text(
                x1,
                min(h - 5, y2 + 15),
                f"P: {names[cls_id]} {confidence:.2f}",
                fontsize=8,
                backgroundcolor="white"
            )

            prediction_count += 1

        ax.set_title(
            f"{image_path.name[:22]}\n"
            f"Predictions: {prediction_count}"
        )

        ax.axis("off")

    # Hide unused axes
    for ax in axes[len(batch_samples):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------
# Prediction summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("D10 PREDICTION SUMMARY")
print("=" * 70)

predicted_classes = []

for image_path, result in zip(
    [x[0] for x in d10_samples],
    results
):

    classes_here = []

    for box in result.boxes:

        cls_id = int(box.cls[0])
        confidence = float(box.conf[0])

        classes_here.append(
            f"{names[cls_id]} ({confidence:.2f})"
        )

        predicted_classes.append(
            names[cls_id]
        )

    print(
        f"\n{image_path.name}:"
    )

    if classes_here:
        print("  " + ", ".join(classes_here))
    else:
        print("  ❌ No predictions")

print("\n" + "=" * 70)
print("Predicted class frequency:")
print("=" * 70)

for cls in names.values():
    print(
        f"{cls}: {predicted_classes.count(cls)}"
    )

print("\n✅ D10 TRAINING DATA ANALYSIS COMPLETE")

In [ ]:
# ============================================================
# RoadVision — EXPERIMENT 3
# Higher Resolution: 704x704
# ============================================================

from ultralytics import YOLO
from pathlib import Path
import torch
import shutil

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

DATA_YAML = "/content/RoadVision/data/raw/data.yaml"

RUN_NAME = "experiment3_highres704"
RUN_DIR = Path(
    f"/content/RoadVision/runs/detect/{RUN_NAME}"
)

# ------------------------------------------------------------
# GPU check
# ------------------------------------------------------------

assert torch.cuda.is_available(), "❌ GPU unavailable!"

print("=" * 65)
print("ROADVISION — EXPERIMENT 3")
print("HIGH-RESOLUTION TRAINING: 704x704")
print("=" * 65)

print("\n🚀 GPU:", torch.cuda.get_device_name(0))
print("💾 VRAM:",
      round(
          torch.cuda.get_device_properties(0).total_memory
          / 1024**3,
          1
      ),
      "GB")

# ------------------------------------------------------------
# Load SAME pretrained YOLO11n
# ------------------------------------------------------------

model = YOLO("yolo11n.pt")

print("\n📦 Model: YOLO11n")
print("📐 Image size: 704")
print("📚 Dataset: ORIGINAL training set")
print("🔬 Experiment: Higher resolution")
print()

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

results = model.train(

    data=DATA_YAML,

    # Same baseline training budget
    epochs=30,
    imgsz=704,
    batch=16,

    device=0,
    workers=2,

    patience=10,

    project="/content/RoadVision/runs/detect",
    name=RUN_NAME,
    exist_ok=True,

    save=True,
    cache=False,

    seed=42,
    plots=True,
    verbose=True
)

# ------------------------------------------------------------
# Locate best model
# ------------------------------------------------------------

BEST_MODEL = RUN_DIR / "weights" / "best.pt"

print("\n" + "=" * 65)
print("✅ EXPERIMENT 3 TRAINING COMPLETE")
print("=" * 65)

print("\nBest model:")
print(BEST_MODEL)

print("Exists:", BEST_MODEL.exists())

# ------------------------------------------------------------
# Save to Google Drive
# ------------------------------------------------------------

DRIVE_DIR = Path(
    "/content/drive/MyDrive/RoadVision/"
    "experiment3_highres704"
)

DRIVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

if BEST_MODEL.exists():

    shutil.copy2(
        BEST_MODEL,
        DRIVE_DIR / "best.pt"
    )

    results_csv = RUN_DIR / "results.csv"

    if results_csv.exists():
        shutil.copy2(
            results_csv,
            DRIVE_DIR / "results.csv"
        )

    print("\n☁️ Saved to Google Drive:")
    print(DRIVE_DIR)

print("\n🎯 NEXT: Evaluate Experiment 3 on the SAME validation set.")

In [ ]:
from pathlib import Path
import yaml
import cv2
import numpy as np
import pandas as pd

# Dataset paths
ROOT = Path("/content/RoadVision/data/raw")
VAL_IMAGES = ROOT / "valid/images"
VAL_LABELS = ROOT / "valid/labels"

CLASS_NAMES = ["D00", "D10", "D20", "D40"]
D10_ID = 1

rows = []

for label_file in sorted(VAL_LABELS.glob("*.txt")):
    lines = label_file.read_text().strip().splitlines()

    for line in lines:
        parts = line.split()
        if len(parts) != 5:
            continue

        cls, xc, yc, w, h = map(float, parts)

        if int(cls) != D10_ID:
            continue

        image_file = VAL_IMAGES / f"{label_file.stem}.jpg"

        if not image_file.exists():
            for ext in [".png", ".jpeg", ".JPG", ".JPEG"]:
                candidate = VAL_IMAGES / f"{label_file.stem}{ext}"
                if candidate.exists():
                    image_file = candidate
                    break

        img = cv2.imread(str(image_file))

        if img is None:
            continue

        H, W = img.shape[:2]

        box_w = w * W
        box_h = h * H
        area = box_w * box_h
        area_pct = (area / (W * H)) * 100

        rows.append({
            "image": image_file.name,
            "width_px": round(box_w, 2),
            "height_px": round(box_h, 2),
            "area_px": round(area, 2),
            "area_%": round(area_pct, 4),
            "center_x_%": round(xc * 100, 2),
            "center_y_%": round(yc * 100, 2)
        })

df = pd.DataFrame(rows)

print("=" * 70)
print("D10 VALIDATION BOX ANALYSIS")
print("=" * 70)

print(f"\nD10 instances found: {len(df)}")

if len(df):
    print("\nIndividual D10 boxes:")
    display(df)

    print("\nSummary:")
    print(df[["width_px", "height_px", "area_px", "area_%"]].describe())

    print("\nSmallest D10 boxes:")
    display(df.sort_values("area_px").head())

    print("\nLargest D10 boxes:")
    display(df.sort_values("area_px", ascending=False).head())
else:
    print("No D10 instances found.")

In [ ]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import math

ROOT = Path("/content/RoadVision/data/raw")
IMG_DIR = ROOT / "valid/images"
LBL_DIR = ROOT / "valid/labels"

d10_images = []

for label_file in sorted(LBL_DIR.glob("*.txt")):
    lines = label_file.read_text().strip().splitlines()

    if any(line.split()[0] == "1" for line in lines if line.strip()):
        img_path = IMG_DIR / f"{label_file.stem}.jpg"

        if not img_path.exists():
            for ext in [".png", ".jpeg", ".JPG", ".JPEG"]:
                candidate = IMG_DIR / f"{label_file.stem}{ext}"
                if candidate.exists():
                    img_path = candidate
                    break

        if img_path.exists():
            d10_images.append((img_path, label_file))

print(f"D10 validation images: {len(d10_images)}")

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

for ax, (img_path, label_file) in zip(axes, d10_images):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    H, W = img.shape[:2]

    for line in label_file.read_text().strip().splitlines():
        parts = line.split()

        if len(parts) != 5:
            continue

        cls, xc, yc, w, h = map(float, parts)

        if int(cls) != 1:
            continue

        x1 = int((xc - w / 2) * W)
        y1 = int((yc - h / 2) * H)
        x2 = int((xc + w / 2) * W)
        y2 = int((yc + h / 2) * H)

        rect = plt.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            fill=False,
            linewidth=2
        )

        ax.add_patch(rect)

    ax.imshow(img)
    ax.set_title(img_path.name[:35])
    ax.axis("off")

for ax in axes[len(d10_images):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import matplotlib.pyplot as plt

MODEL = "/content/RoadVision/runs/detect/baseline_yolo11n/weights/best.pt"
IMG_DIR = Path("/content/RoadVision/data/raw/valid/images")

model = YOLO(MODEL)

# The 5 validation images containing D10
d10_names = [
    "India_001495_jpg.rf.2cf318ffb3fce9a7d1b96c6f6c197c72.jpg",
    "India_006255_jpg.rf.7dc9b77530a19e1cce8bd8507db24192.jpg",
    "India_006490_jpg.rf.03084194c6dd3091b65bc0b42707467c.jpg",
    "India_007116_jpg.rf.e59e1d2126868e4364888c56efd166b6.jpg",
    "India_008133_jpg.rf.3774f38accc008aaf4d61c1f9b613a03.jpg"
]

paths = [IMG_DIR / x for x in d10_names]

results = model.predict(
    source=[str(p) for p in paths],
    conf=0.10,
    imgsz=640,
    device=0,
    verbose=False
)

class_names = model.names

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

for ax, result, path in zip(axes, results, paths):

    plotted = result.plot()
    plotted = plotted[:, :, ::-1]

    ax.imshow(plotted)
    ax.set_title(path.name[:38])
    ax.axis("off")

    print("\n" + "=" * 70)
    print(path.name)

    if len(result.boxes) == 0:
        print("NO PREDICTIONS")
    else:
        for cls, conf in zip(
            result.boxes.cls.tolist(),
            result.boxes.conf.tolist()
        ):
            print(
                f"{class_names[int(cls)]}: "
                f"{conf:.3f}"
            )

for ax in axes[len(results):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import zipfile

# Find uploaded ZIP
zip_files = list(Path("/content").glob("*.zip"))

print("ZIP files found:")
for z in zip_files:
    print(f"  {z.name} — {z.stat().st_size / (1024**2):.2f} MB")

if not zip_files:
    raise FileNotFoundError("ZIP nahi mila. Colab Files panel check karo.")

ZIP_PATH = zip_files[-1]

# Extract
EXTRACT_DIR = Path("/content/RoadVision/data/custom_city")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

print("\nExtracted to:")
print(EXTRACT_DIR)

# Find images recursively
extensions = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}

images = [
    p for p in EXTRACT_DIR.rglob("*")
    if p.is_file() and p.suffix in extensions
]

print(f"\n📸 Images found: {len(images)}")

# Show first few
print("\nFirst 10 images:")
for img in images[:10]:
    print(img)

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import random

IMG_DIR = Path("/content/RoadVision/data/custom_city")

extensions = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}

images = [
    p for p in IMG_DIR.rglob("*")
    if p.is_file() and p.suffix in extensions
]

print(f"Total images: {len(images)}")

# Random sample of 12 images
sample = random.sample(images, min(12, len(images)))

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for ax, img_path in zip(axes, sample):
    img = Image.open(img_path)
    ax.imshow(img)
    ax.set_title(img_path.name[:30])
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

list(Path("/content").glob("*.zip"))

In [ ]:
from pathlib import Path

zip_files = list(Path("/").glob("*.zip"))
print(zip_files)

In [ ]:
from pathlib import Path

print(list(Path("/").glob("*.zip")))

In [ ]:
from pathlib import Path

zip_files = list(Path("/content").glob("*.zip"))
print(zip_files)

In [ ]:
from pathlib import Path
import zipfile

IMG_ZIP = Path("/content/custom_city_images.zip.zip")
IMG_DIR = Path("/content/custom_city_images")
IMG_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(IMG_ZIP, "r") as z:
    z.extractall(IMG_DIR)

images = [
    p for p in IMG_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
]

print("Original images found:", len(images))
print("First 5:")
for p in images[:5]:
    print(p)

In [ ]:
from pathlib import Path
import zipfile

CVAT_ZIP = Path("/content/roadvision_custom_city (1).zip")
CVAT_DIR = Path("/content/cvat_annotations")

CVAT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(CVAT_ZIP, "r") as z:
    z.extractall(CVAT_DIR)

print("Extracted!")
print("train.txt exists:", (CVAT_DIR / "train.txt").exists())
print("Contents:")
for p in CVAT_DIR.iterdir():
    print(" ", p)

In [ ]:
from pathlib import Path

IMG_ROOT = Path("/content/custom_city_images")

original_images = {
    p.name: p
    for p in IMG_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
}

train_txt = Path("/content/cvat_annotations/train.txt")

cvat_names = []
with train_txt.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            cvat_names.append(Path(line).name)

print("Original images:", len(original_images))
print("CVAT image entries:", len(cvat_names))

matched = [name for name in cvat_names if name in original_images]
missing = [name for name in cvat_names if name not in original_images]

print("Matched:", len(matched))
print("Missing:", len(missing))

if missing:
    print("\nFirst missing names:")
    for x in missing[:10]:
        print(x)

In [ ]:
from pathlib import Path
import shutil

IMG_ROOT = Path("/content/custom_city_images")
CVAT_LABELS = Path("/content/cvat_annotations/labels/train")

EVAL_ROOT = Path("/content/RoadVision/data/custom_city_eval")
IMAGES_DIR = EVAL_ROOT / "images"
LABELS_DIR = EVAL_ROOT / "labels"

IMAGES_DIR.mkdir(parents=True, exist_ok=True)
LABELS_DIR.mkdir(parents=True, exist_ok=True)

original_images = [
    p for p in IMG_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
]

for img in original_images:
    # Copy original image
    shutil.copy2(img, IMAGES_DIR / img.name)

    # Copy annotation if it exists; otherwise create empty label
    src_label = CVAT_LABELS / f"{img.stem}.txt"
    dst_label = LABELS_DIR / f"{img.stem}.txt"

    if src_label.exists():
        shutil.copy2(src_label, dst_label)
    else:
        dst_label.write_text("")

print("Images:", len(list(IMAGES_DIR.iterdir())))
print("Labels:", len(list(LABELS_DIR.iterdir())))

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import yaml

MODEL = "/content/RoadVision/runs/detect/baseline_yolo11n/weights/best.pt"

DATA_DIR = Path("/content/RoadVision/data/custom_city_eval")

data_yaml = DATA_DIR / "data.yaml"

data = {
    "path": str(DATA_DIR),
    "val": "images",
    "nc": 4,
    "names": ["D00", "D10", "D20", "D40"]
}

with open(data_yaml, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

model = YOLO(MODEL)

results = model.val(
    data=str(data_yaml),
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    plots=True,
    project="/content/RoadVision/runs/detect",
    name="custom_city_baseline"
)

print("\n===== CUSTOM CITY RESULTS =====")
print("Precision:", results.box.mp)
print("Recall:", results.box.mr)
print("mAP50:", results.box.map50)
print("mAP50-95:", results.box.map)

In [ ]:
!pip install -q ultralytics

In [ ]:
from pathlib import Path

# Find the saved baseline model in Google Drive
matches = list(Path("/content/drive/MyDrive").rglob("best.pt"))

print("Found models:")
for p in matches:
    print(p)

In [ ]:
from pathlib import Path

models = list(Path("/content").rglob("*.pt"))

print("PT files found:", len(models))
for p in models:
    print(p)

In [ ]:
from pathlib import Path

drive = Path("/content/drive/MyDrive")
print("Drive exists:", drive.exists())
print("RoadVision exists:", (drive / "RoadVision").exists())

if (drive / "RoadVision").exists():
    print(list((drive / "RoadVision").iterdir()))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

drive = Path("/content/drive/MyDrive")
print("Drive:", drive.exists())
print("RoadVision:", (drive / "RoadVision").exists())

In [ ]:
from pathlib import Path

roadvision = Path("/content/drive/MyDrive/RoadVision")

for p in roadvision.rglob("*.pt"):
    print(p)

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import yaml

MODEL = "/content/drive/MyDrive/RoadVision/baseline_yolo11n/best.pt"
DATA_DIR = Path("/content/RoadVision/data/custom_city_eval")

data_yaml = DATA_DIR / "data.yaml"

data = {
    "path": str(DATA_DIR),
    "val": "images",
    "nc": 4,
    "names": ["D00", "D10", "D20", "D40"]
}

with open(data_yaml, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

model = YOLO(MODEL)

results = model.val(
    data=str(data_yaml),
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    plots=True,
    project="/content/RoadVision/runs/detect",
    name="custom_city_baseline"
)

print("\n===== CUSTOM CITY RESULTS =====")
print("Precision:", results.box.mp)
print("Recall:", results.box.mr)
print("mAP50:", results.box.map50)
print("mAP50-95:", results.box.map)
print("\nPer-class mAP50:", results.box.maps)

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import yaml

MODEL = "/content/drive/MyDrive/RoadVision/baseline_yolo11n/best.pt"
DATA_DIR = Path("/content/RoadVision/data/custom_city_eval")

data_yaml = DATA_DIR / "data.yaml"

data = {
    "path": str(DATA_DIR),
    "train": "images",   # required by Ultralytics
    "val": "images",
    "nc": 4,
    "names": ["D00", "D10", "D20", "D40"]
}

with open(data_yaml, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

model = YOLO(MODEL)

results = model.val(
    data=str(data_yaml),
    split="val",
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    plots=True,
    project="/content/RoadVision/runs/detect",
    name="custom_city_baseline"
)

print("\n===== CUSTOM CITY RESULTS =====")
print("Precision:", results.box.mp)
print("Recall:", results.box.mr)
print("mAP50:", results.box.map50)
print("mAP50-95:", results.box.map)
print("Per-class mAP50:", results.box.maps)

In [ ]:
# ==============================================================================
# RoadVision — Baseline Qualitative & Quantitative Error Analysis
# Custom-City Dataset | DIAGNOSTIC ONLY — no training, no dataset changes
# Paste this entire cell into Google Colab (Tesla T4) and run once.
# ==============================================================================

get_ipython().system('pip install -q ultralytics --upgrade')

import os, json, random, glob

import cv2
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

# ------------------------------------------------------------------------------
# 0. DRIVE MOUNT (defensive — skips if already mounted)
# ------------------------------------------------------------------------------
try:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
except Exception as e:
    print(f"[info] Drive mount skipped/failed ({e}). Continuing — "
          f"make sure MODEL_PATH below is reachable.")

# ------------------------------------------------------------------------------
# 1. CONFIG
# ------------------------------------------------------------------------------
MODEL_PATH  = "/content/drive/MyDrive/RoadVision/baseline_yolo11n/best.pt"
IMAGES_DIR  = "/content/RoadVision/data/custom_city_eval/images"
LABELS_DIR  = "/content/RoadVision/data/custom_city_eval/labels"

OUT_DIR     = "/content/RoadVision/runs/detect/custom_city_error_analysis"
ANNOT_DIR   = os.path.join(OUT_DIR, "annotated_predictions")   # req 3
REPORT_DIR  = os.path.join(OUT_DIR, "reports")                 # req 9, 10, 11
CONTACT_DIR = os.path.join(OUT_DIR, "contact_sheets")          # req 7

CLASS_NAMES = {0: "D00", 1: "D10", 2: "D20", 3: "D40"}
NUM_CLASSES = len(CLASS_NAMES)
BG          = NUM_CLASSES  # background index in confusion matrix

IOU_MATCH_THRESHOLD = 0.50           # req 10 — explicit IoU threshold for matching
INFER_CONF_FLOOR    = 0.001          # run inference ONCE, very low conf; filter later (req 12)
PRIMARY_CONF_THR    = 0.25           # main reported operating point (req 12)
COMPARE_CONF_THRS   = [0.10, 0.25, 0.50]   # req 12 — sensitivity sweep, no re-inference needed

MAX_TILES_PER_CATEGORY = 4           # contact sheet size control (req 7)
RANDOM_SEED = 42

GT_COLOR   = (0, 200, 0)     # BGR green  — ground truth
TP_COLOR   = (255, 0, 0)     # BGR blue   — true positive prediction
FP_COLOR   = (0, 0, 255)     # BGR red    — false positive prediction
CONF_COLOR = (255, 0, 255)   # BGR magenta— IoU-matched but wrong class (class confusion)

D00, D10, D20, D40 = 0, 1, 2, 3

os.makedirs(ANNOT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(CONTACT_DIR, exist_ok=True)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

device = 0 if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ------------------------------------------------------------------------------
# 2. HELPER FUNCTIONS
# ------------------------------------------------------------------------------

def yolo_txt_to_xyxy(label_path, img_w, img_h):
    """Load a YOLO-format label file -> list of {'cls': int, 'box': (x1,y1,x2,y2)} in pixels."""
    boxes = []
    if not os.path.exists(label_path):
        return boxes
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            c = int(float(parts[0]))
            xc, yc, w, h = map(float, parts[1:5])
            x1 = (xc - w / 2) * img_w
            y1 = (yc - h / 2) * img_h
            x2 = (xc + w / 2) * img_w
            y2 = (yc + h / 2) * img_h
            if c not in CLASS_NAMES:
                continue
            boxes.append({"cls": c, "box": (x1, y1, x2, y2)})
    return boxes


def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def match_predictions(gts, preds, iou_thr):
    """
    Greedy global matching by descending IoU, across ALL class pairs (so a high-IoU
    cross-class overlap surfaces as 'class confusion' instead of being hidden).
    Returns: matches=[(gt_idx, pred_idx, iou)], unmatched_gt=[idx], unmatched_pred=[idx]
    """
    pairs = []
    for gi, g in enumerate(gts):
        for pi, p in enumerate(preds):
            iou = iou_xyxy(g["box"], p["box"])
            if iou >= iou_thr:
                pairs.append((iou, gi, pi))
    pairs.sort(key=lambda x: x[0], reverse=True)

    matched_gt, matched_pred, matches = set(), set(), []
    for iou, gi, pi in pairs:
        if gi in matched_gt or pi in matched_pred:
            continue
        matched_gt.add(gi)
        matched_pred.add(pi)
        matches.append((gi, pi, iou))

    unmatched_gt = [i for i in range(len(gts)) if i not in matched_gt]
    unmatched_pred = [i for i in range(len(preds)) if i not in matched_pred]
    return matches, unmatched_gt, unmatched_pred


def analyze_at_threshold(conf_thr, iou_thr=IOU_MATCH_THRESHOLD):
    """Re-filters the cached raw (low-conf) predictions at conf_thr and rematches. No re-inference."""
    confusion = np.zeros((NUM_CLASSES + 1, NUM_CLASSES + 1), dtype=int)
    records = []
    per_image = {}

    for fname, preds_all in raw_preds_cache.items():
        gts = gt_cache[fname]
        preds = [p for p in preds_all if p["conf"] >= conf_thr]
        matches, unmatched_gt, unmatched_pred = match_predictions(gts, preds, iou_thr)
        per_image[fname] = (matches, unmatched_gt, unmatched_pred, gts, preds)

        for gi, pi, iou in matches:
            g, p = gts[gi], preds[pi]
            confusion[g["cls"], p["cls"]] += 1
            err = "TP" if g["cls"] == p["cls"] else "class_confusion"
            records.append({
                "image": fname, "gt_class": CLASS_NAMES[g["cls"]],
                "pred_class": CLASS_NAMES[p["cls"]], "iou": round(iou, 3),
                "confidence": round(p["conf"], 3), "error_type": err,
            })
        for gi in unmatched_gt:
            g = gts[gi]
            confusion[g["cls"], BG] += 1
            records.append({
                "image": fname, "gt_class": CLASS_NAMES[g["cls"]],
                "pred_class": "None", "iou": 0.0, "confidence": None,
                "error_type": "FN",
            })
        for pi in unmatched_pred:
            p = preds[pi]
            confusion[BG, p["cls"]] += 1
            records.append({
                "image": fname, "gt_class": "None",
                "pred_class": CLASS_NAMES[p["cls"]], "iou": 0.0,
                "confidence": round(p["conf"], 3), "error_type": "FP",
            })
    return records, confusion, per_image


def classwise_stats(confusion):
    rows = []
    for c in range(NUM_CLASSES):
        gt_count = int(confusion[c, :].sum())
        tp = int(confusion[c, c])
        fn = gt_count - tp
        pred_count = int(confusion[:, c].sum())
        fp = pred_count - tp
        precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
        recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
        rows.append({
            "class": CLASS_NAMES[c], "gt_instances": gt_count,
            "TP": tp, "FN": fn, "FP": fp,
            "precision": round(precision, 4) if precision == precision else None,
            "recall": round(recall, 4) if recall == recall else None,
        })
    return pd.DataFrame(rows)


def draw_annotated(fname, matches, unmatched_gt, unmatched_pred, gts, preds):
    img = cv2.imread(os.path.join(IMAGES_DIR, fname))
    if img is None:
        return None

    for gi, g in enumerate(gts):
        x1, y1, x2, y2 = map(int, g["box"])
        is_fn = gi in unmatched_gt
        thickness = 3 if is_fn else 1
        cv2.rectangle(img, (x1, y1), (x2, y2), GT_COLOR, thickness)
        label = f'GT:{CLASS_NAMES[g["cls"]]}' + (" [FN]" if is_fn else "")
        cv2.putText(img, label, (x1, max(y1 - 6, 12)), cv2.FONT_HERSHEY_SIMPLEX,
                    0.45, GT_COLOR, 1, cv2.LINE_AA)

    match_lookup = {pi: (gi, iou) for gi, pi, iou in matches}
    for pi, p in enumerate(preds):
        x1, y1, x2, y2 = map(int, p["box"])
        if pi in match_lookup:
            gi, iou = match_lookup[pi]
            same_class = gts[gi]["cls"] == p["cls"]
            color = TP_COLOR if same_class else CONF_COLOR
            tag = "TP" if same_class else f'CONF(gt={CLASS_NAMES[gts[gi]["cls"]]})'
        else:
            color, tag = FP_COLOR, "FP"
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        label = f'{CLASS_NAMES[p["cls"]]} {p["conf"]:.2f} [{tag}]'
        cv2.putText(img, label, (x1, min(y2 + 16, img.shape[0] - 4)), cv2.FONT_HERSHEY_SIMPLEX,
                    0.45, color, 1, cv2.LINE_AA)
    return img


def collect_category_images(per_image, category, target_class=None):
    found = []
    for fname, (matches, unmatched_gt, unmatched_pred, gts, preds) in per_image.items():
        if category == "FN":
            for gi in unmatched_gt:
                if target_class is None or gts[gi]["cls"] == target_class:
                    found.append(fname)
                    break
        elif category == "FP":
            if len(unmatched_pred) > 0:
                found.append(fname)
        elif category == "class_confusion":
            for gi, pi, _ in matches:
                if gts[gi]["cls"] != preds[pi]["cls"]:
                    found.append(fname)
                    break
        elif category == "TP":
            for gi, pi, _ in matches:
                if gts[gi]["cls"] == preds[pi]["cls"]:
                    found.append(fname)
                    break
    return found


# ------------------------------------------------------------------------------
# 3. LOAD MODEL + RUN INFERENCE ONCE (very low conf floor, filtered later per-threshold)
# ------------------------------------------------------------------------------
print("Loading model:", MODEL_PATH)
model = YOLO(MODEL_PATH)
print("Model class names:", model.names)  # sanity check vs CLASS_NAMES above

image_paths = sorted([
    p for p in glob.glob(os.path.join(IMAGES_DIR, "*.*"))
    if p.lower().endswith((".jpg", ".jpeg", ".png"))
])
print(f"Found {len(image_paths)} custom-city images.")
assert len(image_paths) > 0, "No images found — check IMAGES_DIR."

print(f"Running inference once at conf>={INFER_CONF_FLOOR} (raw cache; all thresholds "
      f"below are computed by filtering this cache, no re-inference).")
raw_preds_cache, gt_cache = {}, {}

results = model.predict(
    source=image_paths, conf=INFER_CONF_FLOOR, iou=0.5, imgsz=640,
    device=device, verbose=False, stream=True,
)
for r in results:
    fname = os.path.basename(r.path)
    h, w = r.orig_shape
    preds = []
    if r.boxes is not None and len(r.boxes) > 0:
        xyxy = r.boxes.xyxy.cpu().numpy()
        cls = r.boxes.cls.cpu().numpy().astype(int)
        conf = r.boxes.conf.cpu().numpy()
        for box, c, cf in zip(xyxy, cls, conf):
            preds.append({"cls": int(c), "box": tuple(box.tolist()), "conf": float(cf)})
    raw_preds_cache[fname] = preds
    gt_cache[fname] = yolo_txt_to_xyxy(
        os.path.join(LABELS_DIR, os.path.splitext(fname)[0] + ".txt"), w, h)

print(f"Cached raw predictions + GT for {len(raw_preds_cache)} images.")

# ------------------------------------------------------------------------------
# 4. PRIMARY ANALYSIS @ PRIMARY_CONF_THR
# ------------------------------------------------------------------------------
print(f"\n=== Primary analysis @ conf={PRIMARY_CONF_THR}, IoU={IOU_MATCH_THRESHOLD} ===")
records_primary, confusion_primary, per_image_primary = analyze_at_threshold(PRIMARY_CONF_THR)
stats_primary = classwise_stats(confusion_primary)
print(stats_primary.to_string(index=False))

overall_tp = int(stats_primary["TP"].sum())
overall_fn = int(stats_primary["FN"].sum())
overall_fp = int(stats_primary["FP"].sum())
overall_p = overall_tp / (overall_tp + overall_fp) if (overall_tp + overall_fp) > 0 else float("nan")
overall_r = overall_tp / (overall_tp + overall_fn) if (overall_tp + overall_fn) > 0 else float("nan")
print(f"\nOverall (box-matching, single operating point) @ conf={PRIMARY_CONF_THR}: "
      f"TP={overall_tp} FN={overall_fn} FP={overall_fp} "
      f"Precision={overall_p:.4f} Recall={overall_r:.4f}")
print("[note] This is a single-threshold box-matching P/R, NOT the COCO-style mAP "
      "previously reported (P=0.0069, R=0.1444). Methodology differs — use for "
      "directional consistency only, not as a replacement metric.")

# ------------------------------------------------------------------------------
# 5. SAVE CSV / JSON REPORTS (req 9, 11)
# ------------------------------------------------------------------------------
csv_path = os.path.join(REPORT_DIR, f"error_analysis_conf{PRIMARY_CONF_THR}.csv")
json_path = os.path.join(REPORT_DIR, f"error_analysis_conf{PRIMARY_CONF_THR}.json")
pd.DataFrame(records_primary).to_csv(csv_path, index=False)
with open(json_path, "w") as f:
    json.dump(records_primary, f, indent=2)

classwise_csv = os.path.join(REPORT_DIR, f"classwise_summary_conf{PRIMARY_CONF_THR}.csv")
stats_primary.to_csv(classwise_csv, index=False)

conf_labels = list(CLASS_NAMES.values()) + ["background"]
confusion_df = pd.DataFrame(confusion_primary,
                             index=[f"GT:{c}" for c in conf_labels],
                             columns=[f"Pred:{c}" for c in conf_labels])
confusion_csv = os.path.join(REPORT_DIR, f"confusion_matrix_conf{PRIMARY_CONF_THR}.csv")
confusion_df.to_csv(confusion_csv)

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(confusion_primary, cmap="Blues")
ax.set_xticks(range(len(conf_labels))); ax.set_xticklabels(conf_labels, rotation=45, ha="right")
ax.set_yticks(range(len(conf_labels))); ax.set_yticklabels(conf_labels)
ax.set_xlabel("Predicted"); ax.set_ylabel("Ground truth")
ax.set_title(f"Confusion matrix @ conf={PRIMARY_CONF_THR}, IoU={IOU_MATCH_THRESHOLD}")
for i in range(len(conf_labels)):
    for j in range(len(conf_labels)):
        ax.text(j, i, int(confusion_primary[i, j]), ha="center", va="center", fontsize=8)
plt.tight_layout()
confusion_png = os.path.join(REPORT_DIR, f"confusion_matrix_conf{PRIMARY_CONF_THR}.png")
plt.savefig(confusion_png, dpi=150)
plt.close(fig)

# ------------------------------------------------------------------------------
# 6. SAVE ANNOTATED GT+PREDICTION OVERLAYS FOR ALL 117 IMAGES (req 3, 8)
# ------------------------------------------------------------------------------
print(f"\nSaving annotated GT+prediction overlays for all images -> {ANNOT_DIR}")
print("Legend: green=GT (thick+'[FN]' if missed) | blue=TP | red=FP | magenta=class confusion")
saved = 0
for fname, (matches, unmatched_gt, unmatched_pred, gts, preds) in per_image_primary.items():
    img = draw_annotated(fname, matches, unmatched_gt, unmatched_pred, gts, preds)
    if img is not None:
        cv2.imwrite(os.path.join(ANNOT_DIR, fname), img)
        saved += 1
print(f"Saved {saved} annotated images.")

# ------------------------------------------------------------------------------
# 7. CONTACT SHEET OF REPRESENTATIVE FAILURE CASES (req 6, 7)
# ------------------------------------------------------------------------------
print("\nBuilding contact sheet of representative cases...")
selection = {
    "D10 False Negatives": collect_category_images(per_image_primary, "FN", D10),
    "D20 False Negatives": collect_category_images(per_image_primary, "FN", D20),
    "D40 False Negatives": collect_category_images(per_image_primary, "FN", D40),
    "False Positives": collect_category_images(per_image_primary, "FP"),
    "Class Confusion": collect_category_images(per_image_primary, "class_confusion"),
    "Successful Detections (TP)": collect_category_images(per_image_primary, "TP"),
}
for k in selection:
    random.shuffle(selection[k])
    selection[k] = selection[k][:MAX_TILES_PER_CATEGORY]

tiles = [(cat, fn) for cat, fnames in selection.items() for fn in fnames]
contact_sheet_path = None
if len(tiles) == 0:
    print("No qualifying tiles found for contact sheet — check thresholds/data.")
else:
    cols = 4
    rows = int(np.ceil(len(tiles) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = np.array(axes).reshape(-1)
    for ax, (category, fname) in zip(axes, tiles):
        img = cv2.imread(os.path.join(ANNOT_DIR, fname))
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{category}\n{fname}", fontsize=8)
        ax.axis("off")
    for ax in axes[len(tiles):]:
        ax.axis("off")
    fig.suptitle("Green=GT (thick+FN if missed) | Blue=TP | Red=FP | Magenta=class confusion",
                 fontsize=9)
    plt.tight_layout()
    contact_sheet_path = os.path.join(CONTACT_DIR, f"contact_sheet_conf{PRIMARY_CONF_THR}.png")
    plt.savefig(contact_sheet_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Contact sheet saved -> {contact_sheet_path}")
    for cat, fnames in selection.items():
        print(f"  {cat}: {len(fnames)} tile(s) -> {fnames}")

# ------------------------------------------------------------------------------
# 8. CONFIDENCE-THRESHOLD SENSITIVITY STUDY (req 12) — reuses cached raw predictions
# ------------------------------------------------------------------------------
print(f"\n=== Confidence-threshold sensitivity study (IoU={IOU_MATCH_THRESHOLD}) ===")
threshold_rows = []
per_thr_results = {}
for thr in COMPARE_CONF_THRS:
    recs, conf_mat, _ = analyze_at_threshold(thr)
    stats = classwise_stats(conf_mat)
    per_thr_results[thr] = (recs, conf_mat, stats)
    for _, row in stats.iterrows():
        threshold_rows.append({"conf_threshold": thr, **row.to_dict()})
threshold_df = pd.DataFrame(threshold_rows)
print(threshold_df.to_string(index=False))
threshold_csv = os.path.join(REPORT_DIR, "confidence_threshold_comparison.csv")
threshold_df.to_csv(threshold_csv, index=False)

# ------------------------------------------------------------------------------
# 9. D10-SPECIFIC DIAGNOSTIC (req 6) — public D10 mAP50=0, custom-city D10 recall=0
# ------------------------------------------------------------------------------
print("\n=== D10-specific diagnostic ===")
d10_raw_confs = [p["conf"] for preds in raw_preds_cache.values() for p in preds if p["cls"] == D10]
total_d10_gt = sum(1 for gts in gt_cache.values() for g in gts if g["cls"] == D10)
print(f"Total D10 GT instances in custom-city set: {total_d10_gt}")
print(f"Total raw D10 predictions at conf>={INFER_CONF_FLOOR} (before any thresholding): "
      f"{len(d10_raw_confs)}")

if d10_raw_confs:
    arr = np.array(d10_raw_confs)
    print(f"D10 raw prediction confidence -> mean={arr.mean():.4f}, max={arr.max():.4f}, "
          f"count>=0.10={(arr >= 0.10).sum()}, count>=0.25={(arr >= 0.25).sum()}, "
          f"count>=0.50={(arr >= 0.50).sum()}")
else:
    print("Model produced ZERO D10-class raw predictions on the custom-city set, "
          "even at a near-zero confidence floor.")

d10_localized_any_class = 0
for fname, preds in raw_preds_cache.items():
    for g in gt_cache[fname]:
        if g["cls"] != D10:
            continue
        if any(iou_xyxy(g["box"], p["box"]) >= IOU_MATCH_THRESHOLD for p in preds):
            d10_localized_any_class += 1
print(f"D10 GT boxes with ANY predicted box (any class, any conf) overlapping "
      f">= IoU {IOU_MATCH_THRESHOLD}: {d10_localized_any_class} / {total_d10_gt}")

# ------------------------------------------------------------------------------
# 10. FINAL SUMMARY + DATA-DRIVEN INTERPRETATION (req: end-of-run printout)
# ------------------------------------------------------------------------------
print("\n" + "=" * 78)
print("FINAL SUMMARY")
print("=" * 78)
print(f"Annotated GT+prediction overlays : {ANNOT_DIR}")
print(f"Per-detection CSV report          : {csv_path}")
print(f"Per-detection JSON report         : {json_path}")
print(f"Class-wise summary (primary thr)  : {classwise_csv}")
print(f"Confusion matrix (CSV / PNG)      : {confusion_csv}")
print(f"                                     {confusion_png}")
print(f"Confidence-threshold comparison   : {threshold_csv}")
print(f"Contact sheet                     : {contact_sheet_path or 'not generated'}")

print(f"\nClass-wise error summary @ conf={PRIMARY_CONF_THR}, IoU={IOU_MATCH_THRESHOLD}:")
print(stats_primary.to_string(index=False))

print("\nConfidence-threshold comparison (recall/precision per class):")
print(threshold_df.to_string(index=False))

print("\nInterpretation notes (derived strictly from the numbers above — "
      "cross-check against the CSV/contact sheet before concluding):")
notes = []

r10 = threshold_df.loc[threshold_df.conf_threshold == 0.10, "recall"].mean()
r50 = threshold_df.loc[threshold_df.conf_threshold == 0.50, "recall"].mean()
if pd.notna(r10) and pd.notna(r50) and abs(r10 - r50) < 0.02:
    notes.append(f"- Mean recall barely moves between conf=0.10 ({r10:.3f}) and conf=0.50 "
                 f"({r50:.3f}) => confidence thresholding is NOT the primary bottleneck; "
                 "the model mostly isn't producing correct detections at any threshold.")
else:
    notes.append(f"- Mean recall changes from {r10:.3f} (conf=0.10) to {r50:.3f} (conf=0.50) "
                 "=> confidence thresholding is a real contributor; some correct boxes exist "
                 "but score low.")

if len(d10_raw_confs) == 0:
    notes.append("- D10: zero raw detections anywhere near conf=0 => a representation-level "
                 "failure, not a threshold/NMS issue. Consistent with public D10 mAP50=0. "
                 "Consider: D10 was also the smallest class in training data (possible "
                 "underrepresentation) and/or its visual pattern (transverse cracks) may not "
                 "transfer from the public source domain to this camera/road setting.")
else:
    notes.append(f"- D10: {len(d10_raw_confs)} raw predictions exist (mean conf "
                 f"{np.mean(d10_raw_confs):.3f}); {d10_localized_any_class}/{total_d10_gt} GT "
                 "D10 boxes have some overlapping prediction — check the confusion matrix to "
                 "see which class those predictions were labeled as.")

n_confusion = sum(1 for r in records_primary if r["error_type"] == "class_confusion")
n_tp = sum(1 for r in records_primary if r["error_type"] == "TP")
if n_confusion > 0:
    notes.append(f"- {n_confusion} boxes were localized correctly (IoU>={IOU_MATCH_THRESHOLD}) "
                 f"but classified as the wrong class, vs {n_tp} clean TPs => part of the "
                 "failure is classification confusion between damage types, not pure miss-detection.")
else:
    notes.append("- No IoU-matched cross-class confusions at this threshold => remaining "
                 "errors are dominated by missed (FN) and spurious (FP) detections, not "
                 "classification confusion.")

n_fp = sum(1 for r in records_primary if r["error_type"] == "FP")
notes.append(f"- {n_fp} false positives across {len(image_paths)} images @ conf="
             f"{PRIMARY_CONF_THR} => inspect the 'False Positives' tiles in the contact sheet: "
             "FPs on textures/shadows/lane-markings not typical of the public training data "
             "would point to domain shift; FPs tightly clustered near real damage would point "
             "to localization/NMS issues instead.")

notes.append("- Compare classwise_stats across D00/D20/D40 (which the public model detects "
             "at all) vs D10 (which it doesn't) to separate a general domain-shift problem "
             "(all classes degrade) from a D10-specific problem (only D10 collapses).")

for n in notes:
    print(n)

print("\nNo training, weights, or dataset files were modified by this script.")

In [ ]:
# ==============================================================================
# RoadVision — Leakage-Free, Class-Aware Train/Val/Test Split (custom-city set)
# NO TRAINING. NO FINE-TUNING. Paste this whole cell into Colab and run once.
# ==============================================================================

get_ipython().system('pip install -q iterative-stratification')

import os, json, glob, random, hashlib, shutil
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# ------------------------------------------------------------------------------
# 1. CONFIG
# ------------------------------------------------------------------------------
SRC_IMAGES_DIR = "/content/RoadVision/data/custom_city_eval/images"
SRC_LABELS_DIR = "/content/RoadVision/data/custom_city_eval/labels"
SPLIT_ROOT     = "/content/RoadVision/data/custom_city_split"

CLASS_NAMES = {0: "D00", 1: "D10", 2: "D20", 3: "D40"}
CLASS_COLS  = list(CLASS_NAMES.values())  # ["D00","D10","D20","D40"]

SEED = 42
RATIOS = (0.70, 0.15, 0.15)  # train, val, test

random.seed(SEED)
np.random.seed(SEED)

# ------------------------------------------------------------------------------
# 2. HELPER FUNCTIONS
# ------------------------------------------------------------------------------

def file_md5(path, chunk=8192):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def validate_label_file(path):
    """Returns (is_valid, boxes=[(cls,xc,yc,w,h),...], issues=[str,...])."""
    issues = []
    boxes = []
    if not os.path.exists(path):
        return False, boxes, ["missing_label_file"]
    with open(path) as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    eps = 1e-3
    for ln_no, line in enumerate(lines, 1):
        parts = line.split()
        if len(parts) != 5:
            issues.append(f"line {ln_no}: expected 5 values, got {len(parts)}")
            continue
        try:
            c = int(float(parts[0]))
            xc, yc, w, h = map(float, parts[1:5])
        except ValueError:
            issues.append(f"line {ln_no}: non-numeric value")
            continue
        if c < 0 or c > 3:
            issues.append(f"line {ln_no}: class id {c} out of range [0,3]")
        for name, v in [("xc", xc), ("yc", yc), ("w", w), ("h", h)]:
            if v < -eps or v > 1 + eps:
                issues.append(f"line {ln_no}: {name}={v} outside normalized [0,1] range")
        if w <= 0 or h <= 0:
            issues.append(f"line {ln_no}: non-positive width/height")
        boxes.append((c, xc, yc, w, h))
    return (len(issues) == 0), boxes, issues


def stratified_two_stage(X, Y, seed, val_frac, test_frac):
    """Two-stage MultilabelStratifiedShuffleSplit -> (train_idx, val_idx, test_idx),
    indices relative to the ORIGINAL X/Y passed in."""
    temp_frac = val_frac + test_frac
    msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=temp_frac, random_state=seed)
    train_idx, temp_idx = next(msss1.split(X, Y))
    X_temp, Y_temp = X[temp_idx], Y[temp_idx]
    rel_test = test_frac / temp_frac
    msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=rel_test, random_state=seed)
    val_rel_idx, test_rel_idx = next(msss2.split(X_temp, Y_temp))
    return train_idx, temp_idx[val_rel_idx], temp_idx[test_rel_idx]


def indices_to_assignment(train_idx, val_idx, test_idx, group_records):
    assignment = {}
    for idx in train_idx:
        for s in group_records[idx]["stems"]:
            assignment[s] = "train"
    for idx in val_idx:
        for s in group_records[idx]["stems"]:
            assignment[s] = "val"
    for idx in test_idx:
        for s in group_records[idx]["stems"]:
            assignment[s] = "test"
    return assignment


def classes_covered_all_splits(assignment, records):
    result = {}
    for c in CLASS_COLS:
        imgs_with_c = [r["stem"] for r in records if r[f"has_{c}"]]
        per_split = {"train": 0, "val": 0, "test": 0}
        for s in imgs_with_c:
            sp = assignment.get(s)
            if sp in per_split:
                per_split[sp] += 1
        result[c] = {
            "n_images": len(imgs_with_c),
            "train": per_split["train"], "val": per_split["val"], "test": per_split["test"],
            "present_all": all(per_split[k] > 0 for k in ("train", "val", "test")),
            "achievable": len(imgs_with_c) >= 3,
        }
    return result


def reservation_split(records, group_map, group_records, seed, ratios):
    """Deterministic fallback: guarantees >=1 containing-image per class per split
    wherever mathematically possible (class has >=3 containing-images), then fills
    the remainder with a seeded proportional assignment. Duplicate-hash groups are
    always kept together in one split."""
    rng = random.Random(seed)
    n_total = len(records)
    target = {
        "train": round(ratios[0] * n_total),
        "val": round(ratios[1] * n_total),
        "test": round(ratios[2] * n_total),
    }
    target["train"] += n_total - sum(target.values())

    assignment = {}
    used_groups = set()

    class_order = sorted(CLASS_COLS, key=lambda c: sum(1 for g in group_records if g[f"has_{c}"]))
    for c in class_order:
        candidates = [g for g in group_records if g[f"has_{c}"] and g["gid"] not in used_groups]
        rng.shuffle(candidates)
        if len(candidates) >= 3:
            plan = zip(["val", "test", "train"], candidates[:3])
        elif len(candidates) == 2:
            plan = zip(["val", "train"], candidates[:2])
        elif len(candidates) == 1:
            plan = zip(["train"], candidates[:1])
        else:
            plan = []
        for sp, g in plan:
            for s in g["stems"]:
                assignment[s] = sp
            used_groups.add(g["gid"])

    remaining = [g for g in group_records if g["gid"] not in used_groups]
    rng.shuffle(remaining)
    cur_counts = {"train": 0, "val": 0, "test": 0}
    for s, sp in assignment.items():
        cur_counts[sp] += 1
    for g in remaining:
        deficits = {sp: target[sp] - cur_counts[sp] for sp in ("train", "val", "test")}
        chosen = max(deficits, key=lambda k: deficits[k])
        for s in g["stems"]:
            assignment[s] = chosen
        cur_counts[chosen] += len(g["stems"])
        used_groups.add(g["gid"])

    return assignment


def compute_split_stats(records, assignment):
    stats = {sp: {"images": 0, "boxes_total": 0, "background": 0,
                   **{f"boxes_{c}": 0 for c in CLASS_COLS},
                   **{f"pos_images_{c}": 0 for c in CLASS_COLS}}
             for sp in ("train", "val", "test")}
    for r in records:
        sp = assignment[r["stem"]]
        stats[sp]["images"] += 1
        stats[sp]["boxes_total"] += r["num_boxes"]
        if r["is_bg"]:
            stats[sp]["background"] += 1
        for c in CLASS_COLS:
            stats[sp][f"boxes_{c}"] += r[f"n_{c}"]
            if r[f"has_{c}"]:
                stats[sp][f"pos_images_{c}"] += 1
    return stats


# ------------------------------------------------------------------------------
# 3. INVENTORY PASS — read every image + label, validate, hash
# ------------------------------------------------------------------------------
image_files = sorted([p for p in glob.glob(os.path.join(SRC_IMAGES_DIR, "*.*"))
                       if p.lower().endswith((".jpg", ".jpeg", ".png"))])
label_files_all = sorted(glob.glob(os.path.join(SRC_LABELS_DIR, "*.txt")))
print(f"Found {len(image_files)} images, {len(label_files_all)} label files in source dirs.")

image_stems_all = {os.path.splitext(os.path.basename(p))[0] for p in image_files}
label_stems_all = {os.path.splitext(os.path.basename(p))[0] for p in label_files_all}
missing_labels_for_images = sorted(image_stems_all - label_stems_all)
orphan_labels_no_image = sorted(label_stems_all - image_stems_all)

records = []
issues_log = {"corrupt_images": [], "invalid_labels": {}}

for img_path in image_files:
    stem = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = os.path.join(SRC_LABELS_DIR, stem + ".txt")

    img = cv2.imread(img_path)
    if img is None:
        issues_log["corrupt_images"].append(stem)
        continue

    is_valid, boxes, issues = validate_label_file(lbl_path)
    if not is_valid:
        issues_log["invalid_labels"][stem] = issues

    counts = {c: 0 for c in CLASS_COLS}
    for c, *_ in boxes:
        counts[CLASS_NAMES[c]] += 1

    rec = {
        "stem": stem, "img_path": img_path, "lbl_path": lbl_path,
        "num_boxes": len(boxes), "is_bg": len(boxes) == 0,
        "md5": file_md5(img_path),
    }
    for c in CLASS_COLS:
        rec[f"has_{c}"] = counts[c] > 0
        rec[f"n_{c}"] = counts[c]
    records.append(rec)

print(f"Images usable for split (non-corrupt): {len(records)}")
if issues_log["corrupt_images"]:
    print(f"[WARNING] Corrupt/unreadable images excluded: {issues_log['corrupt_images']}")

# ------------------------------------------------------------------------------
# 4. DUPLICATE-CONTENT GROUPING (md5) — duplicates always stay in one split
# ------------------------------------------------------------------------------
group_map = defaultdict(list)  # md5 -> list of record dicts
for r in records:
    group_map[r["md5"]].append(r)
duplicate_groups = {h: [m["stem"] for m in members] for h, members in group_map.items() if len(members) > 1}

group_records = []
for gid in sorted(group_map.keys()):
    members = group_map[gid]
    group_records.append({
        "gid": gid, "stems": [m["stem"] for m in members],
        **{f"has_{c}": any(m[f"has_{c}"] for m in members) for c in CLASS_COLS},
        "is_bg": all(m["is_bg"] for m in members),
        "num_boxes": sum(m["num_boxes"] for m in members),
    })

# ------------------------------------------------------------------------------
# 5. PRIMARY SPLIT: multilabel-stratified (class presence + background as labels)
# ------------------------------------------------------------------------------
Y = np.array([[int(g[f"has_{c}"]) for c in CLASS_COLS] + [int(g["is_bg"])] for g in group_records])
X = np.zeros((len(group_records), 1))

primary_ok = True
try:
    train_idx, val_idx, test_idx = stratified_two_stage(X, Y, SEED, RATIOS[1], RATIOS[2])
    primary_assignment = indices_to_assignment(train_idx, val_idx, test_idx, group_records)
except Exception as e:
    print(f"[WARNING] Primary multilabel-stratified split failed ({e}); will use fallback.")
    primary_assignment = {}
    primary_ok = False

coverage_primary = classes_covered_all_splits(primary_assignment, records) if primary_ok else {}
need_fallback = (not primary_ok) or any(
    v["achievable"] and not v["present_all"] for v in coverage_primary.values()
)

# ------------------------------------------------------------------------------
# 6. FALLBACK (only if triggered): deterministic rare-class reservation, seed=42
# ------------------------------------------------------------------------------
if need_fallback:
    print("[INFO] Rare-class coverage check failed (or primary split errored) -> "
          "regenerating with a deterministic reservation algorithm (seed=42) that "
          "guarantees >=1 image per class per split wherever mathematically possible.")
    final_assignment = reservation_split(records, group_map, group_records, SEED, RATIOS)
    split_method = "reservation_fallback"
else:
    final_assignment = primary_assignment
    split_method = "multilabel_stratified_shuffle_split"

assert set(final_assignment.keys()) == {r["stem"] for r in records}, \
    "Every usable image must receive exactly one split assignment."

coverage_final = classes_covered_all_splits(final_assignment, records)

# ------------------------------------------------------------------------------
# 7. WRITE OUTPUT DIRECTORY STRUCTURE (copy, never modify originals)
# ------------------------------------------------------------------------------
for sp in ("train", "val", "test"):
    os.makedirs(os.path.join(SPLIT_ROOT, sp, "images"), exist_ok=True)
    os.makedirs(os.path.join(SPLIT_ROOT, sp, "labels"), exist_ok=True)

for r in records:
    sp = final_assignment[r["stem"]]
    shutil.copy2(r["img_path"], os.path.join(SPLIT_ROOT, sp, "images", os.path.basename(r["img_path"])))
    lbl_dst = os.path.join(SPLIT_ROOT, sp, "labels", os.path.basename(r["lbl_path"]))
    if os.path.exists(r["lbl_path"]):
        shutil.copy2(r["lbl_path"], lbl_dst)
    else:
        open(lbl_dst, "w").close()

for sp in ("train", "val", "test"):
    n_img = len(glob.glob(os.path.join(SPLIT_ROOT, sp, "images", "*.*")))
    n_lbl = len(glob.glob(os.path.join(SPLIT_ROOT, sp, "labels", "*.txt")))
    expected = sum(1 for v in final_assignment.values() if v == sp)
    assert n_img == expected, f"{sp}: copied {n_img} images, expected {expected}"
    assert n_lbl == expected, f"{sp}: copied {n_lbl} labels, expected {expected}"

yaml_path = os.path.join(SPLIT_ROOT, "data.yaml")
with open(yaml_path, "w") as f:
    f.write(
        f"path: {SPLIT_ROOT}\n"
        "train: train/images\n"
        "val: val/images\n"
        "test: test/images\n\n"
        f"nc: {len(CLASS_COLS)}\n"
        f"names: {CLASS_COLS}\n"
    )

# ------------------------------------------------------------------------------
# 8. STATS, MANIFEST, SUMMARY FILES
# ------------------------------------------------------------------------------
split_stats = compute_split_stats(records, final_assignment)

manifest_rows = [{
    "image": os.path.basename(r["img_path"]), "split": final_assignment[r["stem"]],
    "has_D00": int(r["has_D00"]), "has_D10": int(r["has_D10"]),
    "has_D20": int(r["has_D20"]), "has_D40": int(r["has_D40"]),
    "num_boxes": r["num_boxes"],
} for r in records]
manifest_df = pd.DataFrame(manifest_rows).sort_values(["split", "image"]).reset_index(drop=True)
manifest_csv = os.path.join(SPLIT_ROOT, "split_manifest.csv")
manifest_df.to_csv(manifest_csv, index=False)

summary_rows = []
for sp in ("train", "val", "test"):
    row = {"split": sp, **split_stats[sp]}
    row["pct_of_total"] = round(100 * split_stats[sp]["images"] / len(records), 2)
    summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows)
summary_csv = os.path.join(SPLIT_ROOT, "split_summary.csv")
summary_df.to_csv(summary_csv, index=False)
summary_json = os.path.join(SPLIT_ROOT, "split_summary.json")
with open(summary_json, "w") as f:
    json.dump(summary_rows, f, indent=2)

# ------------------------------------------------------------------------------
# 9. INTEGRITY VALIDATION CHECKS
# ------------------------------------------------------------------------------
cross_split_dup = []
for gid, members in group_map.items():
    if len(members) > 1:
        splits_in_group = {final_assignment[m["stem"]] for m in members}
        if len(splits_in_group) > 1:
            cross_split_dup.append([m["stem"] for m in members])

checks = {
    "all_images_assigned_exactly_once": set(final_assignment.keys()) == {r["stem"] for r in records},
    "every_image_has_label_file": len(missing_labels_for_images) == 0,
    "every_label_has_matching_image": len(orphan_labels_no_image) == 0,
    "no_duplicate_hash_across_splits": len(cross_split_dup) == 0,
    "no_corrupt_images": len(issues_log["corrupt_images"]) == 0,
    "no_invalid_yolo_labels": len(issues_log["invalid_labels"]) == 0,
}
all_checks_passed = all(checks.values())
rare_class_ok = all((not v["achievable"]) or v["present_all"] for v in coverage_final.values())
split_ready = all_checks_passed and rare_class_ok

# ------------------------------------------------------------------------------
# 10. OPTIONAL CONTACT SHEET
# ------------------------------------------------------------------------------
CONTACT_SHEET_PATH = None
try:
    tiles = []
    for sp in ("train", "val", "test"):
        sp_recs = [r for r in records if final_assignment[r["stem"]] == sp]
        d10_first = [r for r in sp_recs if r["has_D10"]]
        chosen, seen = [], set()
        for r in d10_first + sp_recs:
            if r["stem"] not in seen:
                chosen.append(r); seen.add(r["stem"])
            if len(chosen) == 3:
                break
        tiles += [(sp, r) for r in chosen]

    if tiles:
        cols = 3
        rows_n = int(np.ceil(len(tiles) / cols))
        fig, axes = plt.subplots(rows_n, cols, figsize=(cols * 4, rows_n * 4))
        axes = np.array(axes).reshape(-1)
        for ax, (sp, r) in zip(axes, tiles):
            img = cv2.cvtColor(cv2.imread(r["img_path"]), cv2.COLOR_BGR2RGB)
            h, w = img.shape[:2]
            _, boxes, _ = validate_label_file(r["lbl_path"])
            for c, xc, yc, bw, bh in boxes:
                x1, y1 = int((xc - bw / 2) * w), int((yc - bh / 2) * h)
                x2, y2 = int((xc + bw / 2) * w), int((yc + bh / 2) * h)
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 200, 0), 2)
                cv2.putText(img, CLASS_NAMES[c], (x1, max(y1 - 4, 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 0), 1)
            ax.imshow(img)
            present = ",".join(c for c in CLASS_COLS if r[f"has_{c}"]) or "background"
            ax.set_title(f"[{sp}] {os.path.basename(r['img_path'])}\n{present}", fontsize=8)
            ax.axis("off")
        for ax in axes[len(tiles):]:
            ax.axis("off")
        plt.tight_layout()
        CONTACT_SHEET_PATH = os.path.join(SPLIT_ROOT, "split_contact_sheet.png")
        plt.savefig(CONTACT_SHEET_PATH, dpi=150, bbox_inches="tight")
        plt.close(fig)
except Exception as e:
    print(f"[INFO] Contact sheet skipped: {e}")

# ------------------------------------------------------------------------------
# 11. FINAL REPORT
# ------------------------------------------------------------------------------
print("\n" + "=" * 78)
print("DATASET SPLIT SUMMARY")
print("=" * 78)
print(f"Total images found on disk        : {len(image_files)}")
if issues_log["corrupt_images"]:
    print(f"  excluded as corrupt              : {len(issues_log['corrupt_images'])}")
print(f"Total images included in split    : {len(records)}")
print(f"Train images                      : {split_stats['train']['images']}")
print(f"Val images                        : {split_stats['val']['images']}")
print(f"Test images                       : {split_stats['test']['images']}")
print(f"Total boxes                       : {sum(r['num_boxes'] for r in records)}")

print("\nBoxes per class per split:")
print(summary_df[["split"] + [f"boxes_{c}" for c in CLASS_COLS] + ["boxes_total"]].to_string(index=False))

print("\nPositive images per class per split:")
print(summary_df[["split"] + [f"pos_images_{c}" for c in CLASS_COLS]].to_string(index=False))

print("\nBackground images per split:")
print(summary_df[["split", "background"]].to_string(index=False))

print("\nPercentage of total images per split:")
print(summary_df[["split", "pct_of_total"]].to_string(index=False))

print("\n" + "=" * 78)
print("VALIDATION CHECKS")
print("=" * 78)
for k, v in checks.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")
if issues_log["corrupt_images"]:
    print(f"  Corrupt images                 : {issues_log['corrupt_images']}")
if issues_log["invalid_labels"]:
    print(f"  Invalid label files ({len(issues_log['invalid_labels'])}):")
    for stem, iss in issues_log["invalid_labels"].items():
        print(f"    {stem}: {iss}")
if missing_labels_for_images:
    print(f"  Images missing a label file    : {missing_labels_for_images}")
if orphan_labels_no_image:
    print(f"  Label files with no image      : {orphan_labels_no_image}")
if duplicate_groups:
    print(f"  Duplicate-content image groups (kept together in one split): {duplicate_groups}")
else:
    print("  No duplicate image content (md5) found.")
if cross_split_dup:
    print(f"  [FAIL] Duplicate content spanning multiple splits: {cross_split_dup}")

print("\n" + "=" * 78)
print("CLASS PRESENCE ACROSS SPLITS")
print("=" * 78)
print(f"Split method used: {split_method}")
for c in CLASS_COLS:
    cov = coverage_final[c]
    if cov["present_all"]:
        status = "present in all 3 splits"
    elif not cov["achievable"]:
        status = f"NOT mathematically achievable (only {cov['n_images']} containing-image(s))"
    else:
        status = "MISSING from at least one split (unexpected — investigate)"
    print(f"{c}: train={cov['train']} val={cov['val']} test={cov['test']} "
          f"(total containing-images={cov['n_images']}) -> {status}")

print("\n" + "=" * 78)
print("PATHS CREATED")
print("=" * 78)
print(f"Split root         : {SPLIT_ROOT}")
print(f"data.yaml           : {yaml_path}")
print(f"split_manifest.csv  : {manifest_csv}")
print(f"split_summary.csv   : {summary_csv}")
print(f"split_summary.json  : {summary_json}")
print(f"contact sheet       : {CONTACT_SHEET_PATH or 'not generated'}")

print("\n" + "=" * 78)
if split_ready:
    print("SPLIT READY FOR EXPERIMENT 4")
else:
    print("SPLIT NOT READY — resolve the FAIL items above before proceeding.")
print("=" * 78)
print("Reminder: test/ must remain untouched until final evaluation — do not use it "
      "during fine-tuning or hyperparameter selection. Original custom_city_eval/ was "
      "not modified; all files above were copied.")

In [ ]:
# ============================================================
# RoadVision - Experiment 4: Custom-City Domain Adaptation Fine-tune
# Fine-tunes baseline YOLO11n checkpoint on custom_city_split (train+val only)
# ============================================================

import os, shutil, torch
from pathlib import Path

# ---- 1. Mount Google Drive ----
from google.colab import drive
drive.mount('/content/drive')

# ---- 2. Verify GPU ----
assert torch.cuda.is_available(), "GPU not available. Set runtime to GPU (T4)."
print(f"GPU: {torch.cuda.get_device_name(0)}")

# ---- 3. Verify baseline checkpoint and dataset paths ----
BASELINE_CKPT = "/content/drive/MyDrive/RoadVision/baseline_yolo11n/best.pt"
DATA_YAML = "/content/RoadVision/data/custom_city_split/data.yaml"
SPLIT_ROOT = "/content/RoadVision/data/custom_city_split"

assert os.path.isfile(BASELINE_CKPT), f"Baseline checkpoint not found: {BASELINE_CKPT}"
assert os.path.isfile(DATA_YAML), f"data.yaml not found: {DATA_YAML}"

for req in ["train/images", "train/labels", "val/images", "val/labels"]:
    p = os.path.join(SPLIT_ROOT, req)
    assert os.path.isdir(p), f"Missing required split directory: {p}"

# Explicitly do NOT touch test/ — no references to test paths beyond this point.

print("Baseline checkpoint OK:", BASELINE_CKPT)
print("data.yaml OK:", DATA_YAML)
print("train/val directories verified. Test split untouched.")

# ---- 4. Load baseline checkpoint ----
from ultralytics import YOLO
model = YOLO(BASELINE_CKPT)

# ---- 5. Single conservative fine-tuning run ----
RUN_PROJECT = "/content/RoadVision/runs/detect"
RUN_NAME = "experiment4_custom_finetune"

results = model.train(
    data=DATA_YAML,
    epochs=25,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    seed=42,
    cache=False,
    plots=True,
    patience=10,
    lr0=0.001,
    lrf=0.01,          # gentle cosine decay target
    optimizer="AdamW",
    warmup_epochs=3,
    freeze=10,          # freeze backbone layers for small-dataset domain adaptation
    project=RUN_PROJECT,
    name=RUN_NAME,
    exist_ok=False,     # ensure a fresh run dir, never overwrite prior runs
    val=True,           # only train/val used internally by ultralytics; test split not referenced
)

# ---- 6. Identify resulting best.pt ----
run_dir = Path(RUN_PROJECT) / RUN_NAME
best_ckpt = run_dir / "weights" / "best.pt"
assert best_ckpt.is_file(), f"best.pt not found at expected location: {best_ckpt}"
print("Fine-tuned best checkpoint located at:", best_ckpt)

# ---- 7. Copy best.pt to Google Drive (new location, baseline untouched) ----
DEST_DIR = "/content/drive/MyDrive/RoadVision/experiment4_custom_finetune"
DEST_PATH = os.path.join(DEST_DIR, "best.pt")

os.makedirs(DEST_DIR, exist_ok=True)
shutil.copy2(best_ckpt, DEST_PATH)

assert os.path.isfile(DEST_PATH), "Copy to Drive failed."
print("Copied best.pt to:", DEST_PATH)
print("Baseline checkpoint left untouched at:", BASELINE_CKPT)

In [ ]:
# ============================================================
# RoadVision - Experiment 4: Held-out Test Evaluation & Error Analysis
# Baseline vs Fine-tuned (Custom-City) comparison, with dedicated D10 analysis
# NO TRAINING IS PERFORMED IN THIS CELL
# ============================================================

import os, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from ultralytics import YOLO

# ---------------------------------------------------------------
# 0. Paths & setup
# ---------------------------------------------------------------
BASELINE_CKPT = "/content/drive/MyDrive/RoadVision/baseline_yolo11n/best.pt"
FINETUNE_CKPT = "/content/drive/MyDrive/RoadVision/experiment4_custom_finetune/best.pt"

DATA_YAML   = "/content/RoadVision/data/custom_city_split/data.yaml"
TEST_IMG_DIR = "/content/RoadVision/data/custom_city_split/test/images"
TEST_LBL_DIR = "/content/RoadVision/data/custom_city_split/test/labels"

OUT_DIR = Path("/content/RoadVision/runs/detect/experiment4_test_evaluation")
OUT_DIR.mkdir(parents=True, exist_ok=True)
CONTACT_SHEET_DIR = OUT_DIR / "d10_contact_sheets"
CONTACT_SHEET_DIR.mkdir(parents=True, exist_ok=True)

for p in [BASELINE_CKPT, FINETUNE_CKPT, DATA_YAML, TEST_IMG_DIR, TEST_LBL_DIR]:
    assert os.path.exists(p), f"Required path not found: {p}"

CLASS_NAMES = ["D00", "D10", "D20", "D40"]
D10_IDX = CLASS_NAMES.index("D10")

image_files = sorted(glob.glob(os.path.join(TEST_IMG_DIR, "*.*")))
label_files = sorted(glob.glob(os.path.join(TEST_LBL_DIR, "*.txt")))
print(f"Test images found: {len(image_files)} | Test label files found: {len(label_files)}")

# ---------------------------------------------------------------
# 1. Load both models
# ---------------------------------------------------------------
model_baseline = YOLO(BASELINE_CKPT)
model_finetuned = YOLO(FINETUNE_CKPT)

# ---------------------------------------------------------------
# 2. Official evaluation on the held-out test split (Ultralytics val())
# ---------------------------------------------------------------
def run_official_eval(model, tag):
    save_dir = OUT_DIR / f"val_{tag}"
    metrics = model.val(
        data=DATA_YAML,
        split="test",
        imgsz=640,
        batch=16,
        device=0,
        workers=2,
        plots=True,
        save_json=False,
        project=str(OUT_DIR),
        name=f"val_{tag}",
        exist_ok=True,
    )
    return metrics

metrics_baseline = run_official_eval(model_baseline, "baseline")
metrics_finetuned = run_official_eval(model_finetuned, "finetuned")

def extract_summary(metrics, model_names):
    box = metrics.box
    overall = {
        "precision": float(box.mp),
        "recall": float(box.mr),
        "mAP50": float(box.map50),
        "mAP50-95": float(box.map),
    }
    per_class = {}
    # ap_class_index tells which classes were present/evaluated
    ap_class_index = list(box.ap_class_index) if hasattr(box, "ap_class_index") else []
    for i, cls_idx in enumerate(ap_class_index):
        cname = model_names[int(cls_idx)]
        per_class[cname] = {
            "precision": float(box.p[i]) if len(box.p) > i else None,
            "recall": float(box.r[i]) if len(box.r) > i else None,
            "mAP50": float(box.ap50[i]) if len(box.ap50) > i else None,
            "mAP50-95": float(box.ap[i]) if len(box.ap) > i else None,
        }
    # ensure all 4 known classes appear even if absent from predictions/GT
    for cname in CLASS_NAMES:
        if cname not in per_class:
            per_class[cname] = {"precision": None, "recall": None, "mAP50": None, "mAP50-95": None}
    return overall, per_class

summary_baseline, perclass_baseline = extract_summary(metrics_baseline, model_baseline.names)
summary_finetuned, perclass_finetuned = extract_summary(metrics_finetuned, model_finetuned.names)

# ---------------------------------------------------------------
# 3. Comparison table (overall + per-class)
# ---------------------------------------------------------------
rows = []
rows.append({
    "scope": "overall", "class": "ALL",
    "baseline_precision": summary_baseline["precision"],
    "baseline_recall": summary_baseline["recall"],
    "baseline_mAP50": summary_baseline["mAP50"],
    "baseline_mAP50-95": summary_baseline["mAP50-95"],
    "finetuned_precision": summary_finetuned["precision"],
    "finetuned_recall": summary_finetuned["recall"],
    "finetuned_mAP50": summary_finetuned["mAP50"],
    "finetuned_mAP50-95": summary_finetuned["mAP50-95"],
})
for cname in CLASS_NAMES:
    b = perclass_baseline[cname]
    f = perclass_finetuned[cname]
    rows.append({
        "scope": "per_class", "class": cname,
        "baseline_precision": b["precision"], "baseline_recall": b["recall"],
        "baseline_mAP50": b["mAP50"], "baseline_mAP50-95": b["mAP50-95"],
        "finetuned_precision": f["precision"], "finetuned_recall": f["recall"],
        "finetuned_mAP50": f["mAP50"], "finetuned_mAP50-95": f["mAP50-95"],
    })

comparison_df = pd.DataFrame(rows)
comparison_csv_path = OUT_DIR / "baseline_vs_finetuned_comparison.csv"
comparison_df.to_csv(comparison_csv_path, index=False)
print("\n=== Baseline vs Fine-tuned Comparison ===")
print(comparison_df.to_string(index=False))

# ---------------------------------------------------------------
# 4. Manual inference + IoU matching (needed for D10 error analysis)
#    Run at a very low confidence threshold so we can also study the
#    effect of the confidence threshold on D10 detections.
# ---------------------------------------------------------------
def xywhn_to_xyxy(cx, cy, w, h, img_w, img_h):
    x1 = (cx - w / 2) * img_w
    y1 = (cy - h / 2) * img_h
    x2 = (cx + w / 2) * img_w
    y2 = (cy + h / 2) * img_h
    return [x1, y1, x2, y2]

def load_gt_boxes(label_path, img_w, img_h):
    boxes = []
    if not os.path.isfile(label_path):
        return boxes
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls_id, cx, cy, w, h = int(parts[0]), *map(float, parts[1:])
            boxes.append({"cls": cls_id, "box": xywhn_to_xyxy(cx, cy, w, h, img_w, img_h)})
    return boxes

def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = max(0, boxA[2]-boxA[0]) * max(0, boxA[3]-boxA[1])
    areaB = max(0, boxB[2]-boxB[0]) * max(0, boxB[3]-boxB[1])
    union = areaA + areaB - inter
    return inter / union if union > 0 else 0.0

IOU_THRESH = 0.5
LOW_CONF = 0.001   # near-zero, to see everything the model can produce
DEFAULT_CONF = 0.25  # standard operating threshold for comparison

def run_inference(model, conf):
    results = model.predict(
        source=TEST_IMG_DIR,
        imgsz=640,
        conf=conf,
        device=0,
        save=False,
        verbose=False,
    )
    preds_by_image = {}
    for r in results:
        img_path = r.path
        preds = []
        if r.boxes is not None and len(r.boxes) > 0:
            xyxy = r.boxes.xyxy.cpu().numpy()
            cls = r.boxes.cls.cpu().numpy().astype(int)
            conf_arr = r.boxes.conf.cpu().numpy()
            for b, c, cf in zip(xyxy, cls, conf_arr):
                preds.append({"cls": int(c), "box": b.tolist(), "conf": float(cf)})
        preds_by_image[img_path] = preds
    return preds_by_image

preds_finetuned_lowconf = run_inference(model_finetuned, LOW_CONF)
preds_finetuned_default = run_inference(model_finetuned, DEFAULT_CONF)

# ---------------------------------------------------------------
# 5. D10-specific error analysis (evidence-based, class-aware IoU matching)
# ---------------------------------------------------------------
from PIL import Image

d10_records = []          # per-GT-instance outcome
d10_conf_all = []         # confidence of every D10 prediction (low-conf run)
d10_fp_records = []       # predicted D10 that don't match any GT D10

MATCH_CONF_THRESHOLD = DEFAULT_CONF  # what counts as an "operating" detection

for img_path, preds_low in preds_finetuned_lowconf.items():
    fname = os.path.basename(img_path)
    stem = os.path.splitext(fname)[0]
    label_path = os.path.join(TEST_LBL_DIR, stem + ".txt")

    with Image.open(img_path) as im:
        img_w, img_h = im.size

    gt_boxes = load_gt_boxes(label_path, img_w, img_h)
    gt_d10 = [g for g in gt_boxes if g["cls"] == D10_IDX]

    # collect confidences of predictions classified as D10 (any confidence, low-conf run)
    d10_preds_low = [p for p in preds_low if p["cls"] == D10_IDX]
    d10_conf_all.extend([p["conf"] for p in d10_preds_low])

    # only "operating-threshold" predictions are used for TP/FP/FN bookkeeping
    preds_at_thresh = [p for p in preds_low if p["conf"] >= MATCH_CONF_THRESHOLD]

    used_pred_idx = set()
    for gt in gt_d10:
        best_iou = 0.0
        best_pred = None
        best_pred_i = None
        for i, p in enumerate(preds_at_thresh):
            if i in used_pred_idx:
                continue
            cur_iou = iou(gt["box"], p["box"])
            if cur_iou > best_iou:
                best_iou = cur_iou
                best_pred = p
                best_pred_i = i

        if best_pred is not None and best_iou >= IOU_THRESH and best_pred["cls"] == D10_IDX:
            outcome = "TP_correct_class_and_location"
            used_pred_idx.add(best_pred_i)
        elif best_pred is not None and best_iou >= IOU_THRESH and best_pred["cls"] != D10_IDX:
            outcome = f"misclassified_as_{CLASS_NAMES[best_pred['cls']]}"
            used_pred_idx.add(best_pred_i)
        elif best_pred is not None and 0 < best_iou < IOU_THRESH:
            outcome = "localization_failure_low_iou"
        else:
            outcome = "missed_detection_no_overlap"

        # check: does a D10-class prediction with LOWER confidence exist near this GT box
        # (evidence for confidence-threshold issue), independent of the thresholded matching above
        low_conf_overlap = None
        for p in d10_preds_low:
            if p["conf"] < MATCH_CONF_THRESHOLD and iou(gt["box"], p["box"]) >= IOU_THRESH:
                if low_conf_overlap is None or p["conf"] > low_conf_overlap:
                    low_conf_overlap = p["conf"]

        d10_records.append({
            "image": fname,
            "gt_box": gt["box"],
            "outcome": outcome,
            "best_iou_at_operating_thresh": best_iou,
            "sub_threshold_d10_prediction_conf": low_conf_overlap,
        })

    # False positives: predictions classified as D10 (at operating thresh) that never matched any GT D10
    for i, p in enumerate(preds_at_thresh):
        if p["cls"] == D10_IDX and i not in used_pred_idx:
            d10_fp_records.append({"image": fname, "pred_box": p["box"], "conf": p["conf"]})

d10_df = pd.DataFrame(d10_records)
d10_fp_df = pd.DataFrame(d10_fp_records)

gt_d10_total = len(d10_df)
tp_count = int((d10_df["outcome"] == "TP_correct_class_and_location").sum()) if gt_d10_total else 0
missed_count = int((d10_df["outcome"] == "missed_detection_no_overlap").sum()) if gt_d10_total else 0
localization_fail_count = int((d10_df["outcome"] == "localization_failure_low_iou").sum()) if gt_d10_total else 0
misclass_mask = d10_df["outcome"].str.startswith("misclassified_as_") if gt_d10_total else pd.Series([], dtype=bool)
misclass_count = int(misclass_mask.sum()) if gt_d10_total else 0
misclass_breakdown = d10_df.loc[misclass_mask, "outcome"].value_counts().to_dict() if gt_d10_total else {}
fn_total = missed_count + localization_fail_count + misclass_count
fp_total = len(d10_fp_df)

# sub-threshold evidence: FN cases where a D10 prediction existed below operating conf
sub_thresh_recoverable = int(d10_df["sub_threshold_d10_prediction_conf"].notna().sum()) if gt_d10_total else 0

# confidence distribution stats
if len(d10_conf_all) > 0:
    conf_arr = np.array(d10_conf_all)
    conf_stats = {
        "count": int(len(conf_arr)),
        "min": float(conf_arr.min()),
        "max": float(conf_arr.max()),
        "mean": float(conf_arr.mean()),
        "median": float(np.median(conf_arr)),
        "below_default_conf_0.25_count": int((conf_arr < DEFAULT_CONF).sum()),
        "at_or_above_default_conf_0.25_count": int((conf_arr >= DEFAULT_CONF).sum()),
    }
else:
    conf_stats = {"count": 0}

d10_summary = {
    "gt_d10_instances": gt_d10_total,
    "correct_detections_TP": tp_count,
    "false_negatives_total": fn_total,
    "false_negatives_breakdown": {
        "missed_no_overlap": missed_count,
        "localization_failure_low_iou": localization_fail_count,
        "misclassified_as_other_class": misclass_count,
        "misclassified_breakdown": misclass_breakdown,
    },
    "false_positives_D10_predicted_no_matching_gt": fp_total,
    "fn_cases_with_subthreshold_d10_prediction_present": sub_thresh_recoverable,
    "d10_prediction_confidence_distribution_low_conf_run": conf_stats,
    "d10_predictions_count_at_default_conf_0.25": sum(
        1 for preds in preds_finetuned_default.values() for p in preds if p["cls"] == D10_IDX
    ),
    "d10_predictions_count_at_low_conf_0.001": len(d10_conf_all),
}

print("\n=== D10 Error Analysis Summary (Fine-tuned Model) ===")
print(json.dumps(d10_summary, indent=2))

d10_df.to_csv(OUT_DIR / "d10_per_instance_analysis.csv", index=False)
d10_fp_df.to_csv(OUT_DIR / "d10_false_positives.csv", index=False)

# ---------------------------------------------------------------
# 6. Confidence distribution plot for D10 predictions
# ---------------------------------------------------------------
if len(d10_conf_all) > 0:
    plt.figure(figsize=(6, 4))
    plt.hist(d10_conf_all, bins=20, range=(0, 1), edgecolor="black")
    plt.axvline(DEFAULT_CONF, color="red", linestyle="--", label=f"default conf={DEFAULT_CONF}")
    plt.xlabel("Confidence")
    plt.ylabel("Count of D10 predictions")
    plt.title("Confidence distribution of D10 predictions (low-conf inference run)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "d10_confidence_distribution.png", dpi=150)
    plt.close()
else:
    print("No D10 predictions were produced at any confidence (conf >= 0.001).")

# ---------------------------------------------------------------
# 7. Contact sheets for D10 GT instances (GT box vs best-matching prediction)
# ---------------------------------------------------------------
def draw_contact_sheet(img_path, gt_box, pred_info, outcome, out_path):
    im = Image.open(img_path).convert("RGB")
    fig, ax = plt.subplots(1, figsize=(6, 6))
    ax.imshow(im)
    gx1, gy1, gx2, gy2 = gt_box
    ax.add_patch(patches.Rectangle((gx1, gy1), gx2-gx1, gy2-gy1,
                                    linewidth=2, edgecolor="lime", facecolor="none", label="GT D10"))
    if pred_info is not None:
        px1, py1, px2, py2 = pred_info["box"]
        pred_label = f"{CLASS_NAMES[pred_info['cls']]} {pred_info['conf']:.2f}"
        ax.add_patch(patches.Rectangle((px1, py1), px2-px1, py2-py1,
                                        linewidth=2, edgecolor="red", facecolor="none", label=pred_label))
    ax.set_title(outcome, fontsize=9)
    ax.axis("off")
    ax.legend(loc="upper right", fontsize=7)
    plt.tight_layout()
    plt.savefig(out_path, dpi=120)
    plt.close(fig)

sheet_count = 0
for rec in d10_records:
    img_path = os.path.join(TEST_IMG_DIR, rec["image"])
    # find best overlapping prediction (any class) at operating threshold, for visualization
    preds_low = preds_finetuned_lowconf.get(img_path, [])
    preds_at_thresh = [p for p in preds_low if p["conf"] >= MATCH_CONF_THRESHOLD]
    best_iou, best_pred = 0.0, None
    for p in preds_at_thresh:
        cur_iou = iou(rec["gt_box"], p["box"])
        if cur_iou > best_iou:
            best_iou, best_pred = cur_iou, p
    out_path = CONTACT_SHEET_DIR / f"{Path(rec['image']).stem}_{sheet_count}_{rec['outcome']}.png"
    draw_contact_sheet(img_path, rec["gt_box"], best_pred, rec["outcome"], out_path)
    sheet_count += 1

print(f"\nSaved {sheet_count} D10 contact sheets to: {CONTACT_SHEET_DIR}")

# ---------------------------------------------------------------
# 8. Save concise JSON/CSV summary of everything
# ---------------------------------------------------------------
final_summary = {
    "overall_metrics": {
        "baseline": summary_baseline,
        "finetuned": summary_finetuned,
    },
    "per_class_metrics": {
        "baseline": perclass_baseline,
        "finetuned": perclass_finetuned,
    },
    "d10_error_analysis": d10_summary,
    "test_set_size": {
        "images": len(image_files),
        "gt_d10_instances_found_in_labels": gt_d10_total,
    },
    "notes": {
        "iou_threshold_for_matching": IOU_THRESH,
        "operating_confidence_threshold": MATCH_CONF_THRESHOLD,
        "low_confidence_inference_threshold": LOW_CONF,
    },
}

with open(OUT_DIR / "experiment4_evaluation_summary.json", "w") as f:
    json.dump(final_summary, f, indent=2)

comparison_df.to_csv(OUT_DIR / "baseline_vs_finetuned_comparison.csv", index=False)

print(f"\nAll evaluation outputs saved under: {OUT_DIR}")
print(f" - {comparison_csv_path.name}")
print(" - d10_per_instance_analysis.csv")
print(" - d10_false_positives.csv")
print(" - d10_confidence_distribution.png")
print(" - experiment4_evaluation_summary.json")
print(f" - {sheet_count} contact sheets in d10_contact_sheets/")

# ---------------------------------------------------------------
# 9. Evidence-based interpretation (derived only from computed values above)
# ---------------------------------------------------------------
print("\n=== D10 Failure Interpretation (evidence-based, derived from measured values) ===")
if gt_d10_total == 0:
    print("No D10 ground-truth instances were found in the test labels; no interpretation possible.")
else:
    print(f"GT D10 instances: {gt_d10_total}")
    print(f"Correct D10 detections (TP): {tp_count}")
    print(f"False negatives: {fn_total} "
          f"(missed/no-overlap: {missed_count}, localization failure: {localization_fail_count}, "
          f"misclassified as other class: {misclass_count})")
    if misclass_count > 0:
        print(f"Misclassification breakdown: {misclass_breakdown}")
    print(f"D10 false positives (predicted D10, no matching GT): {fp_total}")
    print(f"FN cases where a sub-threshold D10 prediction existed at the same location: "
          f"{sub_thresh_recoverable} of {fn_total} false negatives")
    if conf_stats.get("count", 0) > 0:
        print(f"D10 prediction confidences (low-conf run): n={conf_stats['count']}, "
              f"mean={conf_stats['mean']:.3f}, median={conf_stats['median']:.3f}, "
              f"below default conf(0.25): {conf_stats['below_default_conf_0.25_count']}, "
              f"at/above default conf: {conf_stats['at_or_above_default_conf_0.25_count']}")
    else:
        print("The fine-tuned model produced zero D10-class predictions even at conf=0.001, "
              "indicating the model is not proposing D10 as a class for these regions at all "
              "(not solely a confidence-threshold issue).")
    print("\nThe breakdown above (missed vs. localization failure vs. misclassification vs. "
          "sub-threshold recoverability) is computed directly from IoU-matched predictions "
          "and is the full evidentiary basis for diagnosing the D10 failure mode; "
          "no cause is assumed beyond what these counts show.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

print("Drive RoadVision:")
print(os.listdir('/content/drive/MyDrive/RoadVision'))

In [ ]:
import os

path = "/content/RoadVision/runs/detect/experiment4_test_evaluation/d10_contact_sheets"

print("Exists:", os.path.exists(path))

if os.path.exists(path):
    files = sorted(os.listdir(path))
    print("D10 contact sheets:", len(files))
    for f in files:
        print(f)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

print("RoadVision folders:")
for x in sorted(os.listdir('/content/drive/MyDrive/RoadVision')):
    print(" -", x)

In [ ]:
import os

for folder in [
    "baseline_yolo11n",
    "experiment2_d10_aug",
    "experiment3_highres704",
    "experiment4_custom_finetune"
]:
    path = f"/content/drive/MyDrive/RoadVision/{folder}"
    print(f"\n{folder}:")
    print(os.listdir(path))

In [ ]:
from ultralytics import YOLO
import os

# Paths
BASELINE = "/content/drive/MyDrive/RoadVision/baseline_yolo11n/best.pt"
FINETUNED = "/content/drive/MyDrive/RoadVision/experiment4_custom_finetune/best.pt"

TEST_DIR = "/content/RoadVision/data/custom_city_split/test"
LABEL_DIR = f"{TEST_DIR}/labels"
IMAGE_DIR = f"{TEST_DIR}/images"

print("Baseline exists:", os.path.exists(BASELINE))
print("Fine-tuned exists:", os.path.exists(FINETUNED))
print("Test images:", len(os.listdir(IMAGE_DIR)))
print("Test labels:", len(os.listdir(LABEL_DIR)))

# Load models
baseline_model = YOLO(BASELINE)
finetuned_model = YOLO(FINETUNED)

print("\nModels loaded successfully!")

In [ ]:
!pip install ultralytics

In [ ]:
import os

ROOT = "/content/drive/MyDrive/RoadVision"

def show_tree(path, depth=2, prefix=""):
    if depth < 0 or not os.path.exists(path):
        return

    try:
        items = sorted(os.listdir(path))
    except:
        return

    for item in items:
        full = os.path.join(path, item)
        print(prefix + item)
        if os.path.isdir(full) and depth > 0:
            show_tree(full, depth - 1, prefix + "    ")

show_tree(ROOT, depth=3)

In [ ]:
import os

print("Files:")
for f in os.listdir("/content"):
    if f.endswith(".zip"):
        print(" -", f)

In [ ]:
import zipfile
import os

zips = [
    "/content/custom_city_images.zip.zip",
    "/content/roadvision_custom_city (1).zip"
]

for zip_path in zips:
    print("\n" + "="*70)
    print("ZIP:", os.path.basename(zip_path))
    print("="*70)

    with zipfile.ZipFile(zip_path, "r") as z:
        files = z.namelist()

        print("Total entries:", len(files))
        print("\nFirst 30 entries:")
        for f in files[:30]:
            print(" ", f)

In [ ]:
import zipfile
import os

ZIP_PATH = "/content/roadvision_custom_city (1).zip"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    files = z.namelist()

print("TOTAL ENTRIES:", len(files))
print("\n--- ALL ENTRIES ---")

for f in files:
    print(f)

print("\n" + "="*70)
print("DATA.YAML")
print("="*70)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    yaml_text = z.read("data.yaml").decode("utf-8")
    print(yaml_text)

In [ ]:
import zipfile

ZIP_PATH = "/content/roadvision_custom_city (1).zip"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    train_txt = z.read("train.txt").decode("utf-8")

lines = [x.strip() for x in train_txt.splitlines() if x.strip()]

print("Images listed in train.txt:", len(lines))
print("\nFirst 30:")
for x in lines[:30]:
    print(x)

In [ ]:
import os
import zipfile
import shutil

IMAGES_ZIP = "/content/custom_city_images.zip.zip"
LABELS_ZIP = "/content/roadvision_custom_city (1).zip"

ROOT = "/content/RoadVision/data/custom_city_recovered"
IMAGE_DIR = os.path.join(ROOT, "images")
LABEL_DIR = os.path.join(ROOT, "labels")

os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(LABEL_DIR, exist_ok=True)

# ---------------------------------------------------------
# 1. Extract all 117 original images
# ---------------------------------------------------------
with zipfile.ZipFile(IMAGES_ZIP, "r") as z:
    for member in z.namelist():
        if member.lower().endswith((".jpg", ".jpeg", ".png")):
            filename = os.path.basename(member)
            if filename:
                with z.open(member) as src, open(
                    os.path.join(IMAGE_DIR, filename), "wb"
                ) as dst:
                    shutil.copyfileobj(src, dst)

# ---------------------------------------------------------
# 2. Extract YOLO labels
# ---------------------------------------------------------
with zipfile.ZipFile(LABELS_ZIP, "r") as z:
    for member in z.namelist():
        if member.startswith("labels/train/") and member.endswith(".txt"):
            filename = os.path.basename(member)
            with z.open(member) as src, open(
                os.path.join(LABEL_DIR, filename), "wb"
            ) as dst:
                shutil.copyfileobj(src, dst)

# ---------------------------------------------------------
# 3. Create empty label files for background images
# ---------------------------------------------------------
image_names = sorted(os.listdir(IMAGE_DIR))

for image_name in image_names:
    stem = os.path.splitext(image_name)[0]
    label_path = os.path.join(LABEL_DIR, stem + ".txt")

    if not os.path.exists(label_path):
        open(label_path, "w").close()

# ---------------------------------------------------------
# 4. Verify
# ---------------------------------------------------------
images = sorted([
    f for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

labels = sorted([
    f for f in os.listdir(LABEL_DIR)
    if f.endswith(".txt")
])

print("Images:", len(images))
print("Labels:", len(labels))

missing_labels = [
    f for f in images
    if not os.path.exists(
        os.path.join(LABEL_DIR, os.path.splitext(f)[0] + ".txt")
    )
]

print("Images without labels:", len(missing_labels))

In [ ]:
import os

IMAGE_DIR = "/content/RoadVision/data/custom_city_recovered/images"
LABEL_DIR = "/content/RoadVision/data/custom_city_recovered/labels"

D10_IMAGES = []

for image_name in sorted(os.listdir(IMAGE_DIR)):
    if not image_name.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    stem = os.path.splitext(image_name)[0]
    label_path = os.path.join(LABEL_DIR, stem + ".txt")

    with open(label_path, "r") as f:
        lines = [x.strip() for x in f if x.strip()]

    # class 1 = D10
    if any(line.split()[0] == "1" for line in lines):
        D10_IMAGES.append(image_name)

print("D10-containing images:", len(D10_IMAGES))

for i, name in enumerate(D10_IMAGES, 1):
    print(f"{i:02d}. {name}")

In [ ]:
from ultralytics import YOLO
import os
import cv2
import numpy as np
from PIL import Image
from IPython.display import display

BASELINE = "/content/drive/MyDrive/RoadVision/baseline_yolo11n/best.pt"
FINETUNED = "/content/drive/MyDrive/RoadVision/experiment4_custom_finetune/best.pt"

IMAGE_DIR = "/content/RoadVision/data/custom_city_recovered/images"
LABEL_DIR = "/content/RoadVision/data/custom_city_recovered/labels"

baseline_model = YOLO(BASELINE)
finetuned_model = YOLO(FINETUNED)

OUT_DIR = "/content/RoadVision/d10_visual_inspection"
os.makedirs(OUT_DIR, exist_ok=True)

CLASS_NAMES = ["D00", "D10", "D20", "D40"]

D10_IMAGES = [
    "images (1).jpg",
    "images (10).jpg",
    "images (11).jpg",
    "images (12).jpg",
    "images (13).jpg",
    "images (14).jpg",
    "images (15).jpg",
    "images (2).jpg",
    "images (21).jpg",
    "images (27).jpg",
    "images (28).jpg",
    "images (3).jpg",
    "images (30).jpg",
    "images (31).jpg",
    "images (33).jpg",
    "images (34).jpg",
    "images (36).jpg",
    "images (37).jpg",
    "images (52).jpg",
    "images (57).jpg",
    "images (58).jpg",
    "images (61).jpg",
    "images (62).jpg",
    "images (66).jpg",
    "images (7).jpg",
    "images (77).jpg",
    "images (9).jpg",
]

def draw_gt(image_path, label_path):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    h, w = img.shape[:2]

    with open(label_path, "r") as f:
        lines = [x.strip() for x in f if x.strip()]

    for line in lines:
        parts = line.split()
        cls = int(parts[0])
        xc, yc, bw, bh = map(float, parts[1:5])

        x1 = int((xc - bw / 2) * w)
        y1 = int((yc - bh / 2) * h)
        x2 = int((xc + bw / 2) * w)
        y2 = int((yc + bh / 2) * h)

        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 3)

        cv2.putText(
            img,
            f"GT {CLASS_NAMES[cls]}",
            (x1, max(20, y1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (255, 0, 0),
            2
        )

    return img


def add_header(img, text):
    h, w = img.shape[:2]

    canvas = np.ones((h + 45, w, 3), dtype=np.uint8) * 255
    canvas[45:] = img

    cv2.putText(
        canvas,
        text,
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 0, 0),
        2
    )

    return canvas


for idx, image_name in enumerate(D10_IMAGES, 1):

    image_path = os.path.join(IMAGE_DIR, image_name)

    label_path = os.path.join(
        LABEL_DIR,
        os.path.splitext(image_name)[0] + ".txt"
    )

    # Ground truth
    gt = draw_gt(image_path, label_path)

    # Baseline prediction
    b_result = baseline_model.predict(
        image_path,
        conf=0.001,
        verbose=False
    )[0]

    baseline = b_result.plot()
    baseline = cv2.cvtColor(baseline, cv2.COLOR_BGR2RGB)

    # Fine-tuned prediction
    f_result = finetuned_model.predict(
        image_path,
        conf=0.001,
        verbose=False
    )[0]

    finetuned = f_result.plot()
    finetuned = cv2.cvtColor(finetuned, cv2.COLOR_BGR2RGB)

    # Same dimensions
    h = min(gt.shape[0], baseline.shape[0], finetuned.shape[0])
    w = min(gt.shape[1], baseline.shape[1], finetuned.shape[1])

    gt = cv2.resize(gt, (w, h))
    baseline = cv2.resize(baseline, (w, h))
    finetuned = cv2.resize(finetuned, (w, h))

    gt = add_header(gt, "GROUND TRUTH")
    baseline = add_header(baseline, "BASELINE")
    finetuned = add_header(finetuned, "FINE-TUNED")

    comparison = np.hstack([gt, baseline, finetuned])

    output = os.path.join(
        OUT_DIR,
        f"{idx:02d}_{os.path.splitext(image_name)[0]}.jpg"
    )

    Image.fromarray(comparison).save(output, quality=90)

print(f"Created {len(os.listdir(OUT_DIR))} comparison sheets.")
print("Saved at:", OUT_DIR)

In [ ]:
from PIL import Image
import os
import math
from IPython.display import display

files = sorted([
    os.path.join(OUT_DIR, f)
    for f in os.listdir(OUT_DIR)
    if f.endswith(".jpg")
])

cols = 2
thumb_w = 900
thumb_h = 330
rows = math.ceil(len(files) / cols)

sheet = Image.new(
    "RGB",
    (cols * thumb_w, rows * thumb_h),
    "white"
)

for i, path in enumerate(files):
    img = Image.open(path).convert("RGB")
    img.thumbnail((thumb_w, thumb_h))

    x = (i % cols) * thumb_w
    y = (i // cols) * thumb_h

    sheet.paste(img, (x, y))

sheet_path = "/content/RoadVision/d10_visual_inspection_all.jpg"
sheet.save(sheet_path, quality=90)

print("Saved:", sheet_path)

display(sheet)

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
from PIL import Image, ImageDraw
from IPython.display import display

# --------------------------------------------------
# Paths
# --------------------------------------------------

IMAGE_DIR = "/content/RoadVision/data/custom_city_recovered/images"
LABEL_DIR = "/content/RoadVision/data/custom_city_recovered/labels"

BASELINE = "/content/drive/MyDrive/RoadVision/baseline_yolo11n/best.pt"
FINETUNED = "/content/drive/MyDrive/RoadVision/experiment4_custom_finetune/best.pt"

OUT_DIR = "/content/RoadVision/d10_targeted_inspection"
os.makedirs(OUT_DIR, exist_ok=True)

baseline = YOLO(BASELINE)
finetuned = YOLO(FINETUNED)

CLASS_NAMES = ["D00", "D10", "D20", "D40"]

D10_IMAGES = [
    "images (1).jpg",
    "images (10).jpg",
    "images (11).jpg",
    "images (12).jpg",
    "images (13).jpg",
    "images (14).jpg",
    "images (15).jpg",
    "images (2).jpg",
    "images (21).jpg",
    "images (27).jpg",
    "images (28).jpg",
    "images (3).jpg",
    "images (30).jpg",
    "images (31).jpg",
    "images (33).jpg",
    "images (34).jpg",
    "images (36).jpg",
    "images (37).jpg",
    "images (52).jpg",
    "images (57).jpg",
    "images (58).jpg",
    "images (61).jpg",
    "images (62).jpg",
    "images (66).jpg",
    "images (7).jpg",
    "images (77).jpg",
    "images (9).jpg",
]

# --------------------------------------------------
# IoU
# --------------------------------------------------

def iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2-x1) * max(0, y2-y1)

    a1 = max(0, box1[2]-box1[0]) * max(0, box1[3]-box1[1])
    a2 = max(0, box2[2]-box2[0]) * max(0, box2[3]-box2[1])

    union = a1 + a2 - inter

    return inter / union if union > 0 else 0


# --------------------------------------------------
# Read D10 GT boxes
# --------------------------------------------------

records = []

for image_name in D10_IMAGES:

    image_path = os.path.join(IMAGE_DIR, image_name)
    label_path = os.path.join(
        LABEL_DIR,
        os.path.splitext(image_name)[0] + ".txt"
    )

    img = cv2.imread(image_path)
    h, w = img.shape[:2]

    with open(label_path) as f:
        lines = [x.strip() for x in f if x.strip()]

    gt_boxes = []

    for line in lines:

        p = line.split()

        cls = int(p[0])

        if cls != 1:
            continue

        xc, yc, bw, bh = map(float, p[1:5])

        gt = [
            (xc-bw/2)*w,
            (yc-bh/2)*h,
            (xc+bw/2)*w,
            (yc+bh/2)*h
        ]

        gt_boxes.append(gt)

    # --------------------------------------------------
    # Predictions
    # --------------------------------------------------

    b = baseline.predict(
        image_path,
        conf=0.001,
        verbose=False
    )[0]

    f = finetuned.predict(
        image_path,
        conf=0.001,
        verbose=False
    )[0]

    def get_predictions(result):

        preds = []

        if result.boxes is None:
            return preds

        for box, cls, conf in zip(
            result.boxes.xyxy.cpu().numpy(),
            result.boxes.cls.cpu().numpy(),
            result.boxes.conf.cpu().numpy()
        ):
            preds.append({
                "box": box.tolist(),
                "class": int(cls),
                "confidence": float(conf)
            })

        return preds

    bp = get_predictions(b)
    fp = get_predictions(f)

    # --------------------------------------------------
    # Analyze every D10 GT
    # --------------------------------------------------

    for gt_idx, gt in enumerate(gt_boxes, 1):

        def best_overlap(preds):

            candidates = []

            for p in preds:

                overlap = iou(gt, p["box"])

                candidates.append({
                    "iou": overlap,
                    "class": p["class"],
                    "confidence": p["confidence"]
                })

            if not candidates:
                return None

            return max(
                candidates,
                key=lambda x: x["iou"]
            )

        bbest = best_overlap(bp)
        fbest = best_overlap(fp)

        records.append({
            "image": image_name,
            "gt_index": gt_idx,

            "baseline_best_iou":
                bbest["iou"] if bbest else 0,

            "baseline_best_class":
                CLASS_NAMES[bbest["class"]] if bbest else "NONE",

            "baseline_best_conf":
                bbest["confidence"] if bbest else 0,

            "finetuned_best_iou":
                fbest["iou"] if fbest else 0,

            "finetuned_best_class":
                CLASS_NAMES[fbest["class"]] if fbest else "NONE",

            "finetuned_best_conf":
                fbest["confidence"] if fbest else 0,
        })

df = pd.DataFrame(records)

print("D10 GT instances analyzed:", len(df))
print("\nResults:")
display(df)

csv_path = os.path.join(
    OUT_DIR,
    "d10_targeted_analysis.csv"
)

df.to_csv(csv_path, index=False)

print("\nSaved:", csv_path)

In [ ]:
import pandas as pd
CSV_PATH = "/content/RoadVision/d10_targeted_inspection/d10_targeted_analysis.csv"

df = pd.read_csv(CSV_PATH)

print("Total D10 GT instances:", len(df))

# ---------------------------------------------------------
# 1. Localization quality
# ---------------------------------------------------------

print("\n=== BASELINE LOCALIZATION ===")
for threshold in [0.1, 0.25, 0.5, 0.75]:
  count = (df["baseline_best_iou"] >= threshold).sum()
  print(
        f"IoU >= {threshold}: "
        f"{count}/{len(df)} "
        f"({count/len(df):.1%})"
    )

print("\n=== FINE-TUNED LOCALIZATION ===")
for threshold in [0.1, 0.25, 0.5, 0.75]:
    count = (df["finetuned_best_iou"] >= threshold).sum()
    print(
        f"IoU >= {threshold}: "
        f"{count}/{len(df)} "
        f"({count/len(df):.1%})"
    )

# ---------------------------------------------------------
# 2. What class gets the highest-overlap prediction?
# ---------------------------------------------------------

print("\n=== BASELINE BEST-OVERLAP CLASS ===")
print(
    df["baseline_best_class"]
    .value_counts()
    .to_string()
)

print("\n=== FINE-TUNED BEST-OVERLAP CLASS ===")
print(
    df["finetuned_best_class"]
    .value_counts()
    .to_string()
)

# ---------------------------------------------------------
# 3. Strong localization but wrong class
# ---------------------------------------------------------

for model in ["baseline", "finetuned"]:

    iou_col = f"{model}_best_iou"
    class_col = f"{model}_best_class"

    print(f"\n=== {model.upper()} ===")

    for threshold in [0.5, 0.75]:

        localized = df[iou_col] >= threshold

        print(
            f"\nIoU >= {threshold}: "
            f"{localized.sum()}/{len(df)}"
        )

        print(
            df.loc[localized, class_col]
            .value_counts()
            .to_string()
        )

# ---------------------------------------------------------
# 4. Confidence of highest-overlap predictions
# ---------------------------------------------------------

print("\n=== CONFIDENCE SUMMARY ===")

for model in ["baseline", "finetuned"]:

    col = f"{model}_best_conf"

    print(f"\n{model.upper()}")
    print(
        df[col].describe()[
            ["min", "25%", "50%", "75%", "max"]
        ]
    )

# ---------------------------------------------------------
# 5. Strong localization + D10 classification
# ---------------------------------------------------------

print("\n=== D10 AT CORRECT LOCATION ===")

for model in ["baseline", "finetuned"]:

    iou_col = f"{model}_best_iou"
    class_col = f"{model}_best_class"

    for threshold in [0.5, 0.75]:

        correct = (
            (df[iou_col] >= threshold) &
            (df[class_col] == "D10")
        )

        print(
            f"{model} | IoU >= {threshold} "
            f"+ class D10: "
            f"{correct.sum()}/{len(df)}"
        )

In [ ]:
import os
import cv2
import numpy as np
from PIL import Image, ImageDraw
from IPython.display import display

IMAGE_DIR = "/content/RoadVision/data/custom_city_recovered/images"
LABEL_DIR = "/content/RoadVision/data/custom_city_recovered/labels"

OUT_DIR = "/content/RoadVision/d10_gt_crops"
os.makedirs(OUT_DIR, exist_ok=True)

D10_IMAGES = [
    "images (1).jpg",
    "images (10).jpg",
    "images (11).jpg",
    "images (12).jpg",
    "images (13).jpg",
    "images (14).jpg",
    "images (15).jpg",
    "images (2).jpg",
    "images (21).jpg",
    "images (27).jpg",
    "images (28).jpg",
    "images (3).jpg",
    "images (30).jpg",
    "images (31).jpg",
    "images (33).jpg",
    "images (34).jpg",
    "images (36).jpg",
    "images (37).jpg",
    "images (52).jpg",
    "images (57).jpg",
    "images (58).jpg",
    "images (61).jpg",
    "images (62).jpg",
    "images (66).jpg",
    "images (7).jpg",
    "images (77).jpg",
    "images (9).jpg",
]

crop_files = []

for image_name in D10_IMAGES:

    image_path = os.path.join(IMAGE_DIR, image_name)
    label_path = os.path.join(
        LABEL_DIR,
        os.path.splitext(image_name)[0] + ".txt"
    )

    img = cv2.imread(image_path)

    if img is None:
        continue

    h, w = img.shape[:2]

    with open(label_path) as f:
        lines = [x.strip() for x in f if x.strip()]

    d10_index = 0

    for line in lines:

        p = line.split()

        if int(p[0]) != 1:
            continue

        xc, yc, bw, bh = map(float, p[1:5])

        x1 = int((xc - bw/2) * w)
        y1 = int((yc - bh/2) * h)
        x2 = int((xc + bw/2) * w)
        y2 = int((yc + bh/2) * h)

        # Expand crop around the D10 box
        box_w = x2 - x1
        box_h = y2 - y1

        pad_x = int(box_w * 1.5)
        pad_y = int(box_h * 1.5)

        cx1 = max(0, x1 - pad_x)
        cy1 = max(0, y1 - pad_y)
        cx2 = min(w, x2 + pad_x)
        cy2 = min(h, y2 + pad_y)

        crop = img[cy1:cy2, cx1:cx2]

        if crop.size == 0:
            continue

        crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)

        # Add border + label
        crop_img = Image.fromarray(crop)

        draw = ImageDraw.Draw(crop_img)
        draw.rectangle(
            [x1-cx1, y1-cy1, x2-cx1, y2-cy1],
            outline="red",
            width=4
        )

        d10_index += 1

        filename = (
            f"{os.path.splitext(image_name)[0]}"
            f"_D10_{d10_index}.jpg"
        )

        output = os.path.join(OUT_DIR, filename)

        crop_img.save(output)
        crop_files.append(output)

print("D10 crops created:", len(crop_files))
print("Saved to:", OUT_DIR)

In [ ]:
from PIL import Image, ImageDraw
import os
import math
from IPython.display import display

files = sorted([
    os.path.join(OUT_DIR, f)
    for f in os.listdir(OUT_DIR)
    if f.endswith(".jpg")
])

cols = 4
cell_w = 300
cell_h = 250

rows = math.ceil(len(files) / cols)

sheet = Image.new(
    "RGB",
    (cols * cell_w, rows * cell_h),
    "white"
)

for i, path in enumerate(files):

    img = Image.open(path).convert("RGB")
    img.thumbnail((cell_w - 10, cell_h - 10))

    x = (i % cols) * cell_w
    y = (i // cols) * cell_h

    sheet.paste(img, (x + 5, y + 5))

sheet_path = "/content/RoadVision/d10_gt_crops_all.jpg"
sheet.save(sheet_path, quality=95)

print("Saved:", sheet_path)

display(sheet)

In [ ]:
import os
import pandas as pd

IMAGE_DIR = "/content/RoadVision/data/custom_city_recovered/images"
LABEL_DIR = "/content/RoadVision/data/custom_city_recovered/labels"

D10_IMAGES = [
    "images (1).jpg",
    "images (10).jpg",
    "images (11).jpg",
    "images (12).jpg",
    "images (13).jpg",
    "images (14).jpg",
    "images (15).jpg",
    "images (2).jpg",
    "images (21).jpg",
    "images (27).jpg",
    "images (28).jpg",
    "images (3).jpg",
    "images (30).jpg",
    "images (31).jpg",
    "images (33).jpg",
    "images (34).jpg",
    "images (36).jpg",
    "images (37).jpg",
    "images (52).jpg",
    "images (57).jpg",
    "images (58).jpg",
    "images (61).jpg",
    "images (62).jpg",
    "images (66).jpg",
    "images (7).jpg",
    "images (77).jpg",
    "images (9).jpg",
]

records = []

for image_name in D10_IMAGES:

    label_path = os.path.join(
        LABEL_DIR,
        os.path.splitext(image_name)[0] + ".txt"
    )

    with open(label_path) as f:
        lines = [x.strip() for x in f if x.strip()]

    for idx, line in enumerate(lines, 1):

        p = line.split()

        if int(p[0]) != 1:
            continue

        xc, yc, bw, bh = map(float, p[1:5])

        area_ratio = bw * bh
        aspect_ratio = bw / bh if bh > 0 else 0

        records.append({
            "image": image_name,
            "d10_index": idx,
            "xc": xc,
            "yc": yc,
            "width_ratio": bw,
            "height_ratio": bh,
            "area_ratio": area_ratio,
            "aspect_ratio": aspect_ratio
        })

df_d10 = pd.DataFrame(records)

print("D10 boxes:", len(df_d10))

print("\n=== SIZE STATISTICS ===")
print(
    df_d10[
        ["width_ratio", "height_ratio", "area_ratio", "aspect_ratio"]
    ].describe()
)

print("\n=== 10 LARGEST D10 BOXES ===")

display(
    df_d10
    .sort_values("area_ratio", ascending=False)
    .head(10)
)

print("\n=== 10 SMALLEST D10 BOXES ===")

display(
    df_d10
    .sort_values("area_ratio")
    .head(10)
)

print("\n=== EXTREME ASPECT RATIOS ===")

print("Very wide (aspect > 5):",
      (df_d10["aspect_ratio"] > 5).sum())

print("Very tall (aspect < 0.2):",
      (df_d10["aspect_ratio"] < 0.2).sum())

print("Large boxes (area > 10% image):",
      (df_d10["area_ratio"] > 0.10).sum())

print("Very large boxes (area > 25% image):",
      (df_d10["area_ratio"] > 0.25).sum())

In [ ]:
import os
import cv2
import math
import pandas as pd
from PIL import Image, ImageDraw
from IPython.display import display

IMAGE_DIR = "/content/RoadVision/data/custom_city_recovered/images"
LABEL_DIR = "/content/RoadVision/data/custom_city_recovered/labels"

# Recreate D10 statistics
records = []

for image_name in os.listdir(IMAGE_DIR):

    if not image_name.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    label_path = os.path.join(
        LABEL_DIR,
        os.path.splitext(image_name)[0] + ".txt"
    )

    with open(label_path) as f:
        lines = [x.strip() for x in f if x.strip()]

    for idx, line in enumerate(lines, 1):

        p = line.split()

        if int(p[0]) != 1:
            continue

        xc, yc, bw, bh = map(float, p[1:5])

        records.append({
            "image": image_name,
            "d10_index": idx,
            "xc": xc,
            "yc": yc,
            "width_ratio": bw,
            "height_ratio": bh,
            "area_ratio": bw * bh,
            "aspect_ratio": bw / bh if bh > 0 else 0
        })

df = pd.DataFrame(records)

largest = df.sort_values(
    "area_ratio",
    ascending=False
).head(6)

print("Largest 6 D10 annotations:")
display(largest)

OUT_DIR = "/content/RoadVision/d10_large_box_audit"
os.makedirs(OUT_DIR, exist_ok=True)

panels = []

for _, row in largest.iterrows():

    image_name = row["image"]
    d10_index = int(row["d10_index"])

    image_path = os.path.join(
        IMAGE_DIR,
        image_name
    )

    label_path = os.path.join(
        LABEL_DIR,
        os.path.splitext(image_name)[0] + ".txt"
    )

    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    h, w = img.shape[:2]

    # Find requested D10 box
    d10_count = 0

    with open(label_path) as f:
        lines = [x.strip() for x in f if x.strip()]

    target = None

    for line in lines:

        p = line.split()

        if int(p[0]) != 1:
            continue

        d10_count += 1

        if d10_count == d10_index:

            xc, yc, bw, bh = map(float, p[1:5])

            x1 = int((xc - bw/2) * w)
            y1 = int((yc - bh/2) * h)
            x2 = int((xc + bw/2) * w)
            y2 = int((yc + bh/2) * h)

            target = (x1, y1, x2, y2)
            break

    if target is None:
        continue

    x1, y1, x2, y2 = target

    # Full image with D10 box
    full = img.copy()

    cv2.rectangle(
        full,
        (x1, y1),
        (x2, y2),
        (255, 0, 0),
        4
    )

    # Expanded crop
    bw_px = x2 - x1
    bh_px = y2 - y1

    pad_x = int(bw_px * 0.15)
    pad_y = int(bh_px * 0.15)

    cx1 = max(0, x1 - pad_x)
    cy1 = max(0, y1 - pad_y)
    cx2 = min(w, x2 + pad_x)
    cy2 = min(h, y2 + pad_y)

    crop = img[cy1:cy2, cx1:cx2]

    # Resize panels
    full_img = Image.fromarray(full)
    crop_img = Image.fromarray(crop)

    full_img.thumbnail((500, 350))
    crop_img.thumbnail((500, 350))

    panel = Image.new(
        "RGB",
        (1020, 430),
        "white"
    )

    panel.paste(full_img, (10, 60))
    panel.paste(crop_img, (510, 60))

    draw = ImageDraw.Draw(panel)

    title = (
        f"{image_name} | D10 #{d10_index} | "
        f"area={row['area_ratio']:.1%} | "
        f"aspect={row['aspect_ratio']:.2f}"
    )

    draw.text((10, 10), title, fill="black")
    draw.text((10, 40), "FULL IMAGE", fill="black")
    draw.text((510, 40), "D10 CROP", fill="black")

    panels.append(panel)

# Contact sheet
cols = 1
rows = len(panels)

sheet = Image.new(
    "RGB",
    (1020, rows * 430),
    "white"
)

for i, panel in enumerate(panels):
    sheet.paste(panel, (0, i * 430))

sheet_path = (
    "/content/RoadVision/d10_large_box_audit.jpg"
)

sheet.save(sheet_path, quality=95)

print("\nSaved:", sheet_path)

display(sheet)

In [ ]:
# Show only the remaining 4 suspiciously-large D10 annotations
# (indices from the previous audit)

targets = [
    ("images (62).jpg", 2),
    ("images (37).jpg", 4),
    ("images (37).jpg", 3),
    ("images (77).jpg", 4),
]

print("Remaining large D10 annotations to inspect:")
for x in targets:
    print(x)

In [ ]:
import pandas as pd

# D10 annotation statistics
stats = df_d10.copy().reset_index(drop=True)

# Model diagnostic results
diag = pd.read_csv(
    "/content/RoadVision/d10_targeted_inspection/d10_targeted_analysis.csv"
).reset_index(drop=True)

print("Annotation rows:", len(stats))
print("Diagnostic rows:", len(diag))

if len(stats) != len(diag):
    raise ValueError(
        f"Row count mismatch: annotations={len(stats)}, diagnostics={len(diag)}"
    )

# ---------------------------------------------------------
# Combine by the same 76-instance order
# ---------------------------------------------------------

audit = pd.concat(
    [
        stats,
        diag.drop(
            columns=[
                c for c in stats.columns
                if c in diag.columns
            ],
            errors="ignore"
        )
    ],
    axis=1
)

# ---------------------------------------------------------
# Size categories
# ---------------------------------------------------------

audit["size_category"] = "normal"

audit.loc[
    audit["area_ratio"] > 0.10,
    "size_category"
] = "large"

audit.loc[
    audit["area_ratio"] > 0.25,
    "size_category"
] = "very_large"

# ---------------------------------------------------------
# Fine-tuned localization
# ---------------------------------------------------------

audit["localization_category"] = "low_overlap"

audit.loc[
    audit["finetuned_best_iou"] >= 0.50,
    "localization_category"
] = "high_overlap"

# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

print("\n=== SIZE CATEGORY ===")
print(audit["size_category"].value_counts())

print("\n=== FINE-TUNED LOCALIZATION ===")
print(audit["localization_category"].value_counts())

print("\n=== HIGH OVERLAP + PREDICTED CLASS ===")

high_overlap = audit[
    audit["finetuned_best_iou"] >= 0.50
]

print(
    high_overlap["finetuned_best_class"]
    .value_counts()
)

# ---------------------------------------------------------
# Highest-overlap cases
# ---------------------------------------------------------

print("\n=== 15 HIGHEST-OVERLAP D10 CASES ===")

display(
    audit[
        [
            "image",
            "d10_index",
            "area_ratio",
            "aspect_ratio",
            "finetuned_best_iou",
            "finetuned_best_class",
            "finetuned_best_conf"
        ]
    ]
    .sort_values(
        "finetuned_best_iou",
        ascending=False
    )
    .head(15)
)

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

audit_path = (
    "/content/RoadVision/d10_targeted_inspection/"
    "d10_annotation_audit.csv"
)

audit.to_csv(audit_path, index=False)

print("\nSaved:", audit_path)

In [ ]:
import os
import pandas as pd

IMAGE_DIR = "/content/RoadVision/data/custom_city_recovered/images"
LABEL_DIR = "/content/RoadVision/data/custom_city_recovered/labels"

DIAG_PATH = (
    "/content/RoadVision/d10_targeted_inspection/"
    "d10_targeted_analysis.csv"
)

# Same fixed image order used in the original diagnostic
D10_IMAGES = [
    "images (1).jpg",
    "images (10).jpg",
    "images (11).jpg",
    "images (12).jpg",
    "images (13).jpg",
    "images (14).jpg",
    "images (15).jpg",
    "images (2).jpg",
    "images (21).jpg",
    "images (27).jpg",
    "images (28).jpg",
    "images (3).jpg",
    "images (30).jpg",
    "images (31).jpg",
    "images (33).jpg",
    "images (34).jpg",
    "images (36).jpg",
    "images (37).jpg",
    "images (52).jpg",
    "images (57).jpg",
    "images (58).jpg",
    "images (61).jpg",
    "images (62).jpg",
    "images (66).jpg",
    "images (7).jpg",
    "images (77).jpg",
    "images (9).jpg",
]

# ---------------------------------------------------------
# Build annotation table in the SAME image order
# ---------------------------------------------------------

records = []

for image_name in D10_IMAGES:

    label_path = os.path.join(
        LABEL_DIR,
        os.path.splitext(image_name)[0] + ".txt"
    )

    with open(label_path) as f:
        lines = [x.strip() for x in f if x.strip()]

    gt_index = 0

    for line in lines:

        p = line.split()

        if int(p[0]) != 1:
            continue

        gt_index += 1

        xc, yc, bw, bh = map(float, p[1:5])

        records.append({
            "image": image_name,
            "gt_index": gt_index,
            "width_ratio": bw,
            "height_ratio": bh,
            "area_ratio": bw * bh,
            "aspect_ratio": bw / bh if bh > 0 else 0
        })

stats = pd.DataFrame(records)

# ---------------------------------------------------------
# Load diagnostic results
# ---------------------------------------------------------

diag = pd.read_csv(DIAG_PATH)

print("Annotation rows:", len(stats))
print("Diagnostic rows:", len(diag))

print("\nDiagnostic columns:")
print(diag.columns.tolist())

# ---------------------------------------------------------
# Proper merge
# ---------------------------------------------------------

audit = stats.merge(
    diag,
    on=["image", "gt_index"],
    how="inner",
    validate="one_to_one"
)

print("\nMerged rows:", len(audit))

if len(audit) != 76:
    raise ValueError(
        f"Expected 76 merged rows, got {len(audit)}"
    )

# ---------------------------------------------------------
# Categories
# ---------------------------------------------------------

audit["size_category"] = "normal"

audit.loc[
    audit["area_ratio"] > 0.10,
    "size_category"
] = "large"

audit.loc[
    audit["area_ratio"] > 0.25,
    "size_category"
] = "very_large"

audit["localization_category"] = "low_overlap"

audit.loc[
    audit["finetuned_best_iou"] >= 0.50,
    "localization_category"
] = "high_overlap"

# ---------------------------------------------------------
# Results
# ---------------------------------------------------------

print("\n=== SIZE CATEGORY ===")
print(audit["size_category"].value_counts())

print("\n=== FINE-TUNED LOCALIZATION ===")
print(audit["localization_category"].value_counts())

print("\n=== HIGH OVERLAP + PREDICTED CLASS ===")

high_overlap = audit[
    audit["finetuned_best_iou"] >= 0.50
]

print(
    high_overlap["finetuned_best_class"]
    .value_counts()
)

print("\n=== 15 HIGHEST-OVERLAP D10 CASES ===")

display(
    audit[
        [
            "image",
            "gt_index",
            "area_ratio",
            "aspect_ratio",
            "finetuned_best_iou",
            "finetuned_best_class",
            "finetuned_best_conf"
        ]
    ]
    .sort_values(
        "finetuned_best_iou",
        ascending=False
    )
    .head(15)
)

# Save corrected audit
AUDIT_PATH = (
    "/content/RoadVision/d10_targeted_inspection/"
    "d10_annotation_audit_corrected.csv"
)

audit.to_csv(AUDIT_PATH, index=False)

print("\nSaved:", AUDIT_PATH)

In [ ]:
import os
import cv2
import pandas as pd
import numpy as np
from PIL import Image, ImageDraw
from IPython.display import display

IMAGE_DIR = "/content/RoadVision/data/custom_city_recovered/images"
LABEL_DIR = "/content/RoadVision/data/custom_city_recovered/labels"

DIAG_PATH = (
    "/content/RoadVision/d10_targeted_inspection/"
    "d10_targeted_analysis.csv"
)

OUT_DIR = "/content/RoadVision/d10_overlap_visual_audit"
os.makedirs(OUT_DIR, exist_ok=True)

CLASS_NAMES = ["D00", "D10", "D20", "D40"]

diag = pd.read_csv(DIAG_PATH)

# Top 15 highest-IoU cases
top15 = (
    diag
    .sort_values("finetuned_best_iou", ascending=False)
    .head(15)
    .reset_index(drop=True)
)

print("Top cases:")
display(top15)

def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2-x1) * max(0, y2-y1)

    a1 = max(0, box1[2]-box1[0]) * max(0, box1[3]-box1[1])
    a2 = max(0, box2[2]-box2[0]) * max(0, box2[3]-box2[1])

    union = a1 + a2 - inter

    return inter / union if union > 0 else 0


panels = []

for _, row in top15.iterrows():

    image_name = row["image"]
    gt_index = int(row["gt_index"])

    image_path = os.path.join(
        IMAGE_DIR,
        image_name
    )

    label_path = os.path.join(
        LABEL_DIR,
        os.path.splitext(image_name)[0] + ".txt"
    )

    img = cv2.imread(image_path)

    if img is None:
        continue

    h, w = img.shape[:2]

    # --------------------------------------------------
    # Find the exact GT D10 box
    # --------------------------------------------------

    d10_count = 0
    gt_box = None

    with open(label_path) as f:
        lines = [x.strip() for x in f if x.strip()]

    for line in lines:

        p = line.split()

        if int(p[0]) != 1:
            continue

        d10_count += 1

        if d10_count == gt_index:

            xc, yc, bw, bh = map(float, p[1:5])

            gt_box = [
                (xc-bw/2)*w,
                (yc-bh/2)*h,
                (xc+bw/2)*w,
                (yc+bh/2)*h
            ]

            break

    if gt_box is None:
        continue

    # --------------------------------------------------
    # Re-run fine-tuned model and find highest IoU box
    # --------------------------------------------------

    # Load only once outside in a normal workflow,
    # but here use existing model if available.
    try:
        finetuned_model
    except NameError:
        from ultralytics import YOLO

        finetuned_model = YOLO(
            "/content/drive/MyDrive/RoadVision/"
            "experiment4_custom_finetune/best.pt"
        )

    result = finetuned_model.predict(
        image_path,
        conf=0.001,
        verbose=False
    )[0]

    best_pred = None
    best_iou = -1

    if result.boxes is not None:

        boxes = result.boxes.xyxy.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy()
        confs = result.boxes.conf.cpu().numpy()

        for box, cls, conf in zip(boxes, classes, confs):

            current_iou = calculate_iou(
                gt_box,
                box
            )

            if current_iou > best_iou:

                best_iou = current_iou

                best_pred = {
                    "box": box,
                    "class": int(cls),
                    "conf": float(conf)
                }

    # --------------------------------------------------
    # Draw
    # --------------------------------------------------

    img_rgb = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )

    # GT = RED
    gx1, gy1, gx2, gy2 = map(
        int,
        gt_box
    )

    cv2.rectangle(
        img_rgb,
        (gx1, gy1),
        (gx2, gy2),
        (255, 0, 0),
        4
    )

    cv2.putText(
        img_rgb,
        "GT D10",
        (gx1, max(25, gy1-10)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 0, 0),
        2
    )

    # Prediction = GREEN
    if best_pred is not None:

        px1, py1, px2, py2 = map(
            int,
            best_pred["box"]
        )

        cv2.rectangle(
            img_rgb,
            (px1, py1),
            (px2, py2),
            (0, 180, 0),
            4
        )

        pred_text = (
            f"{CLASS_NAMES[best_pred['class']]} "
            f"{best_pred['conf']:.3f} "
            f"IoU={best_iou:.3f}"
        )

        cv2.putText(
            img_rgb,
            pred_text,
            (px1, min(h-10, py2+25)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 180, 0),
            2
        )

    # Resize
    panel_img = Image.fromarray(img_rgb)
    panel_img.thumbnail((700, 500))

    panel = Image.new(
        "RGB",
        (720, 550),
        "white"
    )

    panel.paste(
        panel_img,
        (
            (720-panel_img.width)//2,
            45
        )
    )

    draw = ImageDraw.Draw(panel)

    title = (
        f"{image_name} | D10 #{gt_index} | "
        f"IoU={best_iou:.3f}"
    )

    draw.text(
        (10, 10),
        title,
        fill="black"
    )

    panels.append(panel)

# --------------------------------------------------
# Contact sheet
# --------------------------------------------------

cols = 3
rows = int(np.ceil(len(panels) / cols))

sheet = Image.new(
    "RGB",
    (cols*720, rows*550),
    "white"
)

for i, panel in enumerate(panels):

    x = (i % cols) * 720
    y = (i // cols) * 550

    sheet.paste(
        panel,
        (x, y)
    )

sheet_path = (
    "/content/RoadVision/"
    "d10_overlap_visual_audit.jpg"
)

sheet.save(
    sheet_path,
    quality=95
)

print("\nSaved:", sheet_path)

display(sheet)

In [ ]:
# ============================================================
# RoadVision - Annotation-only Visual Audit: D10 vs D20 vs D40
# No training, no predictions, no dataset modification
# ============================================================

import os, glob, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

random.seed(42)
np.random.seed(42)

IMAGE_DIR = "/content/RoadVision/data/custom_city_recovered/images"
LABEL_DIR = "/content/RoadVision/data/custom_city_recovered/labels"
OUT_DIR = "/content/RoadVision/class_visual_audit"
os.makedirs(OUT_DIR, exist_ok=True)

CLASS_MAP = {1: "D10", 2: "D20", 3: "D40"}
TARGET_CLASSES = list(CLASS_MAP.keys())
CROPS_PER_CLASS = 25
SIZE_BINS = ["small", "medium", "large"]  # by bbox area ratio tertiles (per class)

# ---------------------------------------------------------------
# 1. Read all YOLO annotations + extract per-box metadata
# ---------------------------------------------------------------
label_files = sorted(glob.glob(os.path.join(LABEL_DIR, "*.txt")))
records = []

for lp in label_files:
    stem = os.path.splitext(os.path.basename(lp))[0]
    img_path = None
    for ext in (".jpg", ".jpeg", ".png"):
        cand = os.path.join(IMAGE_DIR, stem + ext)
        if os.path.isfile(cand):
            img_path = cand
            break
    if img_path is None:
        continue

    with Image.open(img_path) as im:
        img_w, img_h = im.size

    with open(lp, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls_id = int(parts[0])
            cx, cy, w, h = map(float, parts[1:])
            if cls_id not in TARGET_CLASSES:
                continue
            box_w_px = w * img_w
            box_h_px = h * img_h
            x1 = (cx - w / 2) * img_w
            y1 = (cy - h / 2) * img_h
            x2 = (cx + w / 2) * img_w
            y2 = (cy + h / 2) * img_h
            area_ratio = (w * h)  # fraction of image area
            aspect_ratio = box_w_px / box_h_px if box_h_px > 0 else np.nan
            records.append({
                "image": os.path.basename(img_path),
                "image_path": img_path,
                "class_id": cls_id,
                "class_name": CLASS_MAP[cls_id],
                "x1": x1, "y1": y1, "x2": x2, "y2": y2,
                "img_w": img_w, "img_h": img_h,
                "bbox_area_ratio": area_ratio,
                "aspect_ratio": aspect_ratio,
            })

df = pd.DataFrame(records)
print(f"Total annotation files read: {len(label_files)}")
print(f"Total D10/D20/D40 boxes extracted: {len(df)}")
print(df["class_name"].value_counts())

# ---------------------------------------------------------------
# 2. Stratify by bbox area (small/medium/large) per class, using tertiles
# ---------------------------------------------------------------
def assign_size_bin(group):
    try:
        bins = pd.qcut(group["bbox_area_ratio"], q=3, labels=SIZE_BINS, duplicates="drop")
    except ValueError:
        # not enough distinct values to form 3 bins; fallback to single bin
        bins = pd.Series(["medium"] * len(group), index=group.index)
    return bins

df["size_bin"] = df.groupby("class_name", group_keys=False).apply(assign_size_bin)

# ---------------------------------------------------------------
# 3. Balanced, size-stratified selection: ~25 crops/class
# ---------------------------------------------------------------
def stratified_sample(class_df, n_target):
    bins_present = [b for b in SIZE_BINS if b in class_df["size_bin"].unique()]
    if not bins_present:
        return class_df.sample(min(n_target, len(class_df)), random_state=42)
    per_bin = max(1, n_target // len(bins_present))
    selected = []
    for b in bins_present:
        sub = class_df[class_df["size_bin"] == b]
        selected.append(sub.sample(min(per_bin, len(sub)), random_state=42))
    result = pd.concat(selected)
    # top up if short of target
    if len(result) < n_target:
        remaining = class_df.drop(result.index)
        extra = remaining.sample(min(n_target - len(result), len(remaining)), random_state=42)
        result = pd.concat([result, extra])
    return result.head(n_target)

selected_frames = []
for cname in CLASS_MAP.values():
    class_df = df[df["class_name"] == cname]
    if len(class_df) == 0:
        continue
    selected_frames.append(stratified_sample(class_df, CROPS_PER_CLASS))

selected_df = pd.concat(selected_frames).reset_index(drop=True) if selected_frames else pd.DataFrame()
print(f"\nSelected crops for contact sheet: {len(selected_df)}")
print(selected_df.groupby(["class_name", "size_bin"]).size())

# ---------------------------------------------------------------
# 4. Build contact sheet (grid: rows=classes, cols=crops)
# ---------------------------------------------------------------
n_cols = CROPS_PER_CLASS
n_rows = len(CLASS_MAP)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 1.2, n_rows * 1.4))

for row_i, cname in enumerate(CLASS_MAP.values()):
    class_crops = selected_df[selected_df["class_name"] == cname].reset_index(drop=True)
    for col_i in range(n_cols):
        ax = axes[row_i, col_i] if n_rows > 1 else axes[col_i]
        ax.axis("off")
        if col_i < len(class_crops):
            rec = class_crops.iloc[col_i]
            with Image.open(rec["image_path"]) as im:
                crop = im.crop((max(0, rec["x1"]), max(0, rec["y1"]),
                                 min(rec["img_w"], rec["x2"]), min(rec["img_h"], rec["y2"])))
            ax.imshow(crop)
            ax.set_title(rec["size_bin"][0].upper(), fontsize=6, pad=1)
        if col_i == 0:
            ax.text(-0.3, 0.5, cname, fontsize=10, fontweight="bold",
                     rotation=90, va="center", ha="center", transform=ax.transAxes)

plt.suptitle("D10 vs D20 vs D40 — Annotation Crop Audit (stratified by bbox size)", fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.97])

out_jpg = os.path.join(OUT_DIR, "d10_vs_d20_vs_d40.jpg")
plt.savefig(out_jpg, dpi=150)
plt.show()

# ---------------------------------------------------------------
# 5. Save CSV of full annotation stats (all extracted boxes, not just selected)
# ---------------------------------------------------------------
out_csv = os.path.join(OUT_DIR, "annotation_stats.csv")
df[["image", "class_name", "bbox_area_ratio", "aspect_ratio", "size_bin"]].to_csv(out_csv, index=False)

print(f"\nSaved contact sheet: {out_jpg}")
print(f"Saved annotation stats CSV: {out_csv}")

In [ ]:
import os
import glob
import hashlib
from collections import Counter, defaultdict

# ============================================================
# CONFIG
# ============================================================

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Search these likely source locations
SEARCH_ROOTS = [
    "/content/custom_city_images",
    "/content/roadvision_custom_city",
    "/content",
]

EXPECTED = {
    "n_images": 117,
    "n_background": 53,
    "n_boxes": 479,
    "D00": 10,
    "D10": 76,
    "D20": 220,
    "D40": 173,
}

CLASS_NAMES = {
    0: "D00",
    1: "D10",
    2: "D20",
    3: "D40",
}


# ============================================================
# HELPERS
# ============================================================

def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def get_image_dimensions(path):
    try:
        from PIL import Image
        with Image.open(path) as im:
            return im.size
    except Exception as e:
        return f"ERROR: {e}"


def find_label_for_image(img_path):
    """
    Search for a label having the same stem.
    Avoids assuming a particular directory layout.
    """
    stem = os.path.splitext(os.path.basename(img_path))[0]

    candidates = []

    # Common locations around image
    img_dir = os.path.dirname(img_path)
    candidates.extend([
        os.path.join(img_dir, stem + ".txt"),
        os.path.join(img_dir, "labels", stem + ".txt"),
        os.path.join(os.path.dirname(img_dir), "labels", stem + ".txt"),
    ])

    # Global search under /content
    for root in ["/content"]:
        candidates.extend(
            glob.glob(
                os.path.join(root, "**", stem + ".txt"),
                recursive=True
            )
        )

    # Unique paths
    seen = set()

    for p in candidates:
        if p in seen:
            continue
        seen.add(p)

        if os.path.isfile(p):
            return p

    return None


def read_label(label_path):
    if not label_path or not os.path.isfile(label_path):
        return [], Counter()

    rows = []
    counts = Counter()

    with open(label_path, "r", errors="ignore") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()

            if not line:
                continue

            parts = line.split()

            if len(parts) != 5:
                rows.append({
                    "line": line_no,
                    "raw": line,
                    "valid": False
                })
                continue

            try:
                cls = int(parts[0])
            except ValueError:
                rows.append({
                    "line": line_no,
                    "raw": line,
                    "valid": False
                })
                continue

            rows.append({
                "line": line_no,
                "raw": line,
                "valid": True,
                "class_id": cls
            })

            if cls in CLASS_NAMES:
                counts[CLASS_NAMES[cls]] += 1
            else:
                counts[f"class_{cls}"] += 1

    return rows, counts


# ============================================================
# 1. FIND RAW IMAGES
# ============================================================

print("=" * 70)
print("1. RAW IMAGE INVENTORY")
print("=" * 70)

all_images = []

for root in SEARCH_ROOTS:
    if not os.path.exists(root):
        continue

    for ext in IMAGE_EXTS:
        all_images.extend(
            glob.glob(
                os.path.join(root, "**", f"*{ext}"),
                recursive=True
            )
        )
        all_images.extend(
            glob.glob(
                os.path.join(root, "**", f"*{ext.upper()}"),
                recursive=True
            )
        )

# Remove exact path duplicates
all_images = sorted(set(os.path.abspath(p) for p in all_images))

print("Raw image files found:", len(all_images))

if len(all_images) != 117:
    print("WARNING: Expected exactly 117 raw images, but found",
          len(all_images))

# ============================================================
# 2. HASH ALL IMAGES
# ============================================================

print("\n" + "=" * 70)
print("2. CONTENT-HASH GROUPING")
print("=" * 70)

hash_groups = defaultdict(list)

for img_path in all_images:
    try:
        h = file_hash(img_path)
        hash_groups[h].append(img_path)
    except Exception as e:
        print("HASH ERROR:", img_path, e)

duplicate_groups = {
    h: paths
    for h, paths in hash_groups.items()
    if len(paths) > 1
}

print("Unique content hashes:", len(hash_groups))
print("Duplicate hash groups:", len(duplicate_groups))
print("Images involved in duplicate groups:",
      sum(len(v) for v in duplicate_groups.values()))

# ============================================================
# 3. DETAILED DUPLICATE ANALYSIS
# ============================================================

print("\n" + "=" * 70)
print("3. DUPLICATE IMAGE GROUPS")
print("=" * 70)

if not duplicate_groups:
    print("NO DUPLICATE GROUPS FOUND.")
else:

    duplicate_removed_count = 0
    duplicate_box_total = Counter()

    for group_idx, (h, paths) in enumerate(
        sorted(duplicate_groups.items()),
        start=1
    ):

        print("\n" + "-" * 70)
        print(f"DUPLICATE GROUP #{group_idx}")
        print("SHA256:", h)
        print("Files:", len(paths))

        group_label_signatures = []

        for i, img_path in enumerate(sorted(paths), 1):

            stem = os.path.splitext(
                os.path.basename(img_path)
            )[0]

            label_path = find_label_for_image(img_path)
            label_rows, class_counts = read_label(label_path)

            print(f"\n[{i}] Image")
            print("Path:", img_path)
            print("Stem:", stem)
            print("Dimensions:", get_image_dimensions(img_path))
            print("Label:", label_path)

            if label_path:
                print("Label contents:")

                for row in label_rows:
                    print("   ", row["raw"])

                print("Box count:",
                      sum(class_counts.values()))

                print("Class counts:",
                      dict(class_counts))

                duplicate_box_total.update(class_counts)

                # Exact label signature
                label_signature = tuple(
                    row["raw"] for row in label_rows
                )
                group_label_signatures.append(
                    label_signature
                )

            else:
                print("NO LABEL FOUND")
                group_label_signatures.append(None)

        # Compare labels within duplicate image group
        labels_identical = (
            len(set(group_label_signatures)) == 1
        )

        print("\nLABEL COMPARISON:")
        print(
            "Identical labels:",
            labels_identical
        )

        if labels_identical:
            print(
                "Diagnosis for this group: "
                "EXACT IMAGE DUPLICATE WITH SAME LABEL CONTENT"
            )
        else:
            print(
                "Diagnosis for this group: "
                "SAME IMAGE CONTENT BUT DIFFERENT LABEL CONTENT"
            )

        duplicate_removed_count += len(paths) - 1

    print("\nTotal extra duplicate files:",
          duplicate_removed_count)

    print("Boxes associated with duplicate groups:",
          dict(duplicate_box_total))


# ============================================================
# 4. FULL RAW INVENTORY — INCLUDING DUPLICATES
# ============================================================

print("\n" + "=" * 70)
print("4. RAW 117-FILE INVENTORY")
print("=" * 70)

raw_class_counts = Counter()
raw_background_count = 0
raw_box_count = 0

for img_path in all_images:

    label_path = find_label_for_image(img_path)
    _, counts = read_label(label_path)

    box_count = sum(counts.values())

    raw_box_count += box_count

    for cls, n in counts.items():
        raw_class_counts[cls] += n

    if box_count == 0:
        raw_background_count += 1

print("Raw images:", len(all_images))
print("Raw background images:", raw_background_count)
print("Raw total boxes:", raw_box_count)
print("Raw class totals:", dict(raw_class_counts))


# ============================================================
# 5. COMPARE RAW INVENTORY TO EXPECTED
# ============================================================

print("\n" + "=" * 70)
print("5. EXPECTED VS RAW")
print("=" * 70)

print("\nEXPECTED:")
print(EXPECTED)

print("\nRAW:")
print({
    "n_images": len(all_images),
    "n_background": raw_background_count,
    "n_boxes": raw_box_count,
    **{
        cls: raw_class_counts.get(cls, 0)
        for cls in ["D00", "D10", "D20", "D40"]
    }
})


# ============================================================
# 6. CHECK WHETHER DUPLICATES EXPLAIN MISSING BOXES
# ============================================================

print("\n" + "=" * 70)
print("6. CAN DUPLICATES EXPLAIN THE MISSING BOXES?")
print("=" * 70)

missing = {
    cls: EXPECTED[cls] - raw_class_counts.get(cls, 0)
    for cls in ["D00", "D10", "D20", "D40"]
}

print("Missing relative to expected:", missing)
print("Total missing boxes:", sum(missing.values()))

duplicate_class_counts = Counter()

for h, paths in duplicate_groups.items():

    # Count only ONE copy as the representative content
    # and compare all duplicate copies.
    for img_path in paths[1:]:
        label_path = find_label_for_image(img_path)
        _, counts = read_label(label_path)
        duplicate_class_counts.update(counts)

print(
    "Boxes contained in extra duplicate copies:",
    dict(duplicate_class_counts)
)

print("\nInterpretation:")

for cls in ["D00", "D10", "D20", "D40"]:
    print(
        f"{cls}: missing={missing[cls]}, "
        f"duplicate-extra={duplicate_class_counts.get(cls, 0)}"
    )


# ============================================================
# 7. DIAGNOSIS
# ============================================================

print("\n" + "=" * 70)
print("7. FINAL DIAGNOSIS")
print("=" * 70)

raw_matches_expected = (
    len(all_images) == EXPECTED["n_images"]
    and raw_background_count == EXPECTED["n_background"]
    and raw_box_count == EXPECTED["n_boxes"]
    and all(
        raw_class_counts.get(cls, 0) == EXPECTED[cls]
        for cls in ["D00", "D10", "D20", "D40"]
    )
)

if raw_matches_expected:
    print("A/B: RAW INVENTORY MATCHES EXPECTED.")
    print(
        "The mismatch is caused by content-hash deduplication."
    )

elif len(all_images) != EXPECTED["n_images"]:
    print("C) WRONG/INCOMPLETE SOURCE INVENTORY")
    print(
        f"Expected {EXPECTED['n_images']} raw images, "
        f"found {len(all_images)}."
    )

else:
    print(
        "D) RAW IMAGE COUNT IS 117, BUT LABEL/BOX INVENTORY "
        "DOES NOT MATCH EXPECTED."
    )

    print(
        "Further investigation of label mapping/source ZIP "
        "is required."
    )

print("\nNO FILES WERE MODIFIED.")
print("NO FILES WERE DELETED.")
print("NO DATASET WAS CREATED.")
print("NO SPLIT WAS PERFORMED.")
print("NO TRAINING/EVALUATION WAS PERFORMED.")

In [ ]:
!pip install -q iterstrat

In [ ]:
!pip install -q iterative-stratification

In [ ]:
import zipfile
import hashlib
import os
from collections import Counter, defaultdict
from PIL import Image
import io

ZIP_PATH = "/content/custom_city_images.zip.zip"

EXPECTED = {
    "images": 117,
    "background": 53,
    "boxes": 479,
    "D00": 10,
    "D10": 76,
    "D20": 220,
    "D40": 173,
}

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
LABEL_EXT = ".txt"

CLASS_NAMES = {
    0: "D00",
    1: "D10",
    2: "D20",
    3: "D40",
}

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    files = [x for x in z.namelist() if not x.endswith("/")]

    images = [
        x for x in files
        if os.path.splitext(x)[1].lower() in IMG_EXTS
    ]

    labels = {
        os.path.splitext(os.path.basename(x))[0]: x
        for x in files
        if os.path.splitext(x)[1].lower() == LABEL_EXT
    }

    print("=" * 60)
    print("SOURCE ZIP INVENTORY")
    print("=" * 60)
    print("ZIP:", ZIP_PATH)
    print("Total files:", len(files))
    print("Images:", len(images))
    print("Labels:", len(labels))

    # --------------------------------------------------------
    # IMAGE HASH DUPLICATES INSIDE THIS ZIP ONLY
    # --------------------------------------------------------

    hash_groups = defaultdict(list)

    for img_name in images:
        data = z.read(img_name)
        h = hashlib.sha256(data).hexdigest()
        hash_groups[h].append(img_name)

    duplicate_groups = {
        h: paths
        for h, paths in hash_groups.items()
        if len(paths) > 1
    }

    print("\nUnique image hashes:", len(hash_groups))
    print("Duplicate hash groups:", len(duplicate_groups))

    # --------------------------------------------------------
    # INVENTORY / LABEL COUNTS
    # --------------------------------------------------------

    class_counts = Counter()
    background = 0
    total_boxes = 0
    missing_labels = []

    records = []

    for img_name in sorted(images):

        stem = os.path.splitext(os.path.basename(img_name))[0]

        # Try exact basename match first
        label_name = labels.get(stem)

        # Also support labels stored in another directory
        if label_name is None:
            candidates = [
                x for x in files
                if os.path.splitext(os.path.basename(x))[0] == stem
                and os.path.splitext(x)[1].lower() == ".txt"
            ]
            if candidates:
                label_name = candidates[0]

        counts = Counter()
        label_lines = []

        if label_name is not None:
            raw = z.read(label_name).decode("utf-8", errors="ignore")

            for line in raw.splitlines():
                line = line.strip()

                if not line:
                    continue

                parts = line.split()

                if len(parts) != 5:
                    continue

                try:
                    cls = int(parts[0])
                except ValueError:
                    continue

                label_lines.append(line)

                if cls in CLASS_NAMES:
                    counts[CLASS_NAMES[cls]] += 1

        else:
            missing_labels.append(img_name)

        box_count = sum(counts.values())

        if box_count == 0:
            background += 1

        total_boxes += box_count
        class_counts.update(counts)

        records.append({
            "image": img_name,
            "label": label_name,
            "boxes": box_count,
            "classes": dict(counts),
        })

    # --------------------------------------------------------
    # RESULTS
    # --------------------------------------------------------

    actual = {
        "images": len(images),
        "background": background,
        "boxes": total_boxes,
        "D00": class_counts["D00"],
        "D10": class_counts["D10"],
        "D20": class_counts["D20"],
        "D40": class_counts["D40"],
    }

    print("\n" + "=" * 60)
    print("ACTUAL INVENTORY")
    print("=" * 60)

    for k, v in actual.items():
        print(f"{k}: {v}")

    print("\nEXPECTED:")
    for k, v in EXPECTED.items():
        print(f"{k}: {v}")

    # --------------------------------------------------------
    # MISSING LABELS
    # --------------------------------------------------------

    print("\n" + "=" * 60)
    print("LABEL CHECK")
    print("=" * 60)

    print("Images without matching labels:", len(missing_labels))

    if missing_labels:
        for x in missing_labels:
            print("  ", x)

    # --------------------------------------------------------
    # DUPLICATE DETAILS
    # --------------------------------------------------------

    if duplicate_groups:
        print("\n" + "=" * 60)
        print("DUPLICATES INSIDE ORIGINAL ZIP")
        print("=" * 60)

        for i, (h, paths) in enumerate(
            sorted(duplicate_groups.items()), 1
        ):
            print(f"\nDuplicate group #{i}")
            print("SHA256:", h)

            for p in paths:
                data = z.read(p)

                try:
                    with Image.open(io.BytesIO(data)) as im:
                        dimensions = im.size
                except Exception:
                    dimensions = "unknown"

                stem = os.path.splitext(
                    os.path.basename(p)
                )[0]

                label_name = labels.get(stem)

                counts = Counter()

                if label_name:
                    raw = z.read(label_name).decode(
                        "utf-8",
                        errors="ignore"
                    )

                    for line in raw.splitlines():
                        parts = line.strip().split()

                        if len(parts) == 5:
                            try:
                                cls = int(parts[0])
                                if cls in CLASS_NAMES:
                                    counts[CLASS_NAMES[cls]] += 1
                            except ValueError:
                                pass

                print("  Image:", p)
                print("  Dimensions:", dimensions)
                print("  Label:", label_name)
                print("  Boxes:", sum(counts.values()))
                print("  Classes:", dict(counts))

    # --------------------------------------------------------
    # FINAL DIAGNOSIS
    # --------------------------------------------------------

    exact_match = actual == EXPECTED

    print("\n" + "=" * 60)
    print("FINAL DIAGNOSIS")
    print("=" * 60)

    if exact_match and not duplicate_groups:
        print("A) SOURCE ARCHIVE MATCHES EXPECTED INVENTORY")

    elif duplicate_groups:
        print("B) SOURCE ARCHIVE CONTAINS DUPLICATE IMAGES")

    elif (
        len(images) < EXPECTED["images"]
        or missing_labels
        or actual["boxes"] < EXPECTED["boxes"]
    ):
        print("C) SOURCE ARCHIVE IS MISSING IMAGES/LABELS")

    else:
        print("D) SOURCE ARCHIVE HAS A DIFFERENT INVENTORY")

In [ ]:
import os

candidates = [
    "/content/custom_city_images.zip.zip",
    "/content/custom_city_images.zip",
    "/content/custom_city_labels.zip",
    "/content/custom_city_labels.zip.zip",
]

print("POSSIBLE ORIGINAL DATA SOURCES")
print("=" * 60)

for p in candidates:
    print(f"{p}: {'EXISTS' if os.path.exists(p) else 'NOT FOUND'}")

print("\nDIRECT /content FILES")
print("=" * 60)

for name in sorted(os.listdir("/content")):
    low = name.lower()
    if any(x in low for x in ["custom", "city", "label", "annotation"]):
        print(name)


In [ ]:
import zipfile
import os
from collections import Counter

ZIP_PATH = "/content/roadvision_custom_city (1).zip"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    files = [x for x in z.namelist() if not x.endswith("/")]

print("=" * 60)
print("ROADVISION CUSTOM CITY ZIP")
print("=" * 60)
print("Archive:", ZIP_PATH)
print("Total entries:", len(files))

# Show directory/file structure only
print("\nARCHIVE CONTENT:")
for x in files:
    print(x)

# Categorize
ext_counts = Counter(
    os.path.splitext(x)[1].lower() or "<no extension>"
    for x in files
)

print("\n" + "=" * 60)
print("FILE EXTENSIONS")
print("=" * 60)

for ext, count in sorted(ext_counts.items()):
    print(f"{ext}: {count}")

# Explicitly identify likely images / labels / YAML / metadata
images = [
    x for x in files
    if os.path.splitext(x)[1].lower()
    in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
]

labels = [
    x for x in files
    if os.path.splitext(x)[1].lower() == ".txt"
]

yaml_files = [
    x for x in files
    if os.path.splitext(x)[1].lower()
    in {".yaml", ".yml"}
]

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print("Images:", len(images))
print("TXT labels:", len(labels))
print("YAML/YML:", len(yaml_files))

print("\nLabel files:")
for x in labels:
    print(" ", x)

print("\nYAML files:")
for x in yaml_files:
    print(" ", x)

In [ ]:
import zipfile
from collections import Counter
import os

IMG_ZIP = "/content/custom_city_images.zip.zip"
LBL_ZIP = "/content/roadvision_custom_city (1).zip"

CLASS_NAMES = {
    0: "D00",
    1: "D10",
    2: "D20",
    3: "D40",
}

# ------------------------------------------------------------
# Read image archive
# ------------------------------------------------------------

with zipfile.ZipFile(IMG_ZIP, "r") as iz:
    image_files = [
        x for x in iz.namelist()
        if not x.endswith("/")
        and os.path.splitext(x)[1].lower()
        in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    ]

# ------------------------------------------------------------
# Read label archive
# ------------------------------------------------------------

with zipfile.ZipFile(LBL_ZIP, "r") as lz:

    label_files = [
        x for x in lz.namelist()
        if not x.endswith("/")
        and x.lower().endswith(".txt")
        and x != "train.txt"
    ]

    data_yaml = (
        lz.read("data.yaml").decode("utf-8", errors="ignore")
        if "data.yaml" in lz.namelist()
        else ""
    )

    # Match by basename:
    # images (29).jpg -> images (29).txt
    labels_by_stem = {
        os.path.splitext(os.path.basename(x))[0]: x
        for x in label_files
    }

    class_counts = Counter()
    total_boxes = 0
    background = 0
    matched = 0
    missing = []
    label_only = []

    image_stems = set()

    for img in image_files:

        stem = os.path.splitext(
            os.path.basename(img)
        )[0]

        image_stems.add(stem)

        label_path = labels_by_stem.get(stem)

        if label_path is None:
            missing.append(img)
            background += 1
            continue

        matched += 1

        raw = lz.read(label_path).decode(
            "utf-8",
            errors="ignore"
        )

        boxes = 0
        counts = Counter()

        for line in raw.splitlines():

            parts = line.strip().split()

            if len(parts) != 5:
                continue

            try:
                cls = int(parts[0])
            except ValueError:
                continue

            if cls in CLASS_NAMES:
                counts[CLASS_NAMES[cls]] += 1
                boxes += 1

        if boxes == 0:
            background += 1

        total_boxes += boxes
        class_counts.update(counts)

    # Labels that don't correspond to an image
    for stem, path in labels_by_stem.items():
        if stem not in image_stems:
            label_only.append(path)

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

print("=" * 65)
print("IMAGE + LABEL ARCHIVE CROSS-CHECK")
print("=" * 65)

print("Images:", len(image_files))
print("Labels:", len(label_files))
print("Matched image-label pairs:", matched)
print("Images without labels:", len(missing))
print("Labels without images:", len(label_only))

print("\n" + "=" * 65)
print("CALCULATED INVENTORY")
print("=" * 65)

print("Images:", len(image_files))
print("Background:", background)
print("Total boxes:", total_boxes)

for cls in ["D00", "D10", "D20", "D40"]:
    print(f"{cls}:", class_counts[cls])

print("\n" + "=" * 65)
print("EXPECTED")
print("=" * 65)

print("Images: 117")
print("Background: 53")
print("Total boxes: 479")
print("D00: 10")
print("D10: 76")
print("D20: 220")
print("D40: 173")

print("\n" + "=" * 65)
print("DATA.YAML")
print("=" * 65)
print(data_yaml)

if missing:
    print("\nImages without labels:")
    for x in missing:
        print(" ", x)

if label_only:
    print("\nLabels without matching images:")
    for x in label_only:
        print(" ", x)

actual = {
    "images": len(image_files),
    "background": background,
    "boxes": total_boxes,
    "D00": class_counts["D00"],
    "D10": class_counts["D10"],
    "D20": class_counts["D20"],
    "D40": class_counts["D40"],
}

expected = {
    "images": 117,
    "background": 53,
    "boxes": 479,
    "D00": 10,
    "D10": 76,
    "D20": 220,
    "D40": 173,
}

print("\n" + "=" * 65)
print("DIAGNOSIS")
print("=" * 65)

if actual == expected and not missing and not label_only:
    print("A) SOURCE ARCHIVE + LABEL ARCHIVE MATCH EXPECTED INVENTORY")
elif missing or label_only:
    print("C) IMAGE/LABEL MATCHING IS INCOMPLETE")
else:
    print("D) IMAGE + LABEL INVENTORY DIFFERS FROM EXPECTED")

In [ ]:
# ============================================================
# EXACT CUSTOM-CITY SPLIT RECONSTRUCTION
# Source:
#   /content/custom_city_images.zip.zip
#   /content/roadvision_custom_city (1).zip
#
# IMPORTANT:
# - NO deduplication
# - NO training
# - NO evaluation
# - ALL 117 images are retained
# ============================================================

import zipfile
import os
import sys
import subprocess
from collections import Counter
import numpy as np

# Install/import iterative stratification
try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "iterative-stratification"],
        check=True
    )
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit


IMG_ZIP = "/content/custom_city_images.zip.zip"
LBL_ZIP = "/content/roadvision_custom_city (1).zip"

CLASS_NAMES = ["D00", "D10", "D20", "D40"]

# ------------------------------------------------------------
# READ IMAGES
# ------------------------------------------------------------

with zipfile.ZipFile(IMG_ZIP, "r") as iz:
    image_files = sorted([
        x for x in iz.namelist()
        if not x.endswith("/")
        and os.path.splitext(x)[1].lower()
        in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    ])

# ------------------------------------------------------------
# READ LABELS
# ------------------------------------------------------------

with zipfile.ZipFile(LBL_ZIP, "r") as lz:
    label_files = [
        x for x in lz.namelist()
        if not x.endswith("/")
        and x.lower().endswith(".txt")
        and x != "train.txt"
    ]

    labels_by_stem = {
        os.path.splitext(os.path.basename(x))[0]: x
        for x in label_files
    }

    records = []

    for img_path in image_files:

        stem = os.path.splitext(
            os.path.basename(img_path)
        )[0]

        label_path = labels_by_stem.get(stem)

        present = [0, 0, 0, 0]
        box_count = 0

        if label_path is not None:
            raw = lz.read(label_path).decode(
                "utf-8",
                errors="ignore"
            )

            for line in raw.splitlines():
                parts = line.strip().split()

                if len(parts) != 5:
                    continue

                try:
                    cls = int(parts[0])
                except ValueError:
                    continue

                if 0 <= cls < 4:
                    present[cls] = 1
                    box_count += 1

        records.append({
            "image": img_path,
            "stem": stem,
            "label": label_path,
            "present": present,
            "boxes": box_count,
            "background": box_count == 0
        })

# ------------------------------------------------------------
# SANITY CHECK SOURCE
# ------------------------------------------------------------

assert len(records) == 117, len(records)
assert sum(r["background"] for r in records) == 53
assert sum(r["boxes"] for r in records) == 479

print("=" * 65)
print("SOURCE VERIFIED")
print("=" * 65)
print("Images:", len(records))
print("Background:", sum(r["background"] for r in records))
print("Boxes:", sum(r["boxes"] for r in records))

# ------------------------------------------------------------
# MULTILABEL MATRIX
# ------------------------------------------------------------

X = np.zeros((len(records), 1))
Y = np.array([r["present"] for r in records])

# ------------------------------------------------------------
# FIRST SPLIT
# 117 -> 80 train + 37 temporary
# ------------------------------------------------------------

split1 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=37,
    random_state=42
)

train_idx, temp_idx = next(
    split1.split(X, Y)
)

# ------------------------------------------------------------
# SECOND SPLIT
# 37 -> 16 val + 21 test
# ------------------------------------------------------------

X_temp = X[temp_idx]
Y_temp = Y[temp_idx]

split2 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=21,
    random_state=42
)

val_rel_idx, test_rel_idx = next(
    split2.split(X_temp, Y_temp)
)

val_idx = temp_idx[val_rel_idx]
test_idx = temp_idx[test_rel_idx]

# ------------------------------------------------------------
# CONVERT TO SETS
# ------------------------------------------------------------

train_set = set(train_idx.tolist())
val_set = set(val_idx.tolist())
test_set = set(test_idx.tolist())

# ------------------------------------------------------------
# SPLIT VALIDATION
# ------------------------------------------------------------

assert len(train_set) == 80
assert len(val_set) == 16
assert len(test_set) == 21

assert not train_set & val_set
assert not train_set & test_set
assert not val_set & test_set

assert len(train_set | val_set | test_set) == 117

print("\n" + "=" * 65)
print("SPLIT SIZES")
print("=" * 65)
print("Train:", len(train_set))
print("Val:  ", len(val_set))
print("Test: ", len(test_set))
print("Total:", len(train_set | val_set | test_set))

# ------------------------------------------------------------
# DETAILED SPLIT REPORT
# ------------------------------------------------------------

def report(name, indices):
    print("\n" + "-" * 65)
    print(name)

    class_presence = Y[list(indices)].sum(axis=0)

    box_total = sum(
        records[i]["boxes"]
        for i in indices
    )

    bg_total = sum(
        records[i]["background"]
        for i in indices
    )

    print("Images:", len(indices))
    print("Background:", bg_total)
    print("Boxes:", box_total)

    for j, cls in enumerate(CLASS_NAMES):
        print(
            f"{cls} image-presence:",
            int(class_presence[j])
        )

report("TRAIN", train_set)
report("VAL", val_set)
report("TEST", test_set)

# ------------------------------------------------------------
# D10 CHECK
# ------------------------------------------------------------

train_d10 = [
    records[i]["stem"]
    for i in train_set
    if records[i]["present"][1] == 1
]

val_d10 = [
    records[i]["stem"]
    for i in val_set
    if records[i]["present"][1] == 1
]

test_d10 = [
    records[i]["stem"]
    for i in test_set
    if records[i]["present"][1] == 1
]

print("\n" + "=" * 65)
print("D10 DISTRIBUTION")
print("=" * 65)

print("Train D10 images:", len(train_d10))
print("Val D10 images:  ", len(val_d10))
print("Test D10 images: ", len(test_d10))

print("\nTrain D10 stems:")
for x in sorted(train_d10):
    print(" ", x)

# ------------------------------------------------------------
# FINAL GATE
# ------------------------------------------------------------

all_good = (
    len(records) == 117
    and len(train_set) == 80
    and len(val_set) == 16
    and len(test_set) == 21
    and len(train_set | val_set | test_set) == 117
    and not (train_set & val_set)
    and not (train_set & test_set)
    and not (val_set & test_set)
    and len(train_d10) > 0
)

print("\n" + "=" * 65)

if all_good:
    print("GATE PASSED: SPLIT RECONSTRUCTION VALID")
else:
    print("GATE FAILED: DO NOT TRAIN")

print("=" * 65)

In [ ]:
# ============================================================
# BUILD VERIFIED YOLO DATASET FROM THE RECONSTRUCTED SPLIT
# ============================================================

import zipfile
import os
import shutil
from pathlib import Path

IMG_ZIP = "/content/custom_city_images.zip.zip"
LBL_ZIP = "/content/roadvision_custom_city (1).zip"

OUT = Path("/content/RoadVision/data/custom_city_recovered")

# Clean ONLY the target reconstruction directory
if OUT.exists():
    shutil.rmtree(OUT)

for split in ["train", "val", "test"]:
    (OUT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT / "labels" / split).mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Reconstruct the SAME split again
# ------------------------------------------------------------

import numpy as np
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

In [ ]:
!pip install iterstrat

In [ ]:
# ============================================================
# BUILD VERIFIED YOLO DATASET
# Uses the EXACT split from the previous successful cell
# ============================================================

import zipfile
import os
import shutil
from pathlib import Path

# ------------------------------------------------------------
# IMPORTANT:
# These variables must still exist from the previous cell:
# records, train_set, val_set, test_set
# ------------------------------------------------------------

required = ["records", "train_set", "val_set", "test_set"]

missing_vars = [
    x for x in required
    if x not in globals()
]

if missing_vars:
    raise RuntimeError(
        "Missing variables: "
        + ", ".join(missing_vars)
        + ". Run the previous successful split cell first."
    )

IMG_ZIP = "/content/custom_city_images.zip.zip"
LBL_ZIP = "/content/roadvision_custom_city (1).zip"

OUT = Path("/content/RoadVision/data/custom_city_recovered")

# ------------------------------------------------------------
# Recreate target directory
# ------------------------------------------------------------

if OUT.exists():
    shutil.rmtree(OUT)

for split in ["train", "val", "test"]:
    (OUT / "images" / split).mkdir(
        parents=True,
        exist_ok=True
    )
    (OUT / "labels" / split).mkdir(
        parents=True,
        exist_ok=True
    )

# ------------------------------------------------------------
# Open source archives READ-ONLY
# ------------------------------------------------------------

with zipfile.ZipFile(IMG_ZIP, "r") as iz, \
     zipfile.ZipFile(LBL_ZIP, "r") as lz:

    split_sets = {
        "train": train_set,
        "val": val_set,
        "test": test_set,
    }

    copied_images = 0
    copied_labels = 0
    copied_backgrounds = 0

    # --------------------------------------------------------
    # Copy each image into its verified split
    # --------------------------------------------------------

    for split, indices in split_sets.items():

        for idx in sorted(indices):

            record = records[idx]

            image_archive_path = record["image"]
            label_archive_path = record["label"]

            filename = os.path.basename(image_archive_path)

            # Image
            image_data = iz.read(image_archive_path)

            image_out = (
                OUT
                / "images"
                / split
                / filename
            )

            image_out.write_bytes(image_data)
            copied_images += 1

            # Label
            if label_archive_path is not None:

                label_filename = (
                    os.path.splitext(filename)[0]
                    + ".txt"
                )

                label_data = lz.read(label_archive_path)

                label_out = (
                    OUT
                    / "labels"
                    / split
                    / label_filename
                )

                label_out.write_bytes(label_data)

                copied_labels += 1

            else:
                copied_backgrounds += 1

# ------------------------------------------------------------
# Create data.yaml
# ------------------------------------------------------------

yaml_text = """path: /content/RoadVision/data/custom_city_recovered
train: images/train
val: images/val
test: images/test

names:
  0: D00
  1: D10
  2: D20
  3: D40
"""

(OUT / "data.yaml").write_text(
    yaml_text,
    encoding="utf-8"
)

# ------------------------------------------------------------
# FINAL FILESYSTEM AUDIT
# ------------------------------------------------------------

print("=" * 65)
print("RECONSTRUCTED YOLO DATASET")
print("=" * 65)

print("Output:", OUT)

for split in ["train", "val", "test"]:

    image_dir = OUT / "images" / split
    label_dir = OUT / "labels" / split

    imgs = [
        x for x in image_dir.iterdir()
        if x.is_file()
    ]

    lbls = [
        x for x in label_dir.iterdir()
        if x.is_file()
    ]

    print(f"\n{split.upper()}")
    print("  Images:", len(imgs))
    print("  Labels:", len(lbls))

print("\n" + "=" * 65)
print("COPY TOTALS")
print("=" * 65)

print("Images copied:", copied_images)
print("Labels copied:", copied_labels)
print("Background images:", copied_backgrounds)

# ------------------------------------------------------------
# Expected structural checks
# ------------------------------------------------------------

assert copied_images == 117
assert copied_labels == 64
assert copied_backgrounds == 53

assert len(list((OUT / "images" / "train").iterdir())) == 80
assert len(list((OUT / "images" / "val").iterdir())) == 16
assert len(list((OUT / "images" / "test").iterdir())) == 21

assert len(list((OUT / "labels" / "train").iterdir())) == 43
assert len(list((OUT / "labels" / "val").iterdir())) == 8
assert len(list((OUT / "labels" / "test").iterdir())) == 13

print("\n" + "=" * 65)
print("GATE")
print("=" * 65)
print("PASS: 117 images / 64 labels / 53 backgrounds")
print("PASS: 80 train / 16 val / 21 test")
print("PASS: NO DEDUPLICATION")
print("PASS: DATASET READY FOR FINAL AUDIT")
print("=" * 65)

In [ ]:
from pathlib import Path
from collections import Counter
import hashlib

ROOT = Path("/content/RoadVision/data/custom_city_recovered")

CLASS_NAMES = {
    0: "D00",
    1: "D10",
    2: "D20",
    3: "D40",
}

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

total_images = 0
total_boxes = 0
total_background = 0
class_counts = Counter()

all_hashes = {}
cross_split_duplicates = []

print("=" * 70)
print("FINAL RECONSTRUCTED DATASET AUDIT")
print("=" * 70)

for split in ["train", "val", "test"]:

    image_dir = ROOT / "images" / split
    label_dir = ROOT / "labels" / split

    images = sorted(
        p for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    )

    labels = sorted(
        p for p in label_dir.iterdir()
        if p.is_file() and p.suffix.lower() == ".txt"
    )

    image_stems = {p.stem for p in images}
    label_stems = {p.stem for p in labels}

    missing_labels = image_stems - label_stems
    orphan_labels = label_stems - image_stems

    split_boxes = 0
    split_background = 0
    split_classes = Counter()

    for img in images:

        # Hash for cross-split duplicate check
        h = hashlib.sha256(img.read_bytes()).hexdigest()

        if h in all_hashes:
            previous_split, previous_file = all_hashes[h]

            if previous_split != split:
                cross_split_duplicates.append(
                    (previous_split, previous_file, split, img.name)
                )
        else:
            all_hashes[h] = (split, img.name)

        label = label_dir / f"{img.stem}.txt"

        if not label.exists():
            split_background += 1
            continue

        boxes = 0

        for line in label.read_text(
            encoding="utf-8",
            errors="ignore"
        ).splitlines():

            parts = line.strip().split()

            if len(parts) != 5:
                continue

            try:
                cls = int(parts[0])
            except ValueError:
                continue

            if cls in CLASS_NAMES:
                class_counts[CLASS_NAMES[cls]] += 1
                split_classes[CLASS_NAMES[cls]] += 1
                boxes += 1

        split_boxes += boxes

    total_images += len(images)
    total_boxes += split_boxes
    total_background += split_background

    print(f"\n{split.upper()}")
    print("-" * 70)
    print("Images:", len(images))
    print("Labels:", len(labels))
    print("Background:", split_background)
    print("Boxes:", split_boxes)
    print("Classes:", dict(split_classes))
    print("Images missing labels:", len(missing_labels))
    print("Orphan labels:", len(orphan_labels))

print("\n" + "=" * 70)
print("GLOBAL TOTALS")
print("=" * 70)

print("Images:", total_images)
print("Background:", total_background)
print("Boxes:", total_boxes)

for cls in ["D00", "D10", "D20", "D40"]:
    print(f"{cls}:", class_counts[cls])

print("\nCross-split duplicate groups:", len(cross_split_duplicates))

if cross_split_duplicates:
    for x in cross_split_duplicates:
        print(x)

print("\n" + "=" * 70)
print("EXPECTED")
print("=" * 70)

expected = {
    "images": 117,
    "background": 53,
    "boxes": 479,
    "D00": 10,
    "D10": 76,
    "D20": 220,
    "D40": 173,
}

actual = {
    "images": total_images,
    "background": total_background,
    "boxes": total_boxes,
    "D00": class_counts["D00"],
    "D10": class_counts["D10"],
    "D20": class_counts["D20"],
    "D40": class_counts["D40"],
}

print("\n" + "=" * 70)
print("FINAL GATE")
print("=" * 70)

if actual == expected and not cross_split_duplicates:
    print("PASS: EXACT INVENTORY MATCH")
    print("PASS: NO CROSS-SPLIT DUPLICATES")
    print("PASS: DATASET READY")
else:
    print("FAIL: DO NOT TRAIN")
    print("\nActual:", actual)
    print("Expected:", expected)

In [ ]:
from pathlib import Path
import hashlib
from collections import defaultdict, Counter

ROOT = Path("/content/RoadVision/data/custom_city_recovered")

hash_groups = defaultdict(list)

for split in ["train", "val", "test"]:
    image_dir = ROOT / "images" / split

    for img in image_dir.iterdir():
        if img.is_file():
            h = hashlib.sha256(img.read_bytes()).hexdigest()
            hash_groups[h].append(
                (split, img.name)
            )

duplicates = {
    h: items
    for h, items in hash_groups.items()
    if len(items) > 1
}

print("=" * 70)
print("DUPLICATE GROUP ANALYSIS")
print("=" * 70)

print("Duplicate hash groups:", len(duplicates))

for i, (h, items) in enumerate(
    duplicates.items(), 1
):
    print(f"\nGROUP #{i}")
    print("SHA256:", h)

    for split, filename in sorted(items):

        label = (
            ROOT
            / "labels"
            / split
            / (Path(filename).stem + ".txt")
        )

        counts = Counter()

        if label.exists():
            for line in label.read_text(
                encoding="utf-8",
                errors="ignore"
            ).splitlines():

                parts = line.strip().split()

                if len(parts) == 5:
                    try:
                        cls = int(parts[0])
                        counts[cls] += 1
                    except ValueError:
                        pass

        print(
            f"  {split:5s} | "
            f"{filename:45s} | "
            f"boxes={sum(counts.values()):2d} | "
            f"classes={dict(counts)}"
        )

print("\n" + "=" * 70)
print("INTERPRETATION")
print("=" * 70)

if any(
    len({split for split, _ in items}) > 1
    for items in duplicates.values()
):
    print("WARNING: identical image content crosses splits.")
    print("Do NOT train on this split until the intended split rule is established.")
else:
    print("PASS: duplicate contents stay within splits.")

In [ ]:
import zipfile

ZIP_PATH = "/content/roadvision_custom_city (1).zip"

with zipfile.ZipFile(ZIP_PATH, "r") as z:

    train_txt = z.read("train.txt").decode(
        "utf-8",
        errors="ignore"
    )

print("=" * 70)
print("ORIGINAL TRAIN.TXT")
print("=" * 70)

print(train_txt)

lines = [
    x.strip()
    for x in train_txt.splitlines()
    if x.strip()
]

print("\n" + "=" * 70)
print("TRAIN.TXT SUMMARY")
print("=" * 70)

print("Non-empty lines:", len(lines))

print("\nFirst 10:")
for x in lines[:10]:
    print(x)

print("\nLast 10:")
for x in lines[-10:]:
    print(x)

In [ ]:
from pathlib import Path

ROOT = Path("/content/RoadVision/data/custom_city_recovered")
MANIFEST = ROOT / "split_manifest.txt"

lines = []

for split in ["train", "val", "test"]:
    image_dir = ROOT / "images" / split

    for img in sorted(image_dir.iterdir()):
        if img.is_file():
            lines.append(
                f"{split}\t{img.name}"
            )

MANIFEST.write_text(
    "\n".join(lines) + "\n",
    encoding="utf-8"
)

print("=" * 65)
print("SPLIT MANIFEST FROZEN")
print("=" * 65)

print("Path:", MANIFEST)
print("Entries:", len(lines))

from collections import Counter

counts = Counter(
    line.split("\t")[0]
    for line in lines
)

print("Train:", counts["train"])
print("Val:", counts["val"])
print("Test:", counts["test"])

assert len(lines) == 117
assert counts["train"] == 80
assert counts["val"] == 16
assert counts["test"] == 21

print("\nPASS: 80/16/21 split frozen.")

In [ ]:
from ultralytics import YOLO

MODEL_PATH = "/content/drive/MyDrive/RoadVision/experiment4_custom_finetune/best.pt"
DATA_YAML = "/content/RoadVision/data/custom_city_recovered/data.yaml"

model = YOLO(MODEL_PATH)

results = model.val(
    data=DATA_YAML,
    split="test",
    imgsz=640,
    conf=0.001,
    iou=0.7,
    plots=True,
    verbose=True
)

print("\n" + "=" * 65)
print("TEST EVALUATION COMPLETE")
print("=" * 65)
print("Results:", results)

In [ ]:
from ultralytics import YOLO
from pathlib import Path

MODEL_PATH = "/content/drive/MyDrive/RoadVision/experiment4_custom_finetune/best.pt"
TEST_DIR = Path("/content/RoadVision/data/custom_city_recovered/images/test")

model = YOLO(MODEL_PATH)

# D10 = class 1
D10 = 1

print("=" * 75)
print("D10 RAW PREDICTION DIAGNOSTIC")
print("=" * 75)

d10_gt_images = []

for img in sorted(TEST_DIR.iterdir()):
    if not img.is_file():
        continue

    label = (
        Path("/content/RoadVision/data/custom_city_recovered")
        / "labels/test"
        / f"{img.stem}.txt"
    )

    if not label.exists():
        continue

    has_d10 = False

    for line in label.read_text(
        encoding="utf-8",
        errors="ignore"
    ).splitlines():

        parts = line.strip().split()

        if len(parts) == 5 and int(parts[0]) == D10:
            has_d10 = True
            break

    if has_d10:
        d10_gt_images.append(img)

print("D10 test images:", len(d10_gt_images))
print()

for img in d10_gt_images:

    print("=" * 75)
    print("IMAGE:", img.name)

    # Ground-truth D10 count
    label = (
        Path("/content/RoadVision/data/custom_city_recovered")
        / "labels/test"
        / f"{img.stem}.txt"
    )

    gt_d10 = 0

    for line in label.read_text(
        encoding="utf-8",
        errors="ignore"
    ).splitlines():

        parts = line.strip().split()

        if len(parts) == 5 and int(parts[0]) == D10:
            gt_d10 += 1

    print("GT D10 boxes:", gt_d10)

    # VERY LOW confidence prediction
    result = model.predict(
        source=str(img),
        conf=0.001,
        iou=0.7,
        verbose=False
    )[0]

    d10_predictions = []

    for box in result.boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])

        if cls == D10:
            xyxy = box.xyxy[0].tolist()

            d10_predictions.append(
                (conf, xyxy)
            )

    print("Raw D10 predictions:", len(d10_predictions))

    if d10_predictions:
        for conf, box in sorted(
            d10_predictions,
            reverse=True
        ):
            print(
                f"  conf={conf:.6f} "
                f"box={[round(x, 2) for x in box]}"
            )
    else:
        print("  NONE")

print("\n" + "=" * 75)
print("END")
print("=" * 75)

In [ ]:
from ultralytics import YOLO
from pathlib import Path

MODEL_PATH = "/content/drive/MyDrive/RoadVision/experiment4_custom_finetune/best.pt"

ROOT = Path(
    "/content/RoadVision/data/custom_city_recovered"
)

TEST_DIR = ROOT / "images/test"
LABEL_DIR = ROOT / "labels/test"

model = YOLO(MODEL_PATH)

D10 = 1


def xywhn_to_xyxy(line, w, h):
    cls, xc, yc, bw, bh = map(float, line.split())

    x1 = (xc - bw / 2) * w
    y1 = (yc - bh / 2) * h
    x2 = (xc + bw / 2) * w
    y2 = (yc + bh / 2) * h

    return [x1, y1, x2, y2]


def iou(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])

    inter_w = max(0, x2 - x1)
    inter_h = max(0, y2 - y1)
    inter = inter_w * inter_h

    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])

    union = area_a + area_b - inter

    return inter / union if union > 0 else 0


print("=" * 80)
print("D10 PREDICTION ↔ GROUND-TRUTH IoU ANALYSIS")
print("=" * 80)

for img in sorted(TEST_DIR.iterdir()):

    if not img.is_file():
        continue

    label_path = LABEL_DIR / f"{img.stem}.txt"

    if not label_path.exists():
        continue

    # --------------------------------------------------------
    # Image dimensions
    # --------------------------------------------------------

    from PIL import Image

    with Image.open(img) as im:
        w, h = im.size

    # --------------------------------------------------------
    # Ground-truth D10 boxes
    # --------------------------------------------------------

    gt_boxes = []

    for line in label_path.read_text(
        encoding="utf-8",
        errors="ignore"
    ).splitlines():

        parts = line.strip().split()

        if len(parts) != 5:
            continue

        if int(parts[0]) != D10:
            continue

        gt_boxes.append(
            xywhn_to_xyxy(line, w, h)
        )

    if not gt_boxes:
        continue

    # --------------------------------------------------------
    # Raw D10 predictions
    # --------------------------------------------------------

    result = model.predict(
        source=str(img),
        conf=0.001,
        iou=0.7,
        verbose=False
    )[0]

    predictions = []

    for box in result.boxes:

        cls = int(box.cls[0])

        if cls != D10:
            continue

        conf = float(box.conf[0])
        xyxy = box.xyxy[0].tolist()

        predictions.append(
            (conf, xyxy)
        )

    print("\n" + "-" * 80)
    print("IMAGE:", img.name)
    print("GT D10:", len(gt_boxes))
    print("Pred D10:", len(predictions))

    # --------------------------------------------------------
    # Each GT -> best prediction
    # --------------------------------------------------------

    for gi, gt in enumerate(gt_boxes, 1):

        matches = []

        for pi, (conf, pred) in enumerate(
            predictions, 1
        ):
            matches.append(
                (
                    iou(gt, pred),
                    conf,
                    pi
                )
            )

        matches.sort(reverse=True)

        if matches:
            best_iou, best_conf, pred_no = matches[0]

            print(
                f"GT #{gi}: "
                f"best IoU={best_iou:.4f}, "
                f"pred_conf={best_conf:.6f}, "
                f"pred=#{pred_no}"
            )
        else:
            print(
                f"GT #{gi}: NO D10 PREDICTION"
            )

print("\n" + "=" * 80)
print("END")
print("=" * 80)

In [ ]:
from pathlib import Path
from collections import Counter
import numpy as np

ROOT = Path("/content/RoadVision/data/custom_city_recovered")
LABEL_DIR = ROOT / "labels/train"

D10 = 1

widths = []
heights = []
areas = []
aspect_ratios = []

print("=" * 80)
print("D10 TRAINING ANNOTATION GEOMETRY")
print("=" * 80)

for label_file in sorted(LABEL_DIR.glob("*.txt")):

    for line in label_file.read_text(
        encoding="utf-8",
        errors="ignore"
    ).splitlines():

        p = line.strip().split()

        if len(p) != 5:
            continue

        cls, xc, yc, w, h = map(float, p)

        if int(cls) != D10:
            continue

        widths.append(w)
        heights.append(h)
        areas.append(w * h)
        aspect_ratios.append(w / h if h > 0 else 0)

print("D10 boxes:", len(widths))

print("\nWidth normalized:")
print(" min =", min(widths))
print(" mean =", np.mean(widths))
print(" median =", np.median(widths))
print(" max =", max(widths))

print("\nHeight normalized:")
print(" min =", min(heights))
print(" mean =", np.mean(heights))
print(" median =", np.median(heights))
print(" max =", max(heights))

print("\nArea normalized:")
print(" min =", min(areas))
print(" mean =", np.mean(areas))
print(" median =", np.median(areas))
print(" max =", max(areas))

print("\nAspect ratio:")
print(" min =", min(aspect_ratios))
print(" mean =", np.mean(aspect_ratios))
print(" median =", np.median(aspect_ratios))
print(" max =", max(aspect_ratios))

print("\n" + "=" * 80)
print("SIZE BUCKETS")
print("=" * 80)

for threshold in [0.001, 0.0025, 0.005, 0.01, 0.02, 0.05]:

    count = sum(a < threshold for a in areas)

    print(
        f"area < {threshold:.4f}: "
        f"{count}/{len(areas)} "
        f"({100*count/len(areas):.1f}%)"
    )

print("\n" + "=" * 80)

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import math

ROOT = Path("/content/RoadVision/data/custom_city_recovered")
IMG_DIR = ROOT / "images/train"
LBL_DIR = ROOT / "labels/train"

D10 = 1

items = []

for img_path in sorted(IMG_DIR.iterdir()):

    if not img_path.is_file():
        continue

    label_path = LBL_DIR / f"{img_path.stem}.txt"

    if not label_path.exists():
        continue

    lines = label_path.read_text(
        encoding="utf-8",
        errors="ignore"
    ).splitlines()

    has_d10 = any(
        len(p := line.strip().split()) == 5
        and int(p[0]) == D10
        for line in lines
    )

    if has_d10:
        items.append(img_path)

print("D10 training images:", len(items))
print([x.name for x in items])

In [ ]:
from ultralytics import YOLO
from pathlib import Path

MODEL_PATH = "/content/drive/MyDrive/RoadVision/experiment4_custom_finetune/best.pt"

ROOT = Path("/content/RoadVision/data/custom_city_recovered")
IMG_DIR = ROOT / "images/train"
LBL_DIR = ROOT / "labels/train"

model = YOLO(MODEL_PATH)

D10 = 1

print("=" * 80)
print("D10 TRAINING-IMAGE RAW PREDICTION TEST")
print("=" * 80)

total_gt = 0
total_pred = 0

for img_path in sorted(IMG_DIR.iterdir()):

    if not img_path.is_file():
        continue

    label_path = LBL_DIR / f"{img_path.stem}.txt"

    if not label_path.exists():
        continue

    gt_count = 0

    for line in label_path.read_text(
        encoding="utf-8",
        errors="ignore"
    ).splitlines():

        p = line.strip().split()

        if len(p) == 5 and int(p[0]) == D10:
            gt_count += 1

    if gt_count == 0:
        continue

    result = model.predict(
        source=str(img_path),
        conf=0.001,
        iou=0.7,
        verbose=False
    )[0]

    preds = []

    for box in result.boxes:

        cls = int(box.cls[0])

        if cls == D10:
            preds.append(float(box.conf[0]))

    total_gt += gt_count
    total_pred += len(preds)

    print(
        f"{img_path.name:25s} "
        f"GT={gt_count:2d}  "
        f"Pred={len(preds):2d}  "
        f"MaxConf={max(preds):.6f}" if preds
        else
        f"{img_path.name:25s} "
        f"GT={gt_count:2d}  "
        f"Pred= 0  "
        f"MaxConf=NONE"
    )

print("\n" + "=" * 80)
print("TOTAL GT D10:", total_gt)
print("TOTAL RAW D10 PREDICTIONS:", total_pred)
print("=" * 80)


In [ ]:
from ultralytics import YOLO
from pathlib import Path

MODEL_PATH = "/content/drive/MyDrive/RoadVision/experiment4_custom_finetune/best.pt"

ROOT = Path("/content/RoadVision/data/custom_city_recovered")
TEST_DIR = ROOT / "images/test"
LABEL_DIR = ROOT / "labels/test"

model = YOLO(MODEL_PATH)

for TARGET_CLASS, CLASS_NAME in [(2, "D20"), (3, "D40")]:

    print("\n" + "=" * 80)
    print(f"{CLASS_NAME} RAW PREDICTION DIAGNOSTIC")
    print("=" * 80)

    for img in sorted(TEST_DIR.iterdir()):

        if not img.is_file():
            continue

        label = LABEL_DIR / f"{img.stem}.txt"

        if not label.exists():
            continue

        gt_count = 0

        for line in label.read_text(
            encoding="utf-8",
            errors="ignore"
        ).splitlines():

            p = line.strip().split()

            if len(p) == 5 and int(p[0]) == TARGET_CLASS:
                gt_count += 1

        if gt_count == 0:
            continue

        result = model.predict(
            source=str(img),
            conf=0.001,
            iou=0.7,
            verbose=False
        )[0]

        preds = []

        for box in result.boxes:

            cls = int(box.cls[0])

            if cls == TARGET_CLASS:
                preds.append(
                    (
                        float(box.conf[0]),
                        box.xyxy[0].tolist()
                    )
                )

        print(f"\nIMAGE: {img.name}")
        print(f"GT {CLASS_NAME}: {gt_count}")
        print(f"Raw {CLASS_NAME} predictions: {len(preds)}")

        for conf, box in preds:
            print(
                f"  conf={conf:.6f} "
                f"box={[round(x,2) for x in box]}"
            )

print("\n" + "=" * 80)
print("END")
print("=" * 80)

In [ ]:
from ultralytics import YOLO
from pathlib import Path
from PIL import Image
import numpy as np

# ============================================================
# FINAL D20 / D40 DIAGNOSTIC
# ============================================================

MODEL_PATH = "/content/drive/MyDrive/RoadVision/experiment4_custom_finetune/best.pt"

ROOT = Path("/content/RoadVision/data/custom_city_recovered")
TEST_DIR = ROOT / "images/test"
LABEL_DIR = ROOT / "labels/test"

model = YOLO(MODEL_PATH)

CLASSES = {
    2: "D20",
    3: "D40",
}

CONF_LEVELS = [0.001, 0.01, 0.05, 0.10]
IOU_MATCH = 0.50


def xywhn_to_xyxy(parts, w, h):
    _, xc, yc, bw, bh = map(float, parts)

    return [
        (xc - bw / 2) * w,
        (yc - bh / 2) * h,
        (xc + bw / 2) * w,
        (yc + bh / 2) * h,
    ]


def box_iou(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])

    iw = max(0, x2 - x1)
    ih = max(0, y2 - y1)

    inter = iw * ih

    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])

    union = area_a + area_b - inter

    return inter / union if union > 0 else 0


# ------------------------------------------------------------
# Storage
# ------------------------------------------------------------

stats = {
    cls_id: {
        "images": 0,
        "gt": 0,
        "best_ious": [],
        "max_conf": [],
        "pred_counts": {c: 0 for c in CONF_LEVELS},
        "matched": {c: 0 for c in CONF_LEVELS},
    }
    for cls_id in CLASSES
}


# ============================================================
# PROCESS EVERY TEST IMAGE
# ============================================================

for img_path in sorted(TEST_DIR.iterdir()):

    if not img_path.is_file():
        continue

    label_path = LABEL_DIR / f"{img_path.stem}.txt"

    if not label_path.exists():
        continue

    with Image.open(img_path) as im:
        W, H = im.size

    # --------------------------------------------------------
    # Ground truth
    # --------------------------------------------------------

    gt = {
        cls_id: []
        for cls_id in CLASSES
    }

    for line in label_path.read_text(
        encoding="utf-8",
        errors="ignore"
    ).splitlines():

        parts = line.strip().split()

        if len(parts) != 5:
            continue

        cls_id = int(parts[0])

        if cls_id not in CLASSES:
            continue

        gt[cls_id].append(
            xywhn_to_xyxy(parts, W, H)
        )

    # --------------------------------------------------------
    # Only images containing D20/D40
    # --------------------------------------------------------

    if not any(gt[c] for c in CLASSES):
        continue

    # One inference per image
    result = model.predict(
        source=str(img_path),
        conf=0.001,
        iou=0.7,
        verbose=False
    )[0]

    predictions = {
        cls_id: []
        for cls_id in CLASSES
    }

    for box in result.boxes:

        cls_id = int(box.cls[0])

        if cls_id not in CLASSES:
            continue

        conf = float(box.conf[0])
        xyxy = box.xyxy[0].tolist()

        predictions[cls_id].append(
            (conf, xyxy)
        )

    # --------------------------------------------------------
    # Analyse each class
    # --------------------------------------------------------

    for cls_id in CLASSES:

        gt_boxes = gt[cls_id]

        if not gt_boxes:
            continue

        stats[cls_id]["images"] += 1
        stats[cls_id]["gt"] += len(gt_boxes)

        preds = predictions[cls_id]

        max_conf = (
            max((p[0] for p in preds), default=0)
        )

        stats[cls_id]["max_conf"].append(max_conf)

        # Best IoU for every GT
        for gt_box in gt_boxes:

            best = 0

            for conf, pred_box in preds:
                best = max(
                    best,
                    box_iou(gt_box, pred_box)
                )

            stats[cls_id]["best_ious"].append(best)

        # Threshold analysis
        for conf_threshold in CONF_LEVELS:

            filtered = [
                box
                for conf, box in preds
                if conf >= conf_threshold
            ]

            stats[cls_id]["pred_counts"][conf_threshold] += len(filtered)

            # Count GTs having >= 0.5 IoU
            for gt_box in gt_boxes:

                best = max(
                    (
                        box_iou(gt_box, pred_box)
                        for pred_box in filtered
                    ),
                    default=0
                )

                if best >= IOU_MATCH:
                    stats[cls_id]["matched"][conf_threshold] += 1


# ============================================================
# FINAL REPORT
# ============================================================

print("\n")
print("=" * 95)
print("FINAL D20 / D40 DIAGNOSTIC")
print("=" * 95)

for cls_id, name in CLASSES.items():

    s = stats[cls_id]

    best_ious = np.array(s["best_ious"])
    max_confs = np.array(s["max_conf"])

    print(f"\n{name}")
    print("-" * 95)

    print(f"GT images              : {s['images']}")
    print(f"GT boxes               : {s['gt']}")

    print(
        f"Best IoU mean          : "
        f"{best_ious.mean():.4f}"
    )

    print(
        f"Best IoU median        : "
        f"{np.median(best_ious):.4f}"
    )

    print(
        f"Best IoU >= 0.50       : "
        f"{np.sum(best_ious >= 0.50)}/{s['gt']} "
        f"({100*np.mean(best_ious >= 0.50):.1f}%)"
    )

    print(
        f"Best IoU >= 0.25       : "
        f"{np.sum(best_ious >= 0.25)}/{s['gt']} "
        f"({100*np.mean(best_ious >= 0.25):.1f}%)"
    )

    print(
        f"Maximum raw confidence : "
        f"{max_confs.max():.6f}"
    )

    print(
        f"Median max confidence  : "
        f"{np.median(max_confs):.6f}"
    )

    print("\nThreshold analysis:")

    for c in CONF_LEVELS:

        n_pred = s["pred_counts"][c]
        n_match = s["matched"][c]

        print(
            f"  conf >= {c:<5} | "
            f"predictions={n_pred:<5} | "
            f"GT matched IoU>=0.50="
            f"{n_match}/{s['gt']}"
        )


print("\n" + "=" * 95)
print("INTERPRETATION")
print("=" * 95)

for cls_id, name in CLASSES.items():

    s = stats[cls_id]

    mean_iou = np.mean(s["best_ious"])
    recall50 = (
        s["matched"][0.01] / s["gt"]
        if s["gt"] else 0
    )

    max_conf = max(s["max_conf"])

    print(f"\n{name}:")

    if recall50 >= 0.5 and max_conf >= 0.1:
        print(
            "  -> Model is producing reasonably localized "
            "detections. Focus on threshold/evaluation."
        )

    elif mean_iou >= 0.25:
        print(
            "  -> Some localization signal exists, "
            "but detection quality is weak."
        )

    elif max_conf >= 0.1:
        print(
            "  -> Model is confident but localization "
            "is poor. Investigate box regression."
        )

    else:
        print(
            "  -> Weak class signal / learning problem. "
            "Investigate training/data before tuning threshold."
        )

print("\n" + "=" * 95)
print("DONE")
print("=" * 95)

In [ ]:
from ultralytics import YOLO

MODEL_PATH = "/content/drive/MyDrive/RoadVision/experiment4_custom_finetune/best.pt"
DATA_YAML = "/content/RoadVision/data/custom_city_recovered/data.yaml"

model = YOLO(MODEL_PATH)

print("=" * 80)
print("CONFIDENCE THRESHOLD SWEEP")
print("=" * 80)

for conf in [0.001, 0.005, 0.01, 0.02, 0.03, 0.05, 0.10]:

    print(f"\n{'='*25} CONF = {conf} {'='*25}")

    metrics = model.val(
        data=DATA_YAML,
        split="test",
        imgsz=640,
        conf=conf,
        iou=0.7,
        plots=False,
        verbose=False
    )

    print(
        f"Precision : {metrics.box.mp:.4f}\n"
        f"Recall    : {metrics.box.mr:.4f}\n"
        f"mAP50    : {metrics.box.map50:.4f}\n"
        f"mAP50-95 : {metrics.box.map:.4f}"
    )

    for cls_id, name in enumerate(["D00", "D10", "D20", "D40"]):
        print(
            f"{name}: "
            f"P={metrics.box.p[cls_id]:.4f}, "
            f"R={metrics.box.r[cls_id]:.4f}, "
            f"mAP50={metrics.box.ap50[cls_id]:.4f}"
        )


In [ ]:
from pathlib import Path
import yaml

RUN = Path("/content/drive/MyDrive/RoadVision/experiment4_custom_finetune")

print("=" * 80)
print("EXPERIMENT 4 TRAINING ARTIFACTS")
print("=" * 80)

for p in sorted(RUN.rglob("*")):
    if p.is_file():
        print(p)

print("\n" + "=" * 80)
print("ARGS / CONFIG")
print("=" * 80)

for name in ["args.yaml", "args.yml"]:
    p = RUN / name
    if p.exists():
        print(f"\n--- {p.name} ---")
        print(p.read_text(errors="ignore"))

print("\n" + "=" * 80)
print("RESULTS CSV HEADER")
print("=" * 80)

csv = RUN / "results.csv"
if csv.exists():
    with open(csv, "r") as f:
        print(f.readline().strip())

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# ============================================================
# EXPERIMENT 5 — CONTROLLED CUSTOM-CITY FINE-TUNING
# ============================================================

DATA_YAML = "/content/RoadVision/data/custom_city_recovered/data.yaml"

# Start from the pretrained YOLO11n model,
# NOT Experiment 4's already-undertrained checkpoint.
MODEL = "yolo11n.pt"

PROJECT = "/content/drive/MyDrive/RoadVision"
NAME = "experiment5_custom_finetune"

model = YOLO(MODEL)

results = model.train(
    data=DATA_YAML,

    # -------------------------
    # Training
    # -------------------------
    epochs=50,
    imgsz=640,
    batch=16,

    # -------------------------
    # Optimizer / LR
    # -------------------------
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,

    # -------------------------
    # Augmentation
    # -------------------------
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,

    degrees=5.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,

    fliplr=0.5,
    flipud=0.0,

    mosaic=1.0,
    mixup=0.0,
    close_mosaic=10,

    # -------------------------
    # Validation / saving
    # -------------------------
    val=True,
    plots=True,
    save=True,
    save_period=10,

    # -------------------------
    # Reproducibility
    # -------------------------
    seed=42,
    deterministic=True,

    # -------------------------
    # Output
    # -------------------------
    project=PROJECT,
    name=NAME,
    exist_ok=True,

    device=0,
    workers=2,
    verbose=True,
)

print("\n" + "=" * 80)
print("EXPERIMENT 5 TRAINING COMPLETE")
print("=" * 80)

run_dir = Path(PROJECT) / NAME

print("Run directory:", run_dir)
print("Best checkpoint:", run_dir / "weights/best.pt")
print("Last checkpoint:", run_dir / "weights/last.pt")

In [ ]:
from ultralytics import YOLO

BEST = "/content/drive/MyDrive/RoadVision/experiment5_custom_finetune/weights/best.pt"
DATA = "/content/RoadVision/data/custom_city_recovered/data.yaml"

model = YOLO(BEST)

m = model.val(
    data=DATA,
    split="test",
    imgsz=640,
    conf=0.001,
    iou=0.7,
    plots=True,
    verbose=True
)

print("\n" + "=" * 70)
print("EXPERIMENT 5 TEST RESULTS")
print("=" * 70)

print(f"Precision : {m.box.mp:.4f}")
print(f"Recall    : {m.box.mr:.4f}")
print(f"mAP50     : {m.box.map50:.4f}")
print(f"mAP50-95  : {m.box.map:.4f}")

for i, name in enumerate(["D00", "D10", "D20", "D40"]):
    print(
        f"{name}: "
        f"P={m.box.p[i]:.4f}, "
        f"R={m.box.r[i]:.4f}, "
        f"mAP50={m.box.ap50[i]:.4f}, "
        f"mAP50-95={m.box.ap[i]:.4f}"
    )

In [ ]:
from ultralytics import YOLO
from pathlib import Path

DATA = "/content/RoadVision/data/custom_city_recovered/data.yaml"

PREV = "/content/drive/MyDrive/RoadVision/experiment5_custom_finetune/weights/best.pt"

PROJECT = "/content/drive/MyDrive/RoadVision"
NAME = "experiment6_custom_finetune"

model = YOLO(PREV)

model.train(
    data=DATA,

    # Continue training
    epochs=100,
    imgsz=640,
    batch=16,

    optimizer="AdamW",
    lr0=0.0005,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=2,

    # Augmentation
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,

    degrees=5.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,

    fliplr=0.5,
    flipud=0.0,

    mosaic=1.0,
    mixup=0.0,
    close_mosaic=15,

    val=True,
    plots=True,
    save=True,
    save_period=10,

    seed=42,
    deterministic=True,

    project=PROJECT,
    name=NAME,
    exist_ok=True,

    device=0,
    workers=2,
    verbose=True,
)

print("\n" + "=" * 70)
print("EXPERIMENT 6 COMPLETE")
print("=" * 70)
print(
    "Best:",
    Path(PROJECT) / NAME / "weights/best.pt"
)

In [ ]:
from ultralytics import YOLO

BEST = "/content/drive/MyDrive/RoadVision/experiment6_custom_finetune/weights/best.pt"
DATA = "/content/RoadVision/data/custom_city_recovered/data.yaml"

model = YOLO(BEST)

m = model.val(
    data=DATA,
    split="test",
    imgsz=640,
    conf=0.001,
    iou=0.7,
    plots=True,
    verbose=True
)

print("\n" + "=" * 70)
print("EXPERIMENT 6 — FINAL TEST")
print("=" * 70)

print(f"Precision : {m.box.mp:.4f}")
print(f"Recall    : {m.box.mr:.4f}")
print(f"mAP50     : {m.box.map50:.4f}")
print(f"mAP50-95  : {m.box.map:.4f}")

for i, name in enumerate(["D00", "D10", "D20", "D40"]):
    print(
        f"{name}: "
        f"P={m.box.p[i]:.4f}, "
        f"R={m.box.r[i]:.4f}, "
        f"mAP50={m.box.ap50[i]:.4f}, "
        f"mAP50-95={m.box.ap[i]:.4f}"
    )

In [ ]:
from ultralytics import YOLO
from pathlib import Path

DATA = "/content/RoadVision/data/custom_city_recovered/data.yaml"

# Exp 5 was better on the untouched test set.
MODEL = "/content/drive/MyDrive/RoadVision/experiment5_custom_finetune/weights/best.pt"

PROJECT = "/content/drive/MyDrive/RoadVision"
NAME = "experiment7_custom_finetune"

model = YOLO(MODEL)

model.train(
    data=DATA,

    epochs=80,
    imgsz=640,
    batch=16,

    optimizer="AdamW",
    lr0=0.00025,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,

    # Less aggressive augmentation
    hsv_h=0.01,
    hsv_s=0.35,
    hsv_v=0.20,

    degrees=2.0,
    translate=0.05,
    scale=0.30,
    shear=0.0,
    perspective=0.0,

    fliplr=0.5,
    flipud=0.0,

    # Reduce synthetic distortion
    mosaic=0.5,
    mixup=0.0,
    close_mosaic=15,

    val=True,
    plots=True,
    save=True,
    save_period=10,

    seed=42,
    deterministic=True,

    project=PROJECT,
    name=NAME,
    exist_ok=True,

    device=0,
    workers=2,
    verbose=True,
)

In [ ]:
from ultralytics import YOLO

BEST = "/content/drive/MyDrive/RoadVision/experiment7_custom_finetune/weights/best.pt"
DATA = "/content/RoadVision/data/custom_city_recovered/data.yaml"

model = YOLO(BEST)

m = model.val(
    data=DATA,
    split="test",
    imgsz=640,
    conf=0.001,
    iou=0.7,
    plots=True,
    verbose=True
)

print("\n" + "=" * 70)
print("EXPERIMENT 7 — FINAL TEST")
print("=" * 70)

print(f"Precision : {m.box.mp:.4f}")
print(f"Recall    : {m.box.mr:.4f}")
print(f"mAP50     : {m.box.map50:.4f}")
print(f"mAP50-95  : {m.box.map:.4f}")

for i, name in enumerate(["D00", "D10", "D20", "D40"]):
    print(
        f"{name}: "
        f"P={m.box.p[i]:.4f}, "
        f"R={m.box.r[i]:.4f}, "
        f"mAP50={m.box.ap50[i]:.4f}, "
        f"mAP50-95={m.box.ap[i]:.4f}"
    )

In [ ]:
from pathlib import Path
from collections import Counter

LABEL_DIR = Path("/content/RoadVision/data/custom_city_recovered/labels/train")

classes = {
    0: "D00",
    1: "D10",
    2: "D20",
    3: "D40"
}

image_stats = []

for txt in sorted(LABEL_DIR.glob("*.txt")):
    counts = Counter()

    for line in txt.read_text().splitlines():
        if line.strip():
            cls = int(line.split()[0])
            counts[classes[cls]] += 1

    image_stats.append((txt.stem, counts))

d20_images = [
    (stem, counts["D20"])
    for stem, counts in image_stats
    if counts["D20"] > 0
]

print("=" * 60)
print("TRAINING D20 DISTRIBUTION")
print("=" * 60)

print("D20 training images :", len(d20_images))
print("D20 training boxes  :", sum(x[1] for x in d20_images))

print("\nD20 boxes per image:")
for stem, n in sorted(d20_images, key=lambda x: x[1], reverse=True):
    print(f"{stem:20s} {n}")

print("\nAll class box totals:")
total = Counter()

for _, counts in image_stats:
    total.update(counts)

for cls in ["D00", "D10", "D20", "D40"]:
    print(f"{cls}: {total[cls]}")

In [ ]:
from pathlib import Path
import shutil
from ultralytics import YOLO

# ============================================================
# PATHS
# ============================================================

SRC = Path("/content/RoadVision/data/custom_city_recovered")

WORK = Path("/content/RoadVision/data/custom_city_d20_oversampled")

MODEL = "/content/drive/MyDrive/RoadVision/experiment7_custom_finetune/weights/best.pt"

PROJECT = "/content/drive/MyDrive/RoadVision"
NAME = "experiment8_d20_oversampled"


# ============================================================
# CLEAN + COPY DATASET
# ============================================================

if WORK.exists():
    shutil.rmtree(WORK)

shutil.copytree(SRC, WORK)

print("Dataset copied to:")
print(WORK)


# ============================================================
# CREATE TEMPORARY EXTRA D20 TRAINING SAMPLES
# ============================================================

train_img_dir = WORK / "images" / "train"
train_lbl_dir = WORK / "labels" / "train"

extra_img_dir = WORK / "images" / "train_extra_d20"
extra_lbl_dir = WORK / "labels" / "train_extra_d20"

extra_img_dir.mkdir(parents=True)
extra_lbl_dir.mkdir(parents=True)

D20_CLASS = "2"

d20_files = []

for label_file in sorted(train_lbl_dir.glob("*.txt")):

    text = label_file.read_text().splitlines()

    has_d20 = any(
        line.strip() and line.split()[0] == D20_CLASS
        for line in text
    )

    if has_d20:
        d20_files.append(label_file)


print("\nD20 source images:", len(d20_files))


# ============================================================
# ~1.5x OVERSAMPLING
#
# Original D20 images = 34
# Extra copies = 17
# Effective exposure ≈ 1.5x
# ============================================================

extra_count = len(d20_files) // 2

for label_file in d20_files[:extra_count]:

    stem = label_file.stem

    # Find corresponding image
    image_file = None

    for ext in [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]:
        candidate = train_img_dir / f"{stem}{ext}"
        if candidate.exists():
            image_file = candidate
            break

    if image_file is None:
        print("WARNING: image not found:", stem)
        continue

    new_stem = f"{stem}_d20extra"

    shutil.copy2(
        image_file,
        extra_img_dir / f"{new_stem}{image_file.suffix}"
    )

    shutil.copy2(
        label_file,
        extra_lbl_dir / f"{new_stem}.txt"
    )


print("Extra D20 images:", extra_count)


# ============================================================
# CREATE TRAIN LIST
# ============================================================

train_list = WORK / "train_d20_oversampled.txt"

all_images = []

for img_dir in [
    WORK / "images" / "train",
    WORK / "images" / "train_extra_d20"
]:

    for img in sorted(img_dir.iterdir()):

        if img.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            all_images.append(str(img))


train_list.write_text("\n".join(all_images))

print("\nTotal training images:", len(all_images))


# ============================================================
# DATA YAML
# ============================================================

yaml_path = WORK / "data_d20_oversampled.yaml"

yaml_path.write_text(
f"""path: {WORK}
train: {train_list}
val: {SRC / 'images' / 'val'}
test: {SRC / 'images' / 'test'}

names:
  0: D00
  1: D10
  2: D20
  3: D40
"""
)

print("YAML:", yaml_path)


# ============================================================
# TRAIN
# ============================================================

model = YOLO(MODEL)

results = model.train(

    data=str(yaml_path),

    epochs=60,
    imgsz=640,
    batch=16,

    optimizer="AdamW",

    # Same conservative LR philosophy as Exp7
    lr0=0.0002,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,

    # Exp7-style moderate augmentation
    hsv_h=0.01,
    hsv_s=0.35,
    hsv_v=0.20,

    degrees=2.0,
    translate=0.05,
    scale=0.30,
    shear=0.0,
    perspective=0.0,

    fliplr=0.5,
    flipud=0.0,

    mosaic=0.5,
    mixup=0.0,
    close_mosaic=15,

    val=True,
    plots=True,
    save=True,
    save_period=10,

    seed=42,
    deterministic=True,

    project=PROJECT,
    name=NAME,
    exist_ok=True,

    device=0,
    workers=2,
    verbose=True,
)

In [ ]:
from ultralytics import YOLO

BEST = "/content/drive/MyDrive/RoadVision/experiment8_d20_oversampled/weights/best.pt"

DATA = "/content/RoadVision/data/custom_city_recovered/data.yaml"

model = YOLO(BEST)

m = model.val(
    data=DATA,
    split="test",
    imgsz=640,
    conf=0.001,
    iou=0.7,
    plots=True,
    verbose=True
)

print("\n" + "=" * 70)
print("EXPERIMENT 8 — FINAL TEST")
print("=" * 70)

print(f"Precision : {m.box.mp:.4f}")
print(f"Recall    : {m.box.mr:.4f}")
print(f"mAP50     : {m.box.map50:.4f}")
print(f"mAP50-95  : {m.box.map:.4f}")

for i, name in enumerate(["D00", "D10", "D20", "D40"]):
    print(
        f"{name}: "
        f"P={m.box.p[i]:.4f}, "
        f"R={m.box.r[i]:.4f}, "
        f"mAP50={m.box.ap50[i]:.4f}, "
        f"mAP50-95={m.box.ap[i]:.4f}"
    )

In [ ]:
from pathlib import Path
from collections import defaultdict
from PIL import Image
import numpy as np

ROOT = Path("/content/RoadVision/data/custom_city_recovered")

CLASSES = {
    0: "D00",
    1: "D10",
    2: "D20",
    3: "D40",
}

def audit_split(split):
    img_dir = ROOT / "images" / split
    lbl_dir = ROOT / "labels" / split

    rows = []

    for label_file in sorted(lbl_dir.glob("*.txt")):

        lines = [
            x.strip()
            for x in label_file.read_text().splitlines()
            if x.strip()
        ]

        d20 = []

        for line in lines:
            p = line.split()
            if len(p) != 5:
                continue

            cls = int(p[0])

            if cls == 2:
                x, y, w, h = map(float, p[1:])
                d20.append((x, y, w, h))

        if not d20:
            continue

        # Find image
        img_path = None
        for ext in [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]:
            candidate = img_dir / f"{label_file.stem}{ext}"
            if candidate.exists():
                img_path = candidate
                break

        if img_path is None:
            continue

        with Image.open(img_path) as im:
            W, H = im.size

        areas = [w * h for _, _, w, h in d20]
        widths = [w for _, _, w, h in d20]
        heights = [h for _, _, w, h in d20]

        rows.append({
            "stem": label_file.stem,
            "boxes": len(d20),
            "width": W,
            "height": H,
            "mean_box_area": np.mean(areas),
            "min_box_area": np.min(areas),
            "max_box_area": np.max(areas),
            "mean_box_w": np.mean(widths),
            "mean_box_h": np.mean(heights),
            "tiny_boxes": sum(a < 0.005 for a in areas),
            "small_boxes": sum(a < 0.01 for a in areas),
        })

    return rows


for split in ["train", "val", "test"]:

    rows = audit_split(split)

    print("\n" + "=" * 80)
    print(f"D20 {split.upper()} AUDIT")
    print("=" * 80)

    print("Images :", len(rows))
    print("Boxes  :", sum(r["boxes"] for r in rows))

    if not rows:
        continue

    areas = []

    for r in rows:
        areas.extend([r["mean_box_area"]] * r["boxes"])

    print(f"Mean box area   : {np.mean(areas):.5f}")
    print(f"Median box area : {np.median(areas):.5f}")

    print("\nImages sorted by smallest average D20 box:")

    for r in sorted(rows, key=lambda x: x["mean_box_area"])[:10]:
        print(
            f"{r['stem']:<45} "
            f"boxes={r['boxes']:2d}  "
            f"mean_area={r['mean_box_area']:.5f}  "
            f"min={r['min_box_area']:.5f}  "
            f"max={r['max_box_area']:.5f}"
        )

    print("\nImages with highest D20 density:")

    for r in sorted(rows, key=lambda x: x["boxes"], reverse=True)[:10]:
        print(
            f"{r['stem']:<45} "
            f"boxes={r['boxes']:2d}  "
            f"mean_area={r['mean_box_area']:.5f}"
        )

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import math

ROOT = Path("/content/RoadVision/data/custom_city_recovered")

def make_d20_sheet(split, output_name):

    img_dir = ROOT / "images" / split
    lbl_dir = ROOT / "labels" / split

    items = []

    for label_file in sorted(lbl_dir.glob("*.txt")):

        lines = [
            x.strip()
            for x in label_file.read_text().splitlines()
            if x.strip()
        ]

        d20_boxes = []

        for line in lines:
            p = line.split()

            if len(p) == 5 and int(p[0]) == 2:
                x, y, w, h = map(float, p[1:])
                d20_boxes.append((x, y, w, h))

        if not d20_boxes:
            continue

        img_path = None

        for ext in [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]:
            candidate = img_dir / f"{label_file.stem}{ext}"
            if candidate.exists():
                img_path = candidate
                break

        if img_path:
            items.append((img_path, d20_boxes))

    thumb_w = 320
    thumb_h = 220
    cols = 3
    rows = math.ceil(len(items) / cols)

    sheet = Image.new(
        "RGB",
        (cols * thumb_w, rows * thumb_h),
        "white"
    )

    draw = ImageDraw.Draw(sheet)

    for i, (img_path, boxes) in enumerate(items):

        img = Image.open(img_path).convert("RGB")
        W, H = img.size

        scale = min(
            (thumb_w - 10) / W,
            (thumb_h - 30) / H
        )

        nw = int(W * scale)
        nh = int(H * scale)

        img = img.resize((nw, nh))

        x0 = (i % cols) * thumb_w + 5
        y0 = (i // cols) * thumb_h + 25

        sheet.paste(img, (x0, y0))

        for x, y, w, h in boxes:

            x1 = x0 + int((x - w / 2) * scale)
            y1 = y0 + int((y - h / 2) * scale)
            x2 = x0 + int((x + w / 2) * scale)
            y2 = y0 + int((y + h / 2) * scale)

            draw.rectangle(
                [x1, y1, x2, y2],
                outline="red",
                width=2
            )

        draw.text(
            (x0, y0 - 20),
            f"{img_path.stem} | D20={len(boxes)}",
            fill="black"
        )

    out = Path("/content") / output_name
    sheet.save(out)

    print(f"Saved: {out}")
    return out


train_sheet = make_d20_sheet(
    "train",
    "D20_train_contact_sheet.jpg"
)

test_sheet = make_d20_sheet(
    "test",
    "D20_test_contact_sheet.jpg"
)

In [ ]:
from IPython.display import display
from PIL import Image

display(Image.open("/content/D20_train_contact_sheet.jpg"))
display(Image.open("/content/D20_test_contact_sheet.jpg"))

In [ ]:
from google.colab import files

files.download("/content/D20_train_contact_sheet.jpg")
files.download("/content/D20_test_contact_sheet.jpg")

In [ ]:
from IPython.display import display
from PIL import Image

display(Image.open("/content/D20_test_contact_sheet.jpg"))

In [ ]:
from ultralytics import YOLO

# ============================================================
# EXPERIMENT 9
# Higher-resolution D20 fine-tuning
# ============================================================

DATA = "/content/RoadVision/data/custom_city_recovered/data.yaml"

MODEL = (
    "/content/drive/MyDrive/RoadVision/"
    "experiment7_custom_finetune/weights/best.pt"
)

PROJECT = "/content/drive/MyDrive/RoadVision"
NAME = "experiment9_highres_d20"

model = YOLO(MODEL)

results = model.train(
    data=DATA,

    # Higher resolution
    epochs=60,
    imgsz=960,

    # T4 memory-safe
    batch=8,

    optimizer="AdamW",

    # Conservative fine-tuning
    lr0=0.00015,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,

    # Moderate augmentation
    hsv_h=0.01,
    hsv_s=0.35,
    hsv_v=0.20,

    degrees=2.0,
    translate=0.05,
    scale=0.25,
    shear=0.0,
    perspective=0.0,

    fliplr=0.5,
    flipud=0.0,

    # Less aggressive than Exp7
    mosaic=0.3,
    mixup=0.0,
    close_mosaic=15,

    val=True,
    plots=True,
    save=True,
    save_period=10,

    seed=42,
    deterministic=True,

    project=PROJECT,
    name=NAME,
    exist_ok=True,

    device=0,
    workers=2,
    verbose=True,
)

In [ ]:
from ultralytics import YOLO

BEST = (
    "/content/drive/MyDrive/RoadVision/"
    "experiment9_highres_d20/weights/best.pt"
)

DATA = "/content/RoadVision/data/custom_city_recovered/data.yaml"

model = YOLO(BEST)

m = model.val(
    data=DATA,
    split="test",

    imgsz=960,
    conf=0.001,
    iou=0.7,

    plots=True,
    verbose=True
)

print("\n" + "=" * 70)
print("EXPERIMENT 9 — FINAL TEST")
print("=" * 70)

print(f"Precision : {m.box.mp:.4f}")
print(f"Recall    : {m.box.mr:.4f}")
print(f"mAP50     : {m.box.map50:.4f}")
print(f"mAP50-95  : {m.box.map:.4f}")

for i, name in enumerate(["D00", "D10", "D20", "D40"]):
    print(
        f"{name}: "
        f"P={m.box.p[i]:.4f}, "
        f"R={m.box.r[i]:.4f}, "
        f"mAP50={m.box.ap50[i]:.4f}, "
        f"mAP50-95={m.box.ap[i]:.4f}"
    )

In [ ]:
from ultralytics import YOLO

DATA = "/content/RoadVision/data/custom_city_recovered/data.yaml"

MODEL = (
    "/content/drive/MyDrive/RoadVision/"
    "experiment5_custom_finetune/weights/best.pt"
)

PROJECT = "/content/drive/MyDrive/RoadVision"
NAME = "experiment10_exp5_highres"

model = YOLO(MODEL)

results = model.train(
    data=DATA,

    epochs=60,
    imgsz=960,
    batch=8,

    optimizer="AdamW",

    # Conservative fine-tuning
    lr0=0.00015,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,

    # Moderate augmentation
    hsv_h=0.01,
    hsv_s=0.35,
    hsv_v=0.20,

    degrees=2.0,
    translate=0.05,
    scale=0.25,
    shear=0.0,
    perspective=0.0,

    fliplr=0.5,
    flipud=0.0,

    mosaic=0.3,
    mixup=0.0,
    close_mosaic=15,

    val=True,
    plots=True,
    save=True,
    save_period=10,

    seed=42,
    deterministic=True,

    project=PROJECT,
    name=NAME,
    exist_ok=True,

    device=0,
    workers=2,
    verbose=True,
)

In [ ]:
from ultralytics import YOLO

BEST = (
    "/content/drive/MyDrive/RoadVision/"
    "experiment10_exp5_highres/weights/best.pt"
)

DATA = "/content/RoadVision/data/custom_city_recovered/data.yaml"

model = YOLO(BEST)

m = model.val(
    data=DATA,
    split="test",
    imgsz=960,
    conf=0.001,
    iou=0.7,
    plots=True,
    verbose=True
)

print("\n" + "=" * 70)
print("EXPERIMENT 10 — FINAL TEST")
print("=" * 70)

print(f"Precision : {m.box.mp:.4f}")
print(f"Recall    : {m.box.mr:.4f}")
print(f"mAP50     : {m.box.map50:.4f}")
print(f"mAP50-95  : {m.box.map:.4f}")

for i, name in enumerate(["D00", "D10", "D20", "D40"]):
    print(
        f"{name}: "
        f"P={m.box.p[i]:.4f}, "
        f"R={m.box.r[i]:.4f}, "
        f"mAP50={m.box.ap50[i]:.4f}, "
        f"mAP50-95={m.box.ap[i]:.4f}"
    )

In [ ]:
from ultralytics import YOLO

DATA = "/content/RoadVision/data/custom_city_recovered/data.yaml"

MODEL = "yolo11s.pt"

PROJECT = "/content/drive/MyDrive/RoadVision"
NAME = "experiment11_yolo11s"

model = YOLO(MODEL)

results = model.train(
    data=DATA,

    epochs=80,
    imgsz=640,
    batch=8,

    optimizer="AdamW",

    lr0=0.0005,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,

    # Same moderate augmentation philosophy as Exp7
    hsv_h=0.01,
    hsv_s=0.35,
    hsv_v=0.20,

    degrees=2.0,
    translate=0.05,
    scale=0.30,
    shear=0.0,
    perspective=0.0,

    fliplr=0.5,
    flipud=0.0,

    mosaic=0.5,
    mixup=0.0,
    close_mosaic=15,

    val=True,
    plots=True,
    save=True,
    save_period=10,

    seed=42,
    deterministic=True,

    project=PROJECT,
    name=NAME,
    exist_ok=True,

    device=0,
    workers=2,
    verbose=True,
)

In [ ]:
from ultralytics import YOLO

BEST = (
    "/content/drive/MyDrive/RoadVision/"
    "experiment11_yolo11s/weights/best.pt"
)

DATA = "/content/RoadVision/data/custom_city_recovered/data.yaml"

model = YOLO(BEST)

m = model.val(
    data=DATA,
    split="test",
    imgsz=640,
    conf=0.001,
    iou=0.7,
    plots=True,
    verbose=True
)

print("\n" + "=" * 70)
print("EXPERIMENT 11 — YOLO11s FINAL TEST")
print("=" * 70)

print(f"Precision : {m.box.mp:.4f}")
print(f"Recall    : {m.box.mr:.4f}")
print(f"mAP50     : {m.box.map50:.4f}")
print(f"mAP50-95  : {m.box.map:.4f}")

for i, name in enumerate(["D00", "D10", "D20", "D40"]):
    print(
        f"{name}: "
        f"P={m.box.p[i]:.4f}, "
        f"R={m.box.r[i]:.4f}, "
        f"mAP50={m.box.ap50[i]:.4f}, "
        f"mAP50-95={m.box.ap[i]:.4f}"
    )

In [ ]:
from ultralytics import YOLO
import numpy as np

MODEL = (
    "/content/drive/MyDrive/RoadVision/"
    "experiment7_custom_finetune/weights/best.pt"
)

DATA = "/content/RoadVision/data/custom_city_recovered/data.yaml"

model = YOLO(MODEL)

m = model.val(
    data=DATA,
    split="test",
    imgsz=640,
    conf=0.001,
    iou=0.7,
    plots=True,
    verbose=True
)

print("\n" + "=" * 70)
print("CONFUSION MATRIX — EXP 7")
print("=" * 70)

cm = m.confusion_matrix

print("\nRaw matrix:")
print(cm.matrix)

print("\nNormalized matrix:")
print(np.round(cm.matrix / np.maximum(cm.matrix.sum(axis=1, keepdims=True), 1), 3))

print("\nClass names:")
print(model.names)

In [ ]:
from pathlib import Path

print("\n" + "=" * 70)
print("EXP 7 TEST OUTPUT")
print("=" * 70)

print("Confusion matrix image should be saved in:")
print("/content/runs/detect/")

In [ ]:
from ultralytics import YOLO

DATA = "/content/RoadVision/data/custom_city_recovered/data.yaml"

MODEL = (
    "/content/drive/MyDrive/RoadVision/"
    "experiment7_custom_finetune/weights/best.pt"
)

PROJECT = "/content/drive/MyDrive/RoadVision"
NAME = "experiment12_cls_focus"

model = YOLO(MODEL)

results = model.train(
    data=DATA,

    epochs=60,
    imgsz=640,
    batch=16,

    optimizer="AdamW",

    lr0=0.00015,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,

    # ------------------------------------------------
    # IMPORTANT: stronger classification emphasis
    # ------------------------------------------------
    cls=1.5,

    # Normal box / DFL losses
    box=7.5,
    dfl=1.5,

    # Moderate augmentation
    hsv_h=0.01,
    hsv_s=0.35,
    hsv_v=0.20,

    degrees=2.0,
    translate=0.05,
    scale=0.30,
    shear=0.0,
    perspective=0.0,

    fliplr=0.5,
    flipud=0.0,

    mosaic=0.5,
    mixup=0.0,
    close_mosaic=15,

    val=True,
    plots=True,
    save=True,
    save_period=10,

    seed=42,
    deterministic=True,

    project=PROJECT,
    name=NAME,
    exist_ok=True,

    device=0,
    workers=2,
    verbose=True,
)

In [ ]:
from ultralytics import YOLO

BEST = (
    "/content/drive/MyDrive/RoadVision/"
    "experiment12_cls_focus/weights/best.pt"
)

DATA = "/content/RoadVision/data/custom_city_recovered/data.yaml"

model = YOLO(BEST)

m = model.val(
    data=DATA,
    split="test",
    imgsz=640,
    conf=0.001,
    iou=0.7,
    plots=True,
    verbose=True
)

print("\n" + "=" * 70)
print("EXPERIMENT 12 — CLASSIFICATION FOCUS")
print("=" * 70)

print(f"Precision : {m.box.mp:.4f}")
print(f"Recall    : {m.box.mr:.4f}")
print(f"mAP50     : {m.box.map50:.4f}")
print(f"mAP50-95  : {m.box.map:.4f}")

for i, name in enumerate(["D00", "D10", "D20", "D40"]):
    print(
        f"{name}: "
        f"P={m.box.p[i]:.4f}, "
        f"R={m.box.r[i]:.4f}, "
        f"mAP50={m.box.ap50[i]:.4f}, "
        f"mAP50-95={m.box.ap[i]:.4f}"
    )

In [ ]:
from ultralytics import YOLO

DATA = "/content/RoadVision/data/custom_city_recovered/data.yaml"

MODEL = (
    "/content/drive/MyDrive/RoadVision/"
    "experiment7_custom_finetune/weights/best.pt"
)

PROJECT = "/content/drive/MyDrive/RoadVision"
NAME = "experiment13_d20_d40_balance"

model = YOLO(MODEL)

model.train(
    data=DATA,

    epochs=50,
    imgsz=640,
    batch=16,

    optimizer="AdamW",

    lr0=0.0001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,

    # Mild classification emphasis
    cls=1.15,
    box=7.5,
    dfl=1.5,

    # Keep Exp7-style augmentation
    hsv_h=0.01,
    hsv_s=0.35,
    hsv_v=0.20,

    degrees=2.0,
    translate=0.05,
    scale=0.30,
    shear=0.0,
    perspective=0.0,

    fliplr=0.5,
    flipud=0.0,

    mosaic=0.5,
    mixup=0.0,
    close_mosaic=15,

    val=True,
    plots=True,
    save=True,
    save_period=10,

    seed=42,
    deterministic=True,

    project=PROJECT,
    name=NAME,
    exist_ok=True,

    device=0,
    workers=2,
    verbose=True,
)

In [ ]:
from ultralytics import YOLO

BEST = (
    "/content/drive/MyDrive/RoadVision/"
    "experiment13_d20_d40_balance/weights/best.pt"
)

DATA = "/content/RoadVision/data/custom_city_recovered/data.yaml"

model = YOLO(BEST)

m = model.val(
    data=DATA,
    split="test",
    imgsz=640,
    conf=0.001,
    iou=0.7,
    plots=True,
    verbose=True
)

print("\n" + "=" * 70)
print("EXPERIMENT 13 — FINAL TEST")
print("=" * 70)

print(f"Precision : {m.box.mp:.4f}")
print(f"Recall    : {m.box.mr:.4f}")
print(f"mAP50     : {m.box.map50:.4f}")
print(f"mAP50-95  : {m.box.map:.4f}")

for i, name in enumerate(["D00", "D10", "D20", "D40"]):
    print(
        f"{name}: "
        f"P={m.box.p[i]:.4f}, "
        f"R={m.box.r[i]:.4f}, "
        f"mAP50={m.box.ap50[i]:.4f}, "
        f"mAP50-95={m.box.ap[i]:.4f}"
    )

In [ ]:
from ultralytics import YOLO

MODEL = "/content/drive/MyDrive/RoadVision/experiment7_custom_finetune/weights/best.pt"

model = YOLO(MODEL)

results = model.predict(
    source="/content/RoadVision/data/custom_city_recovered/images/test",
    imgsz=640,
    conf=0.001,
    iou=0.7,
    save=True,
    save_txt=True,
    save_conf=True,
    project="/content/drive/MyDrive/RoadVision",
    name="final_test_predictions",
    exist_ok=True,
    device=0,
    verbose=True
)

print("\nPredictions saved.")

In [ ]:
from ultralytics import YOLO

MODEL = "/content/drive/MyDrive/RoadVision/experiment7_custom_finetune/weights/best.pt"
DATA = "/content/RoadVision/data/custom_city_recovered/data.yaml"

model = YOLO(MODEL)

model.val(
    data=DATA,
    split="test",
    imgsz=640,
    conf=0.001,
    iou=0.7,
    plots=True,
    project="/content/drive/MyDrive/RoadVision",
    name="FINAL_EVAL_EXP7",
    exist_ok=True,
    verbose=True
)

In [ ]:
from ultralytics import YOLO

MODEL = "/content/drive/MyDrive/RoadVision/experiment7_custom_finetune/weights/best.pt"

model = YOLO(MODEL)

results = model.predict(
    source="/content/RoadVision/data/custom_city_recovered/images/test",
    imgsz=640,
    conf=0.05,
    iou=0.7,

    save=True,
    save_txt=True,
    save_conf=True,

    project="/content/drive/MyDrive/RoadVision",
    name="FINAL_PREDICTIONS_CONF05",
    exist_ok=True,

    device=0,
    verbose=True
)

print("\n" + "=" * 70)
print("FINAL PREDICTIONS — CONF 0.05")
print("=" * 70)
print("Saved to:")
print("/content/drive/MyDrive/RoadVision/FINAL_PREDICTIONS_CONF05")